In [ ]:
import heapq 
l = [] 
heapq.heappush(l, (0,'hey') )
heapq.heappush(l, (0, 'hi'))

l2 = []
heapq.heappush(l2, (0, {'a':1}) )
# heapq.heappush(l, (0, {'b':2}))  BAD heapq uses the second element of the tuple, when first element is equal. But, you cant compare dict1 > dict2. 
# heapq knows this and discusses using a 3-tuple (priority, idx, data), where idx is autoincrement and thus always works as a tiebreaker. 
# OR, store something comparable. Like a position. ofc, if there were duplicate posiitons, an idx tiebreaker would still be needed. Q: are postions repeatable? A: No. idts. Hulls can&wjill overlap(rarely) but if they do, and a trace is drawn on the point of overlap, 
# I can merely use positions instead of nodes. ( fscore, position ) rather than (fscore, node)

l3=  [] 
heapq.heappush(l3 ,( 0 , (1,1))) 
# How/where are positons guaranteed to be unique? 

In [ ]:
from utils import * 
l = QLineF()
l.setPoint

In [ ]:
# An A star implementation which relies not on a grid, but an adjacency matrix/ visibililty graph. With ffline heuristic and distance
import numpy as np 
from math import sqrt 
import heapq

start_position= (0,0)
goal_position = (9,7)
grid = np.zeros((20,20))

open_list = []
open_dict = {}
closed_set = {}

graph =     {   (0,0) : { 'neighbors': {(2,2) , (4,5) , (5,4)}          } ,             
                (2,2) : { 'neighbors': {(0,0), (6,4)}                   } ,               
                (4,5) : { 'neighbors': {(0,0), }                        } ,               
                (5,4) : { 'neighbors': {(2,2,), (0,0), (3,3) , (9,7) }  } ,                                 
                (6,4) : { 'neighbors': {(2,2)}                          } ,                    
                (3,3) : { 'neighbors': {(5,4), (9,7)}                   } ,                    
                (9,7) : { 'neighbors': {(5,4), (3,3)}                   } ,                    
              }# Draw this out to see it. Initially, only position & neeighbor is known. 

def create_node(position, neighbors, g=float('inf'), h=0.0, predecessors = [] ):
    return{ 'position': position,
            'neighbors': neighbors,
            'g':g,
            'h':h,
            'f':g+h,
            'predecessors':predecessors,
        }

def calculate_distance(position1 , position2):
    x1,y1 = position1
    x2,y2 = position2
    return sqrt( (x2-x1)**2 + (y2-y1)**2 ) 

def calculate_ff_distance(position1 , position2): # Distance, when movements constrained to 45degree angles. Draw it out to see it.
    x1,y1 = position1
    x2,y2 = position2 
    dx = x2-x1 
    dy = y2 - y1 
    return abs(dy - dx) + sqrt(2)*min(dx,dy)

def calculate_heuristic(position1, position2):
    return calculate_ff_distance(position1,position2)

def reconstruct_path(node):
    path = [] 
    while node: 
        path.append(node['position'])
        predecessors = node['predecessors']
        predecessor = predecessors[0] if predecessors else None
        print('PREDECESSOR:', predecessor) 
        node = closed_set.get(predecessor)
    return path[::-1]
        
def a_star( start_position , goal_position):
    print("graph[start_position]['neighbors']", graph[start_position]['neighbors'])
    current_node = create_node(start_position, graph[start_position]['neighbors'], g=0) # remember to set g to zero!

    open_list.append( (current_node['f'] ,  start_position ) ) 
    open_dict.update( {current_node['position'] : current_node} )
    
    while open_list:
        _ , current_position = heapq.heappop(open_list) #pop lowest fscore 2-tuple (fscore, position)
        current_node = open_dict[current_position] # get node from open_dict keyed on position
        
        closed_set.update({current_node['position']: current_node}) # Add to closed_set 
        
        if current_node['position'] == goal_position: # goal_position HAS to be part of graph
            print('DONE')
            print('CURRENT_NODE:', current_node)
            return reconstruct_path(current_node)
        
        neighbor_positions = current_node['neighbors']
        for neighbor_position in neighbor_positions: 
            if neighbor_position in open_dict or neighbor_position in closed_set: 
                continue
            else:
                neighbor_node = create_node(
                                       neighbor_position,
                                       neighbors = graph[neighbor_position]['neighbors'], # Grab the neighbors from the graph
                                       g = current_node['g'] + calculate_ff_distance(current_node['position'] , neighbor_position), # DOES NOT reflect accurate ffline distance
                                       h = calculate_heuristic(neighbor_position , goal_position ),
                                       predecessors = [current_node['position']] # Neighbor node came from current positon 
                )
                # node_name += 1  nodes are named by their position
                

                open_dict.update({neighbor_position : neighbor_node})
                print()
                print("neighbor_node['f']", neighbor_node['f'])
                print('neighbor_node',neighbor_node)
                print('neighbor_position:', neighbor_position)
                heapq.heappush(open_list, (neighbor_node['f'] , neighbor_position) )
                
path = a_star(start_position , goal_position)
print('PATH:', path)

import math
from utils import * 
from TraceItem import *


        

        

        
    
        

graph[start_position]['neighbors'] {(4, 5), (5, 4), (2, 2)}

neighbor_node['f'] 12.485281374238571
neighbor_node {'position': (4, 5), 'neighbors': {(0, 0)}, 'g': 6.656854249492381, 'h': 5.82842712474619, 'f': 12.485281374238571, 'predecessors': [(0, 0)]}
neighbor_position: (4, 5)

neighbor_node['f'] 11.899494936611667
neighbor_node {'position': (5, 4), 'neighbors': {(2, 2), (3, 3), (9, 7), (0, 0)}, 'g': 6.656854249492381, 'h': 5.242640687119286, 'f': 11.899494936611667, 'predecessors': [(0, 0)]}
neighbor_position: (5, 4)

neighbor_node['f'] 11.899494936611665
neighbor_node {'position': (2, 2), 'neighbors': {(6, 4), (0, 0)}, 'g': 2.8284271247461903, 'h': 9.071067811865476, 'f': 11.899494936611665, 'predecessors': [(0, 0)]}
neighbor_position: (2, 2)

neighbor_node['f'] 11.899494936611665
neighbor_node {'position': (6, 4), 'neighbors': {(2, 2)}, 'g': 7.65685424949238, 'h': 4.242640687119286, 'f': 11.899494936611665, 'predecessors': [(2, 2)]}
neighbor_position: (6, 4)

neighbor_node['f'] 1

In [ ]:
# node = {   (0,0) : { 'neighbors': {'B' ,'C' , 'D'}     }  }
# print(node)
d = {(0,0):'a'}
t = ( 1, (0,0) ) 
x, (y,z) = t
print( x , y, z) # Unpacking works well, for tuples 

d = { (0,0): {'g':10.0} }
print(list(d.items()))
for k,v in d.items(): # There is no equivalent unpacking for dictionaries
    k = k
    v = v
    
print(k)
print(v)


In [ ]:
# An A star implementation which relies not on a grid, but an adjacency matrix/ visibililty graph. 
import numpy as np 
from math import sqrt 
import heapq

start_position= (0,0)
goal_position = (9,7)
grid = np.zeros((20,20))

open_list = []
open_dict = {}
closed_set = {}
    
graph =     {   (0,0) : { 'neighbors': {(2,2) , (4,5) , (5,4)}     } ,             
                (2,2) : { 'neighbors': {(0,0), (6,4)}       } ,               
                (4,5) : { 'neighbors': {(0,0), }           } ,               
                (5,4) : { 'neighbors': {(2,2,), (0,0), (3,3) , (9,7) } } ,                                 
                (6,4) : { 'neighbors': {(2,2)}                } ,                    
                (3,3) : { 'neighbors': {(5,4), (9,7)}        } ,                    
                (9,7) : { 'neighbors': {(5,4), (3,3)}                } ,                    
              }# Draw this out to see it. Initially, only position & neeighbor is known. 

def create_node(position, neighbors, g=float('inf'), h=0.0, predecessors = [] ):
    return{ 'position': position,
            'neighbors': neighbors,
            'g':g,
            'h':h,
            'f':g+h,
            'predecessors':predecessors,
        }

def calculate_distance(position1 , position2):
    x1,y1 = position1
    x2,y2 = position2
    return sqrt( (x2-x1)**2 + (y2-y1)**2 ) 

def calculate_heuristic(position1, position2):
    return calculate_distance(position1,position2)

def reconstruct_path(node):
    path = [] 
    while node: 
        path.append(node['position'])
        predecessors = node['predecessors']
        predecessor = predecessors[0] if predecessors else None
        print('PREDECESSOR:', predecessor) 
        node = closed_set.get(predecessor)
    return path[::-1]
        
def a_star( start_position , goal_position):
    print("graph[start_position]['neighbors']", graph[start_position]['neighbors'])
    current_node = create_node(start_position, graph[start_position]['neighbors'], g=0) # remember to set g to zero!

    open_list.append( (current_node['f'] ,  current_node ) ) 
    open_dict.update( {current_node['position'] : current_node} )
    
    while open_list:
        _ , current_node = heapq.heappop(open_list) #pop lowest fscore node
        closed_set.update({current_node['position']: current_node}) # Add to closed_set 
        
        if current_node['position'] == goal_position: # goal_position HAS to be part of graph
            print('DONE')
            print('CURRENT_NODE:', current_node)
            return reconstruct_path(current_node)
        
        neighbor_positions = current_node['neighbors']
        for neighbor_position in neighbor_positions: 
            if neighbor_position in open_dict or neighbor_position in closed_set: 
                continue
            else:
                neighbor_node = create_node(
                                       neighbor_position,
                                       neighbors = graph[neighbor_position]['neighbors'], # Grab the neighbors from the graph
                                       g = current_node['g'] + calculate_distance(current_node['position'] , neighbor_position), # DOES NOT reflect accurate ffline distance
                                       h = calculate_heuristic(neighbor_position , goal_position ),
                                       predecessors = [current_node['position']] # Neighbor node came from current positon 
                )
                # node_name += 1  nodes are named by their position
                
                open_dict.update({neighbor_position : neighbor_node})
                print("neighbor_node['f']", neighbor_node['f'])
                print('neighbor_node',neighbor_node)
                heapq.heappush(open_list, (neighbor_node['f'] , neighbor_node) )
                
path = a_star(start_position , goal_position)
print('PATH:', path)

        

In [ ]:
import timeit
print(timeit.timeit('"-".join(str(n) for n in range(100))', number=10000))

print(timeit.timeit('"-".join([str(n) for n in range(100)])', number=10000))

print(timeit.timeit('"-".join(map(str, range(100)))', number=10000))

def func(a, b):
    return a + b 

# '-'.join( str(x) for x in range(10000))
    
print(timeit.timeit('"-".join(str(x) for x in range(10000))' , number = 10000)) # Really gotta watch order of magnitudes bc this takes 8 seconds.
callable = func(3,4) 
callable = lambda 

print(timeit.timeit(expression))


In [2]:
from utils import * 
p = QPoint(0,0)
print(tuple(p))

tup = (8,8)
print(tuple(tup))



c:\Users\robby\OneDrive\part_database


TypeError: 'PySide6.QtCore.QPoint' object is not iterable

In [ ]:
# A* Datacamp tutorial. 
# Open Set: aka queue

import heapq 
from math import sqrt

def create_node(position, g=float('inf'), h=0.0, parent=None):
    # g: cost from start to node 
    # h: estimated cost from node to goal
    # parent: parent node 
    return { 
        'position' : position,
        'g' : g,
        'h' : h, 
        'f' : g + h,
        'parent': parent
    }

def calculate_heuristic(pos1, pos2):
    x1, y1 = pos1 
    x2, y2 = pos2 
    return sqrt( ( x2-x1 ) ** 2 + ( y2-y1 ) ** 2 )

def get_neighbor_positions(grid, position): 
    num_rows , num_columns = grid.shape
    x , y = position
    
    all_neighbor_positions = [  # All possible moves (eight total)
        (x+1 , y) , (x-1 , y), 
        (x , y+1) , (x , y-1),
        (x+1 , y+1) , (x-1 , y-1), 
        (x+1 , y-1) , (x-1 , y+1)
    ]
    valid_neighbor_positions = []

    for neighbor_x , neighbor_y in all_neighbor_positions:
        if 0 <= neighbor_x < num_columns and 0 <= neighbor_y < num_rows: 
            if grid[neighbor_x , neighbor_y] == 0: 
                valid_neighbor_positions.append((neighbor_x , neighbor_y) )
    return valid_neighbor_positions 

def reconstruct_path(goal_node):
    path = []
    current = goal_node 
    while current is not None: 
        path.append(current['position'])
        current = current['parent']
    return path[::-1] # slice path in reverse, such that returns path from start to goal
    
# We maintain TWO sets: an open set, for nodes we haven't explored, aka the queue, and a CLOSED set, for nodes we checked. As we explore the grid, we'll continuously update path costs whenever we find better routes, until we reach our goal

def find_path(grid , start , goal):
    start_node = create_node(position = start, g=0 , h= calculate_heuristic(start, goal))
    
    open_list = [ (start_node['f'] , start) ] # the queue, needs to be a binary tree for heapq module (binary as in each node has TWO children, not as in base2 numbers 0101)
    open_dict=  {start: start_node} # We keep an open_list AND an open dict bc, its complicated but we need them both
    closed_set = set() # explored_nodes ( keep in memory, bc we have to update & eventually reconstruct_path)
    
    while open_list: 
        _ , current_pos = heapq.heappop(open_list) # Get node with lowest f. heapq.heappop(heap) tries to pop & return smallest item from heap
        current_node = open_dict[current_pos]
        
        if current_pos == goal:
            return reconstruct_path(current_node)
        
        closed_set.add(current_pos) # store pos
        
        # Add neighbors to the queue. Neighbors may be closed, open, or new. If they are new: create node and push onto queue. else, do nothing. 
        for neighbor_pos in get_neighbor_positions(grid, current_pos):
            if neighbor_pos in closed_set :
                continue # If in closed set don't bother updating g; already explored ( Q: Why is this so? )
            tentative_g = current_node['g'] + calculate_heuristic(current_pos, neighbor_pos) # Since heuristic is real-life distance, heuristic is stand-in for actual distance from current to neighbor node 
            
            if neighbor_pos not in open_dict: 
                neighbor = create_node(
                    position = neighbor_pos, 
                    g=tentative_g,
                    h = calculate_heuristic(neighbor_pos, goal),
                    parent= current_node)
                heapq.heappush(open_list, (neighbor['f'], neighbor_pos)) # Add suitable neighbors to the open_list aka heap 
                open_dict[neighbor_pos] = neighbor

                
    return [] # No path found 

# ! pip install matplotlib 
import matplotlib.pyplot as plt
import numpy as np

def visualize_path(grid, path):
    """
    Visualize the grid and found path.
    """
    plt.figure(figsize=(10, 10))
    plt.imshow(grid, cmap='binary')
    
    if path:
        path = np.array(path)
        plt.plot(path[:, 1], path[:, 0], 'b-', linewidth=3, label='Path')
        plt.plot(path[0, 1], path[0, 0], 'go', markersize=15, label='Start')
        plt.plot(path[-1, 1], path[-1, 0], 'ro', markersize=15, label='Goal')
    
    plt.grid(True)
    plt.legend(fontsize=12)
    plt.title("A* Pathfinding Result")
    plt.show()
        

# Create a sample grid
grid = np.zeros((20, 20))  # 20x20 grid, all free space initially
# Add some obstacles
grid[5:15, 10] = 1  # Vertical wall
grid[5, 5:15] = 1   # Horizontal wall
# Define start and goal positions
start_pos = (2, 2)
goal_pos = (18, 18)
# Find the path
path = find_path(grid, start_pos, goal_pos)
if path:
    print(f"Path found with {len(path)} steps!")
    visualize_path(grid, path)
else:
    print("No path found!")

In [ ]:
# Do not Keep working on this: I figure the best use of all shortest paths is to choose btwn either first shortest path found, or the shortest path with the least vertices... so for now, I can just work with first shortest path found.
# Modify AStar to return all shortest paths. Strategy: when neighbor_node is revisited, IF its g is equal to stored g, find it in the open_dict or closed_dict( may ALWAYS be in open dict?) , and append it's predecessors. Then, after first time goal_node == current_node, reconstruct_paths based on predecessors
import heapq 
import math

def create_node(position, g= float('inf') , h = 0.0 , predecessors = []):
    return {
        'position' : position , 
        'g' : g , 
        'h' : h , 
        'f' : g + h ,
        'predecessors' : predecessors,
    }

def calculate_distance(position1 , position2) :
    x1 , y1 = position1 
    x2 , y2 = position2 
    return math.sqrt( (x2 - x1) ** 2 + (y2 - y1) ** 2 )

def calculate_heuristic(position1, position2) :
    return calculate_distance(position1 , position2)

def get_neighbor_positions(grid, position): 
    num_rows , num_columns = grid.shape
    x , y = position
    
    all_neighbor_positions = [  # All possible moves (eight total)
        (x+1 , y) , (x-1 , y), 
        (x , y+1) , (x , y-1),
        (x+1 , y+1) , (x-1 , y-1), 
        (x+1 , y-1) , (x-1 , y+1)
    ]
    valid_neighbor_positions = []

    for neighbor_x , neighbor_y in all_neighbor_positions:
        if 0 <= neighbor_x < num_columns and 0 <= neighbor_y < num_rows: 
            if grid[neighbor_x , neighbor_y] == 0: 
                valid_neighbor_positions.append((neighbor_x , neighbor_y) )
    return valid_neighbor_positions 

def reconstruct_path(goal_node, closed_dict):
    current_node = goal_node 
    path = [] 
    while current_node: 
        current_position = current_node['position']
        path.append(current_position)
        # current_node = current_node['predecessor']
        predecessors = closed_dict[current_position]['predecessors']
        if predecessors: # bc the start nodes predecessors is []
            predecessor_position = closed_dict[current_position]['predecessors'][0]
            current_node = closed_dict.get(predecessor_position) 
            if len(predecessors) > 1: 
                print()
                print('PREDECESSORS:', type(predecessors) , predecessors)
        else:
            print('PATH:', path)
            return path[::-1]



def find_path(grid, start_position , goal_position):
    start_node = create_node(start_position, g = 0, h = calculate_heuristic(start_position , goal_position))
    open_list = [ ( start_node['f'], start_position ) ]
    open_dict = { start_position : start_node }
    # closed_set = set()
    closed_dict = {} # Gotta know both position AND node

    while open_list: 
        _ , current_position = heapq.heappop(open_list)  
        current_node = open_dict.pop(current_position)
        closed_dict[current_position] = current_node
        # current_node = open_dict[current_position]    # Um why are some nodes both open and closed? Oh, its bc I was open_dict[current_position] not open_dict.pop(current_position)
        # closed_set.add(current_position)
        
        if current_position == goal_position: 
            print('current_position == goal_position')
            path = reconstruct_path(current_node, closed_dict)
            print('PATH:', path)
            return open_dict, closed_dict, path


        for neighbor_position in get_neighbor_positions(grid , current_position):
            g = current_node['g'] + calculate_distance(current_position , neighbor_position)
            
            # if neighbor_position in closed_set: 
                # continue  #
            # Nodes can be closed, open, or new. if closed or open, maybe we'll update predecessors. if new, we'll push node into queue.
            closed_neighbor= closed_dict.get(neighbor_position , None)
            
            if closed_neighbor: # If we already visited this node: We'll update its predecessors, if g is equal to stored g
                continue
            #     # print('G:', g)
            #     # print('CLOSED_G:', closed_neighbor['g'])
            #     # print(f'NEIGHBOR_POSITION: {neighbor_position} IS IN CLOSED DICT') # Trad A* would completely ignore nodes in closed_set, but we have to consider them if g-value is equal 
            #     if g == closed_neighbor['g']: 
            #         print('This path is an equal') 
            #         closed_dict[neighbor_position]['predecessors'].append(current_position)
                    
            open_neighbor = open_dict.get(neighbor_position, None) 
            if open_neighbor: #This never happens on most graphs, because it happens when revisiting a node, with equal path, which only happens if there is a symmetric obstacle in the way. Also, when this circumstance happens, the node will not yet have been popped.
                # print(f'NEIGHBOR_POSITION: {neighbor_position} IS IN OPEN DICT') # Trad A* would completely ignore nodes in closed_set, but we have to consider them if g-value is equal 
                if g == open_neighbor['g']:
                    # print('THIS PATH IS AN EQUAL')
                    open_neighbor['predecessors'].append(current_position)

            elif neighbor_position not in open_dict: # THIS LINE EXEMPT CAUSES WANDERING PATH 
                h = calculate_heuristic(neighbor_position , goal_position)
                neighbor_node = create_node(neighbor_position , g , h , [current_position] )
                heapq.heappush(open_list, ( neighbor_node['f'] , neighbor_position))
                open_dict[neighbor_position] = neighbor_node

    return [] 

# ! pip install matplotlib 
import matplotlib.pyplot as plt
import numpy as np

def visualize_path(grid, path, closed_dict , open_dict):
    # print('CLOSED_DICT:', type(closed_dict), closed_dict)
    # print('OPEN_DICT:', type(open_dict), open_dict)
    """
    Visualize the grid and found path.
    """
    plt.figure(figsize=(10, 10))
    plt.imshow(grid, cmap='binary')
    
    closed_positions = np.array(list(closed_dict.keys()))
    open_positions = np.array(list(open_dict.keys()))
    print('closed_positions', list(closed_positions))
    print('OPEN_POSITIONS:', list(open_positions))
    print('CLOSED_DICT[(17, 17)]', closed_dict[(17,17)])
    if path:
        path = np.array(path)
        plt.plot(path[:, 0], path[:, 1], 'b-', linewidth=3, label='Path')
        plt.plot(path[0, 0], path[0, 1], 'go', markersize=15, label='Start')
        plt.plot(path[-1, 0], path[-1, 1], 'ro', markersize=15, label='Goal')
        plt.plot(open_positions[:, 0] , open_positions[:, 1] , 'gx' , markersize = 10, label = 'Open')
        plt.plot(closed_positions[:, 0] , closed_positions[:, 1] , 'r+' , markersize = 10, label = 'Closed')
    plt.grid(True)
    plt.legend(fontsize=12)
    plt.title("A* Pathfinding Result")
    plt.show()
        

# Create a sample grid
grid = np.zeros((20, 20))  # 20x20 grid, all free space initially
# Add some obstacles
grid[10 , 5:15] = 1  # Vertical wall
grid[5:15, 5] = 1   # Horizontal wall
grid[10,10] = 1 
# Define start and goal positions
start_pos = (2, 2)
goal_pos = (18, 18)
# Find the path
open_dict, closed_dict, path = find_path(grid, start_pos, goal_pos)

if path:
    print(f"Path found with {len(path)} steps!")
    visualize_path(grid, path, closed_dict, open_dict)
else:
    print("No path found!")

# TODO Fix axes being transposed or something wrong with xy flipped 

In [ ]:

def meth(x, y):
    print('X:', x)
    print('Y:', y)
meth(1,2)
meth(y=5 , x=6)

def meth(x, y=None):
    print('X:', x)
    print('Y:', y)
meth(3,4)
meth(7, y=8)


In [ ]:
# Advanced Super is very confusing: 
# https://stackoverflow.com/questions/9575409/calling-parent-class-init-with-multiple-inheritance-whats-the-right-way
# Go to the docs :
# https://rhettinger.wordpress.com/2011/05/26/super-considered-super/ ( Wow, these are 15 years old, verbose but still a bit helpful)
#Arguments can be passed to super. It takes two arguments, a class and a class instance.

In [ ]:


class First(object):
    def __init__(self):
        super().__init__() # In this example, super().__init__() does not HAVE to go here, bc First only inherits object, the python builtin from which all classes eventually inherit. But you can include it anyway, to make your code more robust to changes-- say, changing First to inherit from some other class.
        print("first")

class Second(object):
    def __init__(self):
        super().__init__()
        print("second")

class Third(First, Second):
    def __init__(self):
        super().__init__() 
        print("third")

class Thirdv2(First , Second):
    def __init__(self):
        First.__init__(self)
        Second.__init__(self) # You could manually call __init__ methods and not use super(). IMO this is ok for small code, or personal code -- community does not reccomend, bc 'super is more pythonic'
        print('thirdv2')
        # Oh wait-- notice Thirdv2 calls Second Twice, Whats up with that? 
        
Third()
print()
Thirdv2()
# First and Second have to have super() too, else execution stops at the end of Second.__init__().

second
first
third

second
first
second
thirdv2


In [3]:
# Here, constructors of class A and class B DO NOT do super().__init__, and it causes problems for class C, because class C inherits A and B.
class A:
    def __init__(self):
        self.a = 'a'
        
class B: 
    def __init__(self):
        self.b='b'
        
class C(A,B):
    def __init__(self):
        super().__init__()

c = C()
print(c.a)
# print(c.b)  AttributeError: 'C' object has no attribute 'b' # Because super().__init__ was NOT called in class B's constructor, super() cannot follow the MRO chain to the bottom! That is why, super().__init__() ought to be invoked in class B, and class A, and every class, because classes which inherit those classes depend on it! 

# Here is a proper example of inheritance with proper usage of super().__init__()
class A:
    def __init__(self):
        super().__init__()
        self.a = 'a'
        
class B: 
    def __init__(self):
        super().__init__()
        self.b='b'
        
class C(A,B):
    def __init__(self):
        super().__init__()

c = C()
print()
print(c.a)
print(c.b) 

a

a
b


In [5]:
#  SO SUPER USAGE: https://stackoverflow.com/questions/3277367/how-does-pythons-super-work-with-multiple-inheritance
class First(object):
    def __init__(self):
        print('first')

class Second(First):
    def __init__(self):
        print ("second")

class Third(First):
    def __init__(self):
        print ("third")

class Fourth(Second, Third):
    def __init__(self):
        super(Fourth, self).__init__()
        print ("fourth")
        
# f = Fourth() # 
# second
# fourth
# Fourth.mro() # [__main__.Fourth, __main__.Second, __main__.Third, __main__.First, object] The mro is fourth, second, third, first, object. 
# But, this example has intentional 'bug':  super() is only used in Fourth. Normally, you want super() in all classes, even classes that only inherit object. Note instantiating Fourth() prints "second" and "fourth" -- because super() call was lacking in Second, python could not follow the mro down the chain.
# For 'co operative subclassing' to work, add super() calls to First, Second and Third: 

class First(): 
    def __init__(self):
        super().__init__()
        print('first')
        
class Second(First):
    def __init__(self):
        super().__init__()
        print('second')
        
class Third(First):
    def __init__(self):
        super().__init__()
        print('third')
        
class Fourth(Second, Third):
    def __init__(self):
        super().__init__()
        print('fourth')
        
f = Fourth()
# Note that 'third' happens before 'second'. why is that? 


first
third
second
fourth


In [ ]:
# MRO: Method Resolution Order , the order in which python calls methods. Oft used in context of the constructor, the __init__ method. 
    
class Parent1: # A class, Parent1.
    
    def __init__(self):
        # super().__init__() is not called. (This is NOT stonks) (class Child that inherit Parent1, won't be able to follow MRO chain; Child's __init__ stops here.  )
        print('parent1')
    
class Parent2: # A class, Parent2.
    
    def __init__(self):
        # super().__init__() is not called, but it should be. w/o super().__init__() in Parent2, we are messing up Child's  super().__init__, and it can't continue down the MRO chain 
        print('parent2')
    
class Child(Parent1 , Parent2): # Child inherits Parent1 & Parent2 
    def __init__(self):         # 
        print('child')
        super().__init__()      # good to use super() here-- but super() can't traverse the MRO chain past Parent1 , because Parent1 did not call super() in its constructor; it lacks super().__init__().

print("Look at Child's MRO: Child, Parent1 , Parent2, object")
print(Child.mro())
print('Knowing the MRO is good, but w/o usage of super in Parent1 and Parent2, we cannot follow the MRO chain:')
print()
print('W/o super().__init__() in Parent1 & Parent2: Only init Parent1:')
Child() 
# 'child' 
# 'parent1' 
# We see child parent1 printed. Child's super().__init__ only called Parent1's __init__. But, we need it to initialize both Parent1, and Parent2, so be sure to include super().__init__() in Parent1's and Parent2's constructors:

class Parent1: # A class, Parent1.
    
    def __init__(self):
        super().__init__() # W/ super().__init__ in Parent1, class Child that inherit Parent1, can follow MRO chain; Child's __init__ will continue to the next __init__ in the MRO
        print('parent1')
    
class Parent2: # A class, Parent2.
    
    def __init__(self):
        super().__init__() # Called as it should be. With super().__init__() in Parent2, child's  super().__init__ can continue down the MRO chain ( which is behind the scenes in this case, but is 'object', a python builtin from which every class in python inherits) 
        print('parent2')
    
class Child(Parent1 , Parent2): # Child inherits Parent1 & Parent2 
    def __init__(self):         # 
        print('child')
        super().__init__()      # good to use super() here-- and because Paren1 and Parent2 called super() in their constructors, everything is as it was meant to be 

print()
print('Proper Usage of super():')
Child() 
print('Note how parent2 printed before parent1 -- why is that?')

Look at Child's MRO: Child, Parent1 , Parent2, object
[<class '__main__.Child'>, <class '__main__.Parent1'>, <class '__main__.Parent2'>, <class 'object'>]
Knowing the MRO is good, but w/o usage of super in Parent1 and Parent2, we cannot follow the MRO chain:

W/o super().__init__() in Parent1 & Parent2: Only init Parent1:
child
parent1

Proper Usage of super():
child
parent2
parent1
Note how parent2 printed before parent1 -- why is that?


In [ ]:

# Python multiple inheritance super() : call super() in every class, even classes which only inherit object, to allow super() to follow the MRO down the line. The below example is 'broken'-- super() is not called in Parent1 or Parent2, so, Child's super() call cannot follow the MRO.
class Parent1:
    def __init__(self):
        print("Parent1 initialized")

class Parent2:
    def __init__(self):
        print("Parent2 initialized")

class Child(Parent1, Parent2):
    def __init__(self):
        # super(Child, self).__init__() Redundant, equivalent to below: 
        super().__init__() # Will init Parent1 but not Parent2 
        print("Child initialized")

class Child2(Parent1, Parent2): # I understand this:
    def __init__(self):
        Parent1.__init__(self) # Can always call the parent classes directly, w/o usage of super(). Community does not reccomend, but, is good for learning.
        Parent2.__init__(self)
        print("Child2 Initialized")
        
class Child3(Parent1 , Parent2):
    def __init__(self):
        super().__init__() 
        super(Parent1, self).__init__() # get superclass of parent1, which is Parent2. (STILL requires naming 'Parent1', so kinda just worse & confusing)
        print('Child3 initialized')

child3 = Child3()


In [ ]:
# SUPER VARIABLES SUPER KEYWORD ARGUMENTS SUPER ARGUMENTS
# HOW ABOUT VARIABLES AND SUPER()?? spam *args and **kwargs. But favor **kwargs bc if you use *args, which are positional, you have to place args in MRO order, which is hard to remember, hence prefer kwargs
class A:
  def __init__(self, a, *args, **kwargs):
    print("A", a)
    
# class A:
#   def __init__(self, *args, a, **kwargs): # What does switching a to after *args accomplish? 'Keyword only parameters'. Strange bc a is a keyword arg, yet, has no default value: https://docs.python.org/3/tutorial/controlflow.html#keyword-only-arguments
#     print("A", a)

class B(A):
  def __init__(self, b, *args, **kwargs):
    super().__init__(*args, **kwargs)
    print("B", b)

class A1(A):
  def __init__(self, a1, *args, **kwargs):
    super().__init__(*args, **kwargs)
    print("A1", a1)

class B1(A1, B):
  def __init__(self, b1, *args, **kwargs):
    super().__init__(*args, **kwargs)
    print("B1", b1)

B1(a1=6, b1=5, b="hello", a=None)

# For my use case, QGraphicsItem


In [ ]:
class A:
    def __init__(self, a, *args, **kwargs):
        print('a:', a)
        
class B(A):
    def __init__(self, b, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print('b:', b)
        
class A1(A):
    def __init__(self, a1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print('a1:', a1)
        
class B1(A1, B): # Thus this class demands arguments a, b, and a1, in addition to any arguments demanded by itself
    def __init__(self, b1 , *args, **kwargs):
        super().__init__(*args , **kwargs)
        print('b1:', b1)
        
# B1's MRO should be : B1 A1 B A object. Breadth-first search(?)
print(B1.mro())
b1 = B1('b1', 'a1', 'b', 'a')  # Knowing the MRO, we can pass arguments positionally 
print()
b1 = B1(b1 = 'b1', a1 = 'a1', b='b', a='a') # Its also reasonable to use kwargs



        

[<class '__main__.B1'>, <class '__main__.A1'>, <class '__main__.B'>, <class '__main__.A'>, <class 'object'>]
a: a
b: b
a1: a1
b1: b1

a: a
b: b
a1: a1
b1: b1


In [ ]:
# Q : what if a class eats up all *args and **kwargs mid-mro? 
class A:
    def __init__(self, a, *args, **kwargs):
        print('a:', a)
        
class B(A): # Lets have B not pass on *args and **kwargs:  TypeError: A.__init__() missing 1 required positional argument: 'a'
    def __init__(self, b, *args, **kwargs):
        super().__init__()#*args, **kwargs)
        print('b:', b)
         
class A1(A): # Lets have A1 pass on *args and **kwargs as is best practice. Plus, A1 demands positional arg a1 
    def __init__(self, a1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print('a1:', a1)
        
class B1(A1, B): # Thus this class demands arguments a, b, and a1, in addition to any arguments demanded by itself
    def __init__(self, b1 , *args, **kwargs):
        super().__init__(*args , **kwargs)
        print('b1:', b1)
        
# B1's MRO should be : B1 A1 B A object. Breadth-first search(?)
print(B1.mro())
b1 = B1('b1', 'a1', 'b', 'a')  # Knowing the MRO, we can pass arguments positionally 
print()
b1 = B1(b1 = 'b1', a1 = 'a1', b='b', a='a') # Its also reasonable to use kwargs



[<class '__main__.B1'>, <class '__main__.A1'>, <class '__main__.B'>, <class '__main__.A'>, <class 'object'>]
a: a
b: b
a1: a1
b1: b1



TypeError: A.__init__() missing 1 required positional argument: 'a'

In [ ]:
def func( *args, a): # a is a 'keyword-only' parameter
    print(a) 
    
# func('a') TypeError: func() missing 1 required keyword-only argument: 'a'
func(a='a')

def func(a , *args):
    print(a)
    
func('a','b','c')
func('a')

# def func(a=None, b) SyntaxError: parameter without a default follows parameter with a default
    # print(a)
    # print(b)

# def func(self, a, *args, **kwargs , b='b'): SyntaxError: arguments cannot follow var-keyword argument 

def func(a , *args, b='b' , **kwargs):
    print()
    print('a:', a)
    print('args:', args)
    print('b:', b)
    print('kwargs:', kwargs)
    
func('a' , 'x', b='c' , c='c' , d='d')

# make keyword-only arguemnts 
def func(a, *args, b='b'): # b HAS to be specified via keyword
    pass
func('a', 'b', b='b')

def func(a, b='b', *args, **kwargs): # b is actually a positional argument; keyword arguemnts CAN be passed positionally
    print()
    print('a:', a)
    print('b:', b)
    print('args:', args)
    print('kwargs:', kwargs)
    
func('a') # If b is not provided, b will default to b...
# but if any *args are provided, then b has to be provided, positionally . So it gets confusing fast.
# func('a','c', b='b') TypeError: func() got multiple values for argument 'b'
func('a', 'b', 'c', d='d')

def func(line, net=None, *args, **kwargs):
    print('LINE:', line)
    print('NET:', net)
    print('ARGS:', args)
    print('KWARGS:', kwargs)
    
print()
func('line') # 

    



a
a
a

a: a
args: ('x',)
b: c
kwargs: {'c': 'c', 'd': 'd'}

a: a
b: b
args: ()
kwargs: {}

a: a
b: b
args: ('c',)
kwargs: {'d': 'd'}

LINE: line
NET: None
ARGS: ()
KWARGS: {}


In [1]:
# Why is this freezing?
# DANGERS OF .CHILDRENBOUNDINGRECT() and .CONTAINS() WORKING IN LOCAL, NOT SCENE, COORDINATES:
from PySide6.QtCore import * 
from PySide6.QtWidgets import * 
from PySide6.QtGui import * 
import sys 
from MyView import MyView 
        
# class MyScene(QGraphicsScene):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
        
class MyItem(QGraphicsRectItem):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.child = QGraphicsRectItem(*args, **kwargs,  parent = self)
        self.child.setPen(QPen(Qt.red, 0))
        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsMovable | QGraphicsItem.GraphicsItemFlag.ItemIsSelectable)
        
    def boundingRect(self): # All works
        return super().boundingRect() # QGraphicsRectItem's boundingRect() accounts for penWidth & scenePos, while a reimplementation using .childBoundingRect() would not account for self's pen width or scene position.
    
    # def boundingRect(self): # All works 
    #     return self.childrenBoundingRect() # Reimplementing .boundingRect like so, accounts not for self's pen width, nor for scene position.
    
    # def boundingRect(self): # Moving dnw
    #     return self.mapRectToScene(self.childrenBoundingRect()) # Account for scene position (cannot grab&move)
    
    def paint(self,painter,option,widget):
        # super().paint(painter,option,widget)
        painter.setBrush(Qt.blue) 
        painter.setPen(Qt.cyan)
        painter.drawRect(self.childrenBoundingRect())
    
    def mouseMoveEvent(self, event):
        print('MyItem().MouseMoveEvent')
        super().mouseMoveEvent(event)

app = QApplication(sys.argv)
view = MyView()
scene = QGraphicsScene()
origin = QGraphicsEllipseItem(-1,-1,2,2)
origin.setBrush(QBrush(Qt.gray))
scene.addItem(origin)
item = MyItem(-30,-30, 60, 60)
item.setPos(200.0,200.0)
scene.addItem(item)

print('ITEM.CHILDRENBOUNDINGRECT:', item.childrenBoundingRect())
print('ITEM.BOUNDINGRECT():', item.boundingRect())
print('ITEM.mapRectToScene(item.childrenBoundingRect())):')
print(item.mapRectToScene(item.childrenBoundingRect()))
print()
# print('I DID MyITEM().SETPOS(200,200), BUT CHILDRENBOUNDINGRECT -> IN LOCAL COORDINATES, NOT SCENE COORDINATES. THUS, .CONTAINS(point), WHERE POINT IS IN LOCAL COORDINATES, THINKS ITEM DOES NOT CONTAIN (200,200)')

# print('ITEM.CONTAINS(0,0) RETURNS TRUE(When rect is nonzero rect) DESPITE ITEMS LOCATION AT 200,200 BC .BR() IMPLEMENTS .CHILDRENBOUNDINGRECT(), WHICH RETURNS BR IN LOCAL, NOT SCENE, COORDINATES:')
print("ITEM.CONTAINS(QPOINT(0,0))):") # 
print(item.contains(QPoint(0,0)))  # 
print()
# print('ITEM.CONTAINS(QPoint(200,200)) RETURNS FALSE, DESPITE ITEM CONTAINING SCENE COORDINATE 200,200:')
print('ITEM.CONTAINS(QPOINT(200,200))):')
print(item.contains(QPoint(200,200))) # Item is only 20 wide, does not contain local coordinate 200,200
print()
print('ITEM.CONTAINS(item.mapFromScene(QPoint(200,200)))):')
print(item.contains(item.mapFromScene(QPoint(200,200)))) # Item was moved tosscene position 200,200, item contains scenepos(200,200)


view.setScene(scene)
window = QWidget()
layout = QVBoxLayout()
layout.addWidget(view)
window.setLayout(layout)
window.show()
sys.exit(app.exec())



c:\Users\robby\OneDrive\part_database
ITEM.CHILDRENBOUNDINGRECT: PySide6.QtCore.QRectF(-30.000000, -30.000000, 60.000000, 60.000000)
ITEM.BOUNDINGRECT(): PySide6.QtCore.QRectF(-30.500000, -30.500000, 61.000000, 61.000000)
ITEM.mapRectToScene(item.childrenBoundingRect())):
PySide6.QtCore.QRectF(170.000000, 170.000000, 60.000000, 60.000000)

ITEM.CONTAINS(QPOINT(0,0))):
True

ITEM.CONTAINS(QPOINT(200,200))):
False

ITEM.CONTAINS(item.mapFromScene(QPoint(200,200)))):
True


SystemExit: 0

c:\Users\robby\OneDrive\part_database\myenv\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
class A:
    x = 1 
    
class B(A):
    pass

a = A()
b = B()

print(isinstance(b, B))
print(isinstance(b, A)) # True. b is an instance of A. b is also an instance of B.

True
True
True


NameError: name 'QGraphicsItem' is not defined

In [ ]:

import heapq 
from math import sqrt

def create_node(position, g=float('inf'), h=0.0, predecessor=None):
    # g: cost from start to node
    # h: estimated cost from node to goal
    # parent: parent node
    
    return { 
        'position' : position,
        'g' : g,
        'h' : h, 
        'f' : g + h,
        'predecessor': predecessor
    }

def calculate_distance(pos1, pos2):
    x1, y1 = pos1 
    x2, y2 = pos2
    return sqrt( ( x2-x1 ) ** 2 + ( y2-y1 ) ** 2 )

def calculate_heuristic(pos1, pos2):
    return calculate_distance(pos1, pos2)

def get_neighbor_positions(grid, position): 
    num_rows , num_columns = grid.shape
    x , y = position
    
    all_neighbor_positions = [  # All possible moves (eight total)
        (x+1 , y) , (x-1 , y), 
        (x , y+1) , (x , y-1),
        (x+1 , y+1) , (x-1 , y-1), 
        (x+1 , y-1) , (x-1 , y+1)
    ]
    valid_neighbor_positions = []

    for neighbor_x , neighbor_y in all_neighbor_positions:
        if 0 <= neighbor_x < num_columns and 0 <= neighbor_y < num_rows: 
            if grid[neighbor_x , neighbor_y] == 0: 
                valid_neighbor_positions.append((neighbor_x , neighbor_y) )
                
    return valid_neighbor_positions 

def reconstruct_path(goal_node):
    path = []
    current = goal_node 
    while current is not None: 
        path.append(current['position'])
        current = current['predecessor']
    print('PATH:', path)
    return path[::-1] # slice path in reverse, such that returns path from start to goal
    
# We maintain TWO sets: an open set, for nodes we haven't explored, aka the queue, and a CLOSED set, for nodes we checked. As we explore the grid, we'll continuously update path costs whenever we find better routes, until we reach our goal

def find_path(grid, start_position , goal_position):


    # Add start node to queue. Pop start node, add to closed set. Expand start_nodes neighbors, add to queue. 
    start_node = create_node(start_position , g=0 , h=calculate_distance(start_position, goal_position) )
    open_list = [ ( start_node['f'] , start_position ) ] 
    open_dict = { start_position : start_node }
    closed_set = set()
    
    while open_list:
        _ , current_position = heapq.heappop(open_list)
        current_node = open_dict[current_position] 
        
        if current_position in closed_set:
            continue

        closed_set.add(current_position)

        if current_position == goal_position:
            print('DONE.')
            return reconstruct_path(current_node)

        neighbors = get_neighbor_positions(grid, current_position)
        for neighbor in neighbors: 
            if neighbor not in open_dict: # THIS LINE EXEMPT CAUSES WANDERING PATH
                neighbor_node = create_node(neighbor , g=current_node['g']+ calculate_distance(current_position , neighbor) , h=calculate_distance(neighbor , goal_position) , predecessor = current_node )
                heapq.heappush(open_list, ( neighbor_node['f'] , neighbor ) )
                open_dict[neighbor] = neighbor_node
    print('OPEN_LIST:', open_list)
    print('OPEN_DICT:', open_dict)
    print('CLOSED_SET:', closed_set)
    return []


import matplotlib.pyplot as plt
import numpy as np

def visualize_path(grid, path):
    """
    Visualize the grid and found path.
    """
    plt.figure(figsize=(10, 10))
    plt.imshow(grid, cmap='binary')
    
    if path:
        path = np.array(path)
        plt.plot(path[:, 1], path[:, 0], 'b-', linewidth=3, label='Path')
        plt.plot(path[0, 1], path[0, 0], 'go', markersize=15, label='Start')
        plt.plot(path[-1, 1], path[-1, 0], 'ro', markersize=15, label='Goal')
    
    plt.grid(True)
    plt.legend(fontsize=12)
    plt.title("A* Pathfinding Result")
    plt.show()
        

# Create a sample grid
grid = np.zeros((20, 20))  # 20x20 grid, all free space initially
# Add some obstacles
grid[5:15, 10] = 1  # Vertical wall
grid[5, 5:15] = 1   # Horizontal wall
# Define start and goal positions
start_pos = (2, 2)
goal_pos = (18, 18)
# Find the path
path = find_path(grid, start_pos, goal_pos)
if path:
    print(f"Path found with {len(path)} steps!")
    visualize_path(grid, path)
else:
    print("No path found!")

In [ ]:
### practice to insert a MyFootprintItem QGrapicsItem 'hull' into the graph matrix 
import math
graph = [ # representing real coordinates 
    [ [0,0]       , [1,0] ],
    [ [0,1]       , [1,1] ],
]      

def insert_into_graph(graph,position):
    x, y = position
    num_rows = len(graph)
    num_columns = len(graph[0])

    # graph.insert(math.floor(y)+1 , l) # can't just insert at math.floor(y). May have x= .7 while a x=.5 already exists in the graph-- ( This would cause other routing problems but those r dealt with later)
    # So we need to know all the y's already in list, then insert after the nearest lesser match 

    row_insert_index = first_greater_y(graph, y)
    # print('ROW_INSERT_INDEX:', row_insert_index)
    row_to_insert = [ [graph[0][col_idx][0] , y ] for col_idx in range(num_columns) ] # graph[0][col_idx][0] bc graph[0] is the first row, row[col_idx] is a specific position, and specific_position[0] is the x-coordinate of specific_position. We HAVE to use x-coordinate already existing in graph, bc we don't know whats been added to our graph
    graph.insert( row_insert_index,  row_to_insert)
    
    # print('GRAPH STAGE1:')
    # for i in graph:
    #     print(i)
        
    column_insert_index = first_greater_x(graph, x) 
    for row_index, row in enumerate(graph): 
        cell_to_insert = [ x , graph[row_index][0][1] ] # graph[row_index][0][1] because graph[row_index] is a specific row that exists in graph, specific_row[0] is an actual cell, and actual_cell[1] is the y-coordinate of a specific position. We HAVE to use y-coordinate already existing in graph,bc we don't know whats been added to our graph.
        row.insert(column_insert_index, cell_to_insert)
    # print()
    # print('GRAPH STAGE2:')
    # for i in graph:
    #     print(i) 
           
def first_greater_x(graph, value):
    num_columns = len(graph[0])
    
    graph_xs = []
    row = graph[0] 
    for column_index in range(num_columns):
        graph_xs.append(row[column_index][0])
    # print('GRAPH XS:', graph_xs)
    for count, item in enumerate(graph_xs):
        if item>value:
            return count
    return count+1 
    
def first_greater_y(graph , value): # Returns index of first number in lst which is greater than value, or len(lst) if no greater number exists
    graph_ys  = [ row[0][1] for row in graph]
    # print('GRAPH_YS:', graph_ys)
    for count, item in enumerate(graph_ys): 
        if item>value: 
            return count
    return count+1 # Ok (?) to return index-out-of-range. insert() would merely append


def insert_rect_hull_into_graph(graph, hull, wire_width):
    l = hull.left() - wire_width/2
    t = hull.top() - wire_width/2
    b = hull.bottom() + wire_width/2
    r = hull.right() + wire_width/2
    
    insert_into_graph(graph, (l,t) )
    insert_into_graph(graph , (r,b) ) 

from PySide6.QtCore import QRectF
insert_rect_hull_into_graph(graph, QRectF(.2,.5, .2,.2), .1)
import numpy as np
import matplotlib.pyplot as plt
arr = np.array(graph)
print('ARR:')
print(arr)
for row in arr: 
    plt.plot(row[:,0] , row[:,1], 'go')

# Hey there's gotta be a library that already does this--matrix insertion (?) matrix expansion(?) I can't find any...


In [ ]:
# Q : How to go from : 
# (fp_line (start -1.6129 0.8763) (end -1.6129 -0.8763) (layer "F.CrtYd") (width 0.1524))
# (fp_line (start -1.6129 -0.8763) (end 1.6129 -0.8763) (layer "F.CrtYd") (width 0.1524))
# (fp_line (start 1.6129 -0.8763) (end 1.6129 0.8763) (layer "F.CrtYd") (width 0.1524))
# (fp_line (start 1.6129 0.8763) (end -1.6129 0.8763) (layer "F.CrtYd") (width 0.1524))
# to having a 'courtyard' path/rectangle? 
# path = QPainterPath()
# if 'CrtYd' in self.layer():
#     self.courtYards.update(reference_value= self.courtyards.get(reference_value, ''))
# Does QPainterPath have .contains() ? 


In [ ]:
### practice to insert a row and a column into a matrix 
import math
normal_graph = [
    [ [0,0]       , [1,0] ],
    [ [0,1]       , [1,1] ],
]      
position = (.7 , .4 )

def insert_into_graph(graph,position):
    x, y = position
    num_rows, num_columns = len(graph) , len(graph[0])
    # Want to insert: [ (0,.6) , (1,.6) ] . How do I create this list? 
    l = [ [col_idx ,y] for col_idx  in range(num_columns) ]
    graph.insert(math.floor(y)+1 , l )
    
    print('GRAPH STAGE1:')
    for i in graph:
        print(i)

    for row_idx, row in enumerate(graph):
        row.insert(math.floor(x)+1 , [x , row[0][1]]) # row[0][1] bc that's our current-y-level 
        
    print()
    print('GRAPH STAGE2:')
    for i in graph:
        print(i)
        
    return graph



In [ ]:
### Insert a row and a column into a matrix -- It gets harder once you've already inserted a single row/column. Working. Not robust against say, same position being inserted into graph twice
import math
normal_graph = [
    [ [0,0]       , [1,0] ],
    [ [0,1]       , [1,1] ],
]      
position = (.7 , .4 )

def insert_into_graph(graph,position):
    x, y = position
    num_rows = len(graph)
    num_columns = len(graph[0])

    # graph.insert(math.floor(y)+1 , l) # can't just insert at math.floor(y). May have x= .7 while a x=.5 already exists in the graph-- ( This would cause other routing problems but those r dealt with later)
    # So we need to know all the y's already in list, then insert after the nearest lesser match 

    row_insert_index = first_greater_y(graph, y)
    # print('ROW_INSERT_INDEX:', row_insert_index)
    row_to_insert = [ [graph[0][col_idx][0] , y ] for col_idx in range(num_columns) ] # graph[0][col_idx][0] bc graph[0] is the first row, row[col_idx] is a specific position, and specific_position[0] is the x-coordinate of specific_position. We HAVE to use x-coordinate already existing in graph, bc we don't know whats been added to our graph
    graph.insert( row_insert_index,  row_to_insert)
    
    # print('GRAPH STAGE1:')
    # for i in graph:
    #     print(i)
        
    column_insert_index = first_greater_x(graph, x) 
    for row_index, row in enumerate(graph): 
        cell_to_insert = [ x , graph[row_index][0][1] ] # graph[row_index][0][1] because graph[row_index] is a specific row that exists in graph, specific_row[0] is an actual cell, and actual_cell[1] is the y-coordinate of a specific position. We HAVE to use y-coordinate already existing in graph,bc we don't know whats been added to our graph.
        row.insert(column_insert_index, cell_to_insert)
    # print()
    # print('GRAPH STAGE2:')
    # for i in graph:
    #     print(i) 
           
def first_greater_x(graph, value):
    num_columns = len(graph[0])
    
    graph_xs = []
    row = graph[0] 
    for column_index in range(num_columns):
        graph_xs.append(row[column_index][0])
    # print('GRAPH XS:', graph_xs)
    for count, item in enumerate(graph_xs):
        if item>value:
            return count
    return count+1 
    
def first_greater_y(graph , value): # Returns index of first number in lst which is greater than value, or len(lst) if no greater number exists
    graph_ys  = [ row[0][1] for row in graph]
    # print('GRAPH_YS:', graph_ys)
    for count, item in enumerate(graph_ys): 
        if item>value: 
            return count
    return count+1 # Ok (?) to return index-out-of-range. insert() would merely append


insert_into_graph(normal_graph, position)
insert_into_graph(normal_graph , (2.5 , 1.1)) 
insert_into_graph(normal_graph , (2. , 1.0)) 
insert_into_graph(normal_graph , (.3, .4)) 


import numpy as np
import matplotlib.pyplot as plt
arr = np.array(normal_graph)
print('ARR:')
print(arr)
for row in arr: 
    plt.plot(row[:,0] , row[:,1], 'go')

# Hey there's gotta be a library that already does this--matrix insertion (?) matrix expansion(?) I can't find one tho


In [ ]:
# Useful?  np.vstack / np.hstack 
import numpy as np
 
# Create a 2D array
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
 
# Vertical stacking
vertical_extended = np.vstack((a, b))
print("Vertical extended:")
print(vertical_extended)
 
# Horizontal stacking
horizontal_extended = np.hstack((a, b))
print("Horizontal extended:")
print(horizontal_extended)

In [ ]:
print(round(8.8))
import math 
print(math.floor(8.9))

In [ ]:
l= [0,1,2,1,1,2,3] # Note a binary tree can be represented as a flat list, with some clever list sorting. This is called an 'implicit' data structure, because the position of the elements imply meaning. The heapq module implements binary trees as flat lists, as do most other heap algorithms. So a binary tree looks like a flat list [0,1,2,1,1,2,3] -- note this 'binary tree' is in sorted, in special binary tree order, not increasing numerical order.
# l = [0,2,1,[4,3]]  '<' not supported between instances of 'int' and 'list' -- heapq uses the '<' comparator to assemble the heap, but you cannot compar [0] < 3
import heapq 
heapq.heapify(l)
print(l)

# Q: Can Tuples be items on the heap? Yes-- zeroeth index is used to sort, second index ignored prettysure
l = [ (1,10) , (3,10), (0,10) ]
heapq.heapify(l)
print(l) # [(0, 10), (3, 10), (1, 10)]
# Yes they can! 

# Q: Can Second item of tuple be a dict? A: YES! 
l = [ (1,dict(a=1)) , (3,dict(a=1)), (0,dict(a=1)) ]
heapq.heapify(l)
print(l)
#Q: Can I skip having an open_dict AND a open_list and just store node in the heap, as the second item in the tuple? A: Yes, but, its not helpful.

In [ ]:
# Continue is only for 'for' loops  

In [ ]:
from sqlalchemy import * 
database_path = "parts/test.db"
# self.table_name = 'ceramic_capacitors'
# self.db_path = 'parts/parts.db'
# self.record = { "mpn":'123abc',  "A": 'a', "B":'b',}
_engine = create_engine(f"sqlite:///{database_path}")
metadata = MetaData()
table = Table('hi', metadata, Column('asdf', Text) )
metadata.create_all(_engine) # Create ss_filters table if not exists
        
stmt = insert(table).values(asdf = 'asdf')
print(stmt)
with _engine.begin() as connection:
    result = connection.execute(stmt)
    
print(result.rowcount)

stmt = update(table).values({'asdf':';lkj'}) # Update all cells under column'asdf' to ';lkj'
print()
print(stmt)
with _engine.begin() as connection:
    result = connection.execute(stmt)
    
print(result.rowcount)

stmt = update(table).where(table.c.asdf == ';lkj').values({'asdf': 'fdsa'}) # update all cells under column 'asdf' to 'fdsa', if their value at 'asdf' equals ';lkj'
print()
print(stmt)
with _engine.begin() as connection: 
    result = connection.execute(stmt)
    
print(result.rowcount)

In [ ]:
l = [ [0,1] , [2,3] ]

# print(l[: , 1]) # This doesn't work on lists, the comma means nothing
import numpy
l = numpy.array(l)
print(l)
print()
print(l[: , 1])# -> <class 'numpy.ndarray'> [1 3] # Note that slicing of 2D ARRAYS has a syntax that LOOKS like slicing a LIST, but you cannot slice a 2D list like you can slice a 2d array. This is kinda how pandas.loc[] works. Concept of : being your slice start:end:step and , delineating row_slice , column_slice
l[0][1] = 7
print()
print(l)



In [ ]:
# QTableWidget obscures a lot of text when text longer than cell-- how do i prevent obscuring of text? 
# QTableView has special HeaderITems, and QTableWidget inherits QTableView.
# QTableWidgetItem.setSizeHint(size:QSize) : Sets the size hint to size. Default, item delegate computes size hint, based on item data. 
# Ok... so, how do I get the item's delegate's size hint? 
# item.setSizeHint()
from PySide6.QtCore import Qt, QSize
from PySide6.QtWidgets import QApplication, QTableWidget, QTableWidgetItem
import sys 

app = QApplication(sys.argv)
table = QTableWidget(2,2)
item = QTableWidgetItem('asdfasdfasdfasdfasdfasdfasfd') # Here, text is longer than cell width, but ellipsis don't occlude too much... Why 
# item.setSizeHint(QSize(10,10)) No effect
# item.setSizeHint(QSize(100,100)) No effect
item.setSizeHint(QSize(1000,1000) )
table.setItem(0,0, item)
table.show()

sys.exit(app.exec())


In [ ]:
### SO Post on resizing tableWidget ### https://stackoverflow.com/questions/54612127/how-to-i-set-the-size-hint-for-a-qtablewidget-in-python
## Note usage of         tableWidget().setSizeAdjustPolicy(QAbstractScrollArea.AdjustToContents)

import sys
import pandas as pd
from PySide6.QtCore import *
from PySide6.QtWidgets import *
from PySide6.QtGui import *


class MyWin(QMainWindow):
    def __init__(self, df):
        super().__init__()

        centralWidget = QWidget()
        self.setCentralWidget(centralWidget)
        layout = QGridLayout(centralWidget)    

        self.tableWidget = self.build_table(df)
###
        self.tableWidget.setSizeAdjustPolicy(QAbstractScrollArea.AdjustToContents)

        self.tableWidget.setAlternatingRowColors(True)
        self.tableWidget.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)   

        layout.addWidget(self.tableWidget)
        layout.addWidget(QPushButton("Button"))        

    def build_table(self, df):

        table = QTableWidget()
        table.setColumnCount(len(df.columns))
        table.setRowCount(len(df.index))
        table.setHorizontalHeaderLabels(df.columns)

        for row_num, row in enumerate(df.index):
            for col_num, col in enumerate(df.columns):
                item = QTableWidgetItem(str(df.loc[row,col]))

                table.setItem(row_num, col_num, item)                              # +++

                item.setFlags(Qt.ItemIsSelectable | Qt.ItemIsEnabled)

        table.resizeColumnsToContents()
        table.resizeRowsToContents()    
        table.verticalHeader().setVisible(False)
#        self.table.setSizePolicy(QSizePolicy.Maximum, QSizePolicy.Maximum)        # ---

        return table                                                               # +++


df = pd.DataFrame(
    {'a': ['1','2','3','4','Mary','Jim','John'], 'b': ['3','4','1','2',100, 200, 300], 'c': ['1','2','3','4','a','b','c'],
     'd': ['1','2','3','4','Mary','Jim','John'], 'e': ['1','2','3','4',100, 200, 300], 'f': ['1','2','3','4','a','b','c'],
     'g': ['1','2','3','4','Mary','Jim','John'], 'h': ['1','2','3','4',100, 200, 300], 'j': ['1','2','3','4','a','b','c'],
     'k': ['1','2','3','4','Mary','Jim','John'], 'l': ['1','2','3','4',100, 200, 300], 'm': ['1','2','3','4','a','b','c'],
    })

if __name__ == '__main__':
    app = QApplication(sys.argv)
    w = MyWin(df)
    w.show()
    sys.exit(app.exec())    

In [ ]:
from urllib.parse import urlparse


url = 'hi' # Note that this would be placed in the 'parse_path' so we cannot check if a thing is a url via if parse.path.
url = "https://www.digikey.com/en/products/detail/murata-electronics/GRM21BR61E106KA73L/2334874"
o = urlparse(url)
print(o)
print()
print(o.path) # /en/products/detail/murata-electronics/GRM21BR61E106KA73L/2334874 
# The 'path' section STILL has too much info. 
if o.path.split('/')[:-2]: # If theres something in the second-to-last slash, maybe its the mpn: display it.
    ...

In [ ]:
d = dict(a=1)
print(d)
import os
print('C:\myFile.txt')
print('C:\my File.txt') # Thinks our file is 'C:\my' -- the spaces are ruining it.
print('"C:\my File"') # Now we've just created a string. Lets use os.path.join()
file = os.path.join('C:/', 'my File') # spaces still getting in the way?  oh-- it only SEEMS so bc the terminal prints it our like its wrong-- but I can use 'file' no problem.
print("FILE:" , file)

with open(file, 'w') as fo: 
    fo.write('hey')
    
with open(file, 'r') as fo: 
    print(fo.readlines())

print(os.path.join(r'C:\Users\robby\Downloads\lord-of-the-rings-soundtrack-hd-complete\Lord Of The Rings - Soundtrack HD Complete.mp4'))


In [ ]:
# ! pip install moviepy
from moviepy import VideoFileClip

import os 
file = os.path.join(r'C:\Users\robby\Downloads\lord-of-the-rings-soundtrack-hd-complete\LordOfTheRingsFullSoundtrack.mp4')
print('FILE:', file)
video_clip = VideoFileClip(file) # Error opening input file C:\Users\robby\Downloads\lord-of-the-rings-soundtrack-hd-complete\Lord Of The Rings - Soundtrack HD Complete.mp4

video_clip.audio.write_audiofile(os.path.join(r"C:\Users\robby\Downloads\TheLordOfTheRingsFullSoundtrack.mp3"))

In [ ]:
from sqlalchemy import * 

database_path = 'parts/parts.db'
engine = create_engine(f"sqlite:///{database_path}")
metadata = MetaData()
columns = [
        Column('id', Integer, primary_key = True, autoincrement = True, nullable = False),  
        Column("Col1", Text, ),
        Column("Col2", Text),
]
_table = Table('Table1', metadata, *columns)
metadata.create_all(engine) # Create tables if not exist
metadata.reflect(engine) # # Load all tables from database into the metadata object-- Call again to add tables newly added to database. Does not remove tables, if tables no longer in database 

# Make a statement(does not execute)
stmt = _table.select()
print(stmt)
# Execute a statement
with engine.begin() as conn: 
    result = conn.execute(stmt)
    if verbose: 
        print()
        print('RESULT:', result)

# Utilize result from statement
print("RESULT.FETCHONE():", result.fetchone()) # There's no data bc we never added data we only created a blank table

# Insert data into table
# Generate statement
stmt = _table.insert().values(Col2 = 'HeyThere') # key value pairs where key represents a column_name in the table. Note how Col1 isn't a string; sqlalchemy knows it codes for the column named 'Col1'
print(stmt)
# stmt = _table.insert().values({'Col1': 'HeyThere'}) Alt : may pass kv pair in dict form. Alt : may pass tuples if every column in table is provided a tuple.
# stmt = _table.insert().values(Col3 = 'a') # CompileError: Unconsumed column names: Col3 : I provided a column_name, Col3, that DNE.

with engine.begin() as conn: 
    result = conn.execute(stmt)
    print()
    print('RESULT.INSERTED_PRIMARY_KEY:', result.inserted_primary_key[0] ) # Return the primary key for the row just inserted. Why is ()? Oh, bc there wasn't a PK-- you have to specify a column as a PK, in sqlalchemy:         Column('id', Integer, primary_key = True, autoincrement = True, nullable = False) 

# Check on what we just inserted
stmt = _table.select()
with engine.begin() as conn: 
    result = conn.execute(stmt)
    print('RESULT.FETCHALL:', result.fetchall())
    # InvalidRequestError: Statement is not an insert() expression construct.

In [ ]:
### QT ESSENTIALS : Ur gonna have to use these ### 
QDialog vs QFileDialog.getOpenFileName


In [ ]:
d = {'a':1 , 'b':2}
s = ','.join(d)
print(s)
d.update(c =1 )
print(d)
d.update({'d':4})
print(d)


In [ ]:
# Admissible: Never overestimates 
# Consistent: ???
# Open Set: aka queue

# https://www.geeksforgeeks.org/python/a-search-algorithm-in-python/
# https://www.codegenes.net/blog/a-star-algorithm-in-python/

In [ ]:
# Another  https://www.codegenes.net/blog/a-star-algorithm-in-python/ 
graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('A', 1), ('D', 2), ('E', 5)],
    'C': [('A', 4), ('F', 3)],
    'D': [('B', 2)],
    'E': [('B', 5), ('F', 1)],
    'F': [('C', 3), ('E', 1)]
}

start_node = 'A'
goal_node = 'F'

import heapq

def heuristic(node, goal):
    # A simple heuristic (demo purposes)
    return 0 

def a_star(graph, start, goal):
    open_list = [] 
    heapq.heappush(open_list, (0 , start, []))
    closed_set = set()
    
    while open_list:
        _ , current_node, path = heapq.heappop(open_list)
        if current_node == goal: 
            return path + [current_node]
        if current_node in closed_set:
            continue
        closed_set.add(current_node)
        for neighbor, cost in graph[current_node]:
            new_path = path + [current_node]
            g = len(new_path)
            h = heuristic(neighbor, goal)
            f = g+h
            heapq.heappush(open_list, (f, neighbor , new_path))
            
    return None

path = a_star( graph, start_node, goal_node)
print('Path:', path)
    
# Grid-based Pathfinding: represent the grid, as a 2D list, where 0s are free space but 1s are obstacles
# Check if cell is an obstacle a 1, before considering it a valid neighbor 

def get_neighbor_positions(grid, current):
    rows, cols = len(grid), len(grid[0])
    x, y = current 
    neighbors = [] 
    directions = [ (0,1), (0, -1) , (1,0), (-1,0) ]
    for dx, dy in directions:
        new_x , new_y = x + dx, y + dy 
        if 0 <= new_x < rows and 0 <= new_y < cols and grid[new_x][new_y] == 0:
            neighbors.append( ( new_x , new_y) )
    return neighbors

    
    

In [ ]:
# How can I represent infinity in python? A: as a special kind of float
inf = float('inf')
print(inf)
neg_inf = float('-inf')
print(neg_inf)

# You can also use the math module esp to check if a number is infinite with math.isinf()
import math 
inf = math.inf
print(inf)
print(inf *5)
print(-inf +1)
print(math.inf is math.inf)
print(math.isinf(inf))

In [ ]:
### A* ALGORITHM ###
def reconstruct_path(cameFrom, current)
    total_path == cur
    while current in cameFrom.keys():
        current = cameFrom[current]
        total_path.prepend(current)
    return total_path

def A_Star(start, goal, h):
    openSet = {start}
    cameFrom = None
    gScore = {start:0} # map with default value of infinity
    fScore = {start:h(start)}
    
    while openSet is not empty: 
        current = min_f_score()
        if current = goal: 
            return reconstruct_path(cameFrom, current)
        openSet.remove(current)
        for neigbor in current:
            # d(current, neighbor) is weight
            # tentative_g_score is distance from start, to neighbor, through current
            tentative_g_score = gscore[current] + d(current, neighbor)
            if tentative_g_score < g_score[neighbor]: 
                cameFrom[neighbor = current]
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + h(neighbor)
                if neighbor not in openSset:
                    openSet.add(neighbor)
        return failure # open set is empty but goal was never reached 
    
# In this pseudocode, ifa node is reached by a path, removed from openSet, but then reached by a cheaper path, it will again be added to openSet. This is 
    
def min_f_score(d:dict):
    min_f_score = None
    min_f_score_node = None
    
    for node,f_score in d.items():
        if min_f_score is None: 
            min_f_score = f_score()
            min_f_score_node = node
        else:
            if f_score < min_f_score: 
                min_f_score = f_score
                min_f_score_node = node
    return min_f_score_node
        
                
                   
# How to represent infinity in python?

In [ ]:
# HOw to select key with lowest value in dict? 
d = {}
print(d.items())
d = { 'c':3 , 'a':1 , 'b':2}

#d.items()
#d.keys()
#d.values()
_min = 100 
key = None

# for k,v in d.items():
#     if v < _min: 
#         _min = v 
#         key = k 
# print(key)
# print(_min)

_min = sorted(d, key = lambda k: d.get(k))[0]
_min = d.get(_min)
print()
print(_min)




In [ ]:
### Algorithms/techniques for routing traces in board mode ##
# https://stackoverflow.com/questions/6789643/a-for-finding-shortest-path-and-avoiding-lines-as-obstacles#6793562
# A) draw a line(my rout)while avoiding shapes(footprints, keep out zones, etc) 


# Usage of Kicad's PCBNEW api: its cool: https://girishji.github.io/2022/08/17/kicad-python-footprints-curved-tracks-edge-cuts.html

In [ ]:
# For a long time, when using their API, I was NOT getting bot checked. As of today, I now am getting bot checked when using dk api. 
# Try changing 'user agent'. 
mpn = "C503D-WAN-CCBEB152"
url = f"https://api.digikey.com/products/v4/search/{mpn}/productdetails"
url = "https://api.digikey.com/products/v4/search/C503D-WAN-CCBEB152/productdetails"

import requests 
r = requests.get(url)
print('R.REQUEST.HEADERS:', r.request.headers) # {'User-Agent': 'python-requests/2.32.3', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}
# We can see that the 'requests' library default uses a user agent with 'python-requests' in the name, which is probably a red flag we should hide.
#  You can copy your browser's user agent. in firefox, type 'about:support' in the search window. Accept the warning. Copy the 'User Agent' value, mine is : Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:143.0) Gecko/20100101 Firefox/143.0 
# Now, set your request's user agent to your browser's user agent, using it's dict-like interface:
r.request.headers.update({ 'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:143.0) Gecko/20100101 Firefox/143.0'})
print('R.REQUEST.HEADERS:', r.request.headers) # {'User-Agent': 'python-requests/2.32.3', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}
r = requests.get(url, r.request.headers)
print(r.ok) # Fails, but this is expected, because I never included the needed headers, AND even then those headers need access token and special settings...


In [ ]:
l = ['A'] 
for c, i in enumerate(l):
    print()
    print("C:", c)
    print('I:', i)

In [ ]:
import pandas 
df = pandas.DataFrame([[0,1,2], [3,4,5]], columns = ['A', 'symbol', 'footprint'], index = ['a','b'])
df = df.loc[ df['A'] >1 ]
print(df['A']) # Was I getting warnings about accessing columns with [] syntax? No?
print(df)

In [ ]:
# Notice how I can't set data in on QTableWidget--Instead I must set data on a bunch of QTableWidgetItems, then slap those into table at the right location

# # QTABLE WIDGET  AND SLOTS 

# current(Cell,Item)Changed emits(CurrentRow, currentCol, previousRow, previousCol)
# (cell,item)(Activated, Changed, Clicked, DoubleClicked, Entered, Pressed)emits(row, col)
# itemSelectionChanged emitsNone

# activated: when you tab over a cell then press enter (windows) (Not so useful to me)
# clicked: mouse down&up 
# pressed: mouse down (can be used if dragging)

self.cellClicked.connect(self.on_cell_clicked)
self.cellActivated.connect(self.on_cell_activated)
self.currentCellChanged.connect(self.on_current_cell_changed)
self.itemSelectionChanged.connect(self.on_item_selection_changed)


#Most Useful Signals
QTableWidget.(cell,item)(Activated, Clicked, Changed)(row,col)
QTableWidget.current(Cell,Item)Changed(currRow, currCol, prevRow, prevCol)


In [ ]:
# In this example, I .connect to every? Signal of QTableWidget, such as to print a message such that I can see what events happen when. 
from PySide6.QtCore import Qt 
from PySide6.QtWidgets import QApplication, QTableWidget, QTableWidgetItem
from PySide6.QtGui import QIcon, QPixmap
import sys

class MyTableWidget(QTableWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        
        self.cellActivated.connect(self.on_cell_activated) 
        self.cellClicked.connect(self.on_cell_clicked)
        self.cellChanged.connect(self.on_cell_changed)
        self.cellDoubleClicked.connect(self.on_cell_double_clicked)
        self.cellEntered.connect(self.on_cell_entered)
        self.cellPressed.connect(self.on_cell_pressed)
        
        self.itemActivated.connect(self.on_item_activated) 
        self.itemClicked.connect(self.on_item_clicked)
        self.itemChanged.connect(self.on_item_changed) 
        self.itemDoubleClicked.connect(self.on_item_double_clicked)
        self.itemEntered.connect(self.on_item_entered)
        self.itemPressed.connect(self.on_item_pressed)
        
        self.itemSelectionChanged.connect(self.on_item_selection_changed)
        self.horizontalHeader().clicked.connect(self.on_horizontal_header_clicked) # Doesn't do anything
        
        num_rows, num_cols = 2,4
        self.setRowCount(num_rows)
        self.setColumnCount(num_cols)
        table_item = QTableWidgetItem('setText') # Note we'll add items to table via .setItem() not via setting a parent. Table Items hold text, mostly, then checkboxes &icons.

        row, col = 1,1
        self.setItem(row, col, table_item) 
        #.set(Horizontal,Vertical)HeaderLabels() ; create simple text headers for the table's columns and rows.  
        self.setHorizontalHeaderLabels(['id', 'name', 'type', 'four'])
                
        # Advanced Tips 
        # Enable sorting, if desired, only after table is populated.(sorting may interfere with insertion order/ setItem())
        # Create fancy headers or cells, such as with aligned text or icon, by making dedicated header items: 
        mushroomHeaderItem = QTableWidgetItem('Mushroom')
        mushroomHeaderItem.setIcon(QIcon(QPixmap(".//Images/BigMushroom.png")))
        mushroomHeaderItem.setTextAlignment(Qt.AlignVCenter)
        self.setItem(0,0,mushroomHeaderItem)
        self.setHorizontalHeaderItem(0,mushroomHeaderItem)

    def mouseDoubleClickEvent(self, event):
        item = self.itemAt(event.position().toPoint())
        print('ITEM:', item)
        if item is None:
            return 
        row_idx = item.row()
        column_idx = item.column()
        print("HorHead")
        column = self.horizontalHeaderItem(column_idx).text()
        if column == 'Mushroom':
            self.mushroom_delegate = MushroomDelegate() # How do I superimpose a delegate on my table_widget? 
            Goal: User edits 'reference' column in-place on the table, and that change propogates to the database. 
            
        
    def on_cell_activated(self, row, col): 
        print(f"Cell Activated: {row,col}")
    def on_cell_clicked(self, row, col): 
        print(f"Cell Clicked: {row,col}")
    def on_cell_changed(self, row, col): 
        print(f"Cell Changed: {row,col}")
    def on_cell_double_clicked(self, row, col):
        print(f"Cell DoubleClicked: {row, col}")
    def on_cell_entered(self, row, col):
        print(f"Cell Entered: {row, col}")
    def on_cell_pressed(self, row, col):
        print(f"Cell Pressed: {row, col}")
        
    def on_item_activated(self, item): 
        print(f"Item Activated: {item}")
    def on_item_clicked(self, item): 
        print(f"Item Clicked: {item}")
    def on_item_changed(self, item): 
        print(f"Item Changed: {item}")
    def on_item_double_clicked(self, item):
        print(f"Item DoubleClicked: {item}")
    def on_item_entered(self, item):
        print(f"Item Entered: {item}")
    def on_item_pressed(self, item):
        print(f"Item Pressed: {item}")
        
    def on_item_selection_changed(self):  # Q why this emits no args? 
        print(f"Item Selection Changed:")

            
    def on_horizontal_header_clicked(self):
        print('HorizontalHeaderClicked:')
app = QApplication(sys.argv) # Gotta create the application instance before ur widgets table.show()

table = MyTableWidget()
table.show()
sys.exit(app.exec())


        
    

In [ ]:
#In this example, I practice using a pandas.self.dataframe with a QTableWidget. 
# I want # In this example, I .connect to every? Signal of QTableWidget, such as to print a message such that I can see what events happen when. 
from PySide6.QtCore import Qt 
from PySide6.QtWidgets import QApplication, QTableWidget, QTableWidgetItem
from PySide6.QtGui import QIcon, QPixmap
import sys
import os 
import pandas 
from utils import Utils 
from MyGraphicAssign import MyGraphicAssign
from MyDatabase import database # import my database singleton 

class MyTableWidget(QTableWidget):
    def __init__(self, table_name, parent=None):
        super().__init__(parent)
        
        dataframe =database.get_df(table_name)
        print()
        print('DATAFRAME:', dataframe)
        self.table_name = table_name 
        self.setDataframe(dataframe)
        
        self.cellActivated.connect(self.on_cell_activated)
        self.cellClicked.connect(self.on_cell_clicked)
        self.cellChanged.connect(self.on_cell_changed)
        self.cellDoubleClicked.connect(self.on_cell_double_clicked)
        self.cellEntered.connect(self.on_cell_entered)
        self.cellPressed.connect(self.on_cell_pressed)
        
        self.itemActivated.connect(self.on_item_activated) 
        self.itemClicked.connect(self.on_item_clicked)
        self.itemChanged.connect(self.on_item_changed) 
        self.itemDoubleClicked.connect(self.on_item_double_clicked)
        self.itemEntered.connect(self.on_item_entered)
        self.itemPressed.connect(self.on_item_pressed)
        
        self.itemSelectionChanged.connect(self.on_item_selection_changed)
        self.horizontalHeader().clicked.connect(self.on_horizontal_header_clicked) # Doesn't do anything
        

        
    def on_item_selection_changed(self):  # Q why this emits no args? 
        print(f"Item Selection Changed:")

            
    def on_horizontal_header_clicked(self):
        print('HorizontalHeaderClicked:')
        
    def dataframe(self):
        return self._dataframe
    
    def setDataframe(self, dataframe): 
        self._dataframe = dataframe
# Wrangle the tableWidget to display the dataframe
        num_rows, num_cols = self.dataframe().shape
        self.setRowCount(num_rows)
        self.setColumnCount(num_cols)
        if not dataframe.columns.empty:
            self.setHorizontalHeaderLabels(dataframe.columns)
        if not dataframe.index.empty:
            self.setVerticalHeaderLabels(dataframe.index)
        for row in range(num_rows):
            for column in range(num_cols):
                value = str(dataframe.iloc[row, column]) 
                
                if os.path.exists(str(value)): # If this is a path don't show the full path, but the root
                    head,tail = os.path.split(value)
                    print('df value is a path that exists')
                    root, ext = os.path.splitext(tail)
                    table_item = QTableWidgetItem(root) # Only display the name of the file 
                else:
                    table_item = QTableWidgetItem(value) # Note we'll add items to table via .setItem() not via setting a parent. Table Items hold text, mostly, but also checkboxes &icons.
                self.setItem(row, column, table_item)

    def reorder_columns(self, filter:str):
        filter_columns = database.get_filter( self.table_name, filter) 
        # We got a new column order but we still want to show columns not included in the new_column order, tack them onto the end 
        all_columns = self.dataframe().columns
        filter_columns.extend( [ c for c in all_columns if not c in filter_columns])
        print('NEW_COLUMN_ORDER:', filter_columns)
        self.setDataframe( self.dataframe().loc[:, filter_columns] ) 

    def on_cell_activated(self, row_idx, col_idx): 
        print(f"Cell Activated: {row_idx,col_idx}")
    def on_cell_clicked(self, row_idx, col_idx): 
        print(f"Cell Clicked: {row_idx,col_idx}")
    def on_cell_changed(self, row_idx, col_idx): 
        print(f"Cell Changed: {row_idx,col_idx}")
    def on_cell_double_clicked(self, row_idx, col_idx):
        print(f"Cell DoubleClicked: {row_idx, col_idx}")
    def on_cell_entered(self, row_idx, col_idx):
        print(f"Cell Entered: {row_idx, col_idx}")
    def on_cell_pressed(self, row_idx, col_idx):
        print(f"Cell Pressed: {row_idx, col_idx}")
        
    def on_item_activated(self, item): 
        print(f"Item Activated: {item}")
    def on_item_clicked(self, item): 
        print(f"Item Clicked: {item}")
    def on_item_changed(self, item): 
        print(f"Item Changed: {item}")
    def on_item_double_clicked(self, item):
        print(f"Item DoubleClicked: {item}")
        row_idx, col_idx = item.row(), item.column() # 
        
        # print('Col:', self.horizontalHeaderItem(col_idx).text()) Dont bother using the QT api
        col = self.dataframe().columns[col_idx].lower().strip().strip(':')
        print('Col:', col)
        if col in writeable_columns: 
            part = self.dataframe().iloc[row_idx].to_dict()
            graphic_assign = MyGraphicAssign(part,col, row_idx, parent=self) # GA parented on self, such that we can : ga.parent().dataframe().loc[row_idx,col] = selected_file
            graphic_assign.open()
            # In g_a.accept(), we update database, we setDataframe, and  
            
    def on_item_entered(self, item):
        print(f"Item Entered: {item}")
    def on_item_pressed(self, item):
        print(f"Item Pressed: {item}")



app = QApplication(sys.argv) # Gotta create the application instance before ur widgets table.show()
# df = pandas.DataFrame([[0,1,2], [3,4,5]], columns = ['A', 'symbol', 'footprint'], index = ['a','b'])
table_name = database.get_all_table_names()[1]

# self.reorder_columns(database.get_table('ss_filters')) # too hard to get 

# print(df.index.empty) # See if df has a set index
# df['B'].iloc[0] = 'HiThere' # set a new value to the dataframe # DONT USE THIS SYNTAX! It will be depreceated in pandas 3.0-- use .loc[] instead! 
# df.loc[0, 'B'] = 10# To set a new value to the dataframe, you should use .loc[row , col] But, note that new value should be of same type as oldvalue. 
# This is weird-- since I set index, pandas uses that index to lookup rows-- thus there is no row '0' anymore, its row 'a'-- and the call to df.loc[0,'B'] = 10  causes a new row, with row_idx = 0, to be appended to the df.
# df['B'] = 0 # Set whole column to a new value

table = MyTableWidget(table_name)
table.reorder_columns('general_attributes')
table.show()
sys.exit(app.exec())


        
    

In [ ]:
from PySide6.QtWidgets import QTableWidget, QTableWidgetItem, QApplication# QTABLEWIDGET
import sys 
from PySide6.QtCore import Qt
from PySide6.QtGui import QIcon, QPixmap


app = QApplication(sys.argv) # Gotta create the application instance before ur widgets 

table =QTableWidget()
num_rows, num_cols = 2,4
table.setRowCount(num_rows)
table.setColumnCount(num_cols)
table_item = QTableWidgetItem('setText') # Note we'll add items to table via .setItem() not via setting a parent. Table Items hold text, mostly, then checkboxes &icons.

row, col = 1,1
table.setItem(row, col, table_item) 
#.set(Horizontal,Vertical)HeaderLabels() ; create simple text headers for the table's columns and rows.  
table.setHorizontalHeaderLabels(['id', 'name', 'type', 'four'])
        
# Advanced Tips 
# Enable sorting, if desired, only after table is populated.(sorting may interfere with insertion order/ setItem())
# Create fancy headers, suchas with aligned text and icon, by making dedicated header items: 
mushroomHeaderItem = QTableWidgetItem('Mushroom')
mushroomHeaderItem.setIcon(QIcon(QPixmap(".//Images/BigMushroom.png")))
mushroomHeaderItem.setTextAlignment(Qt.AlignVCenter)
table.setItem(1,3,mushroomHeaderItem)

table.show()

sys.exit(app.exec())
# .rowCount() 
# .columnCount()
# .clear()

# QTableView, QTableWidget, QTableWidgetItem, QTableWidgetSelectionRange, 
# MyDatabase(capacitors)

In [ ]:
from PySide6.QtCore import Qt
for i in Qt.ItemDataRole:
    print(i)

In [ ]:
# with MyMainWindow.spreadsheet.database, I was having a hard time ACESSING database-- it turns out, I wanted db access from top-level dialogs, this involved long parent chains and/or creating an object in MyScene.dropEvent(such as to access event.positon()), then emitting it to be picked up in MyMainWindow(such as to item.connect(launch_dialog)(since dialog designed to have mmw as parent, such that dialog would have access to mmw.schematic.database....)(Not a reliable workflow...) 
# Then, I had the brilliant idea to move database BEHIND all the other code, such that MyGraphicAssign dialog has access to the database object:
# https://sqlpey.com/python/top-4-ways-to-implement-singleton-pattern-in-python/ One of the simplest ways to create a singleton in Python is through modules. Since Python modules are singletons by default(???), you can define functions and variables within a module instead of a class, effectively making it a singleton.

# singleton_module.py
# class SingletonModule:
#     def __init__(self):
#         self.value = None

#     def set_value(self, value):
#         self.value = value

#     def get_value(self):
#         return self.value

# singleton_instance = SingletonModule()

class Database(QObject):
    def __init__(self, database_path):
        super().__init__()
        self.setDatabasePath(database_path)

    def setDatabasePath(self, database_path):
        self._database_path = database_path
    def databasePath(self):
        return self._database_path
    
database = Database()

# in other files: 
from MyDatabase import database # import the singleton instance 'database'


In [ ]:
import pandas 
df = pandas.DataFrame([[1,2], [3,4]], columns = ['a','b'] , index= ['A', 'B'])
print(df)

print()
print(df.index.get_loc('A'))
print(df.columns.get_loc('a'))
# print(df.get_loc(1)) AttributeError: 'DataFrame' object has no attribute 'get_loc' # Can't do it on the dataframe, only works on series;flat lists.


In [ ]:
class Snake():
    def __init__(self, parent=None):
        self.parent = parent
        if isinstance(parent, Worm):
            print('type Worm invalid as Snake parent')

class Worm():
    def __init__(self, parent=None):
        self.parent = parent 
        if isinstance(parent, Snake): 
            print('type Snake invalid as Worm parent') # The order in which classes are defined does not matter-- See Snake and Worm are able to access each other
        
worm = Worm()
snake = Snake(worm)
worm = Worm(snake)





In [ ]:
from PySide6.QtWidgets import QGraphicsItem,QGraphicsEllipseItem, QGraphicsScene, QApplication
from PySide6.QtCore import QObject

app = QApplication()
item = QGraphicsEllipseItem(-10,-10,20,20)
scene= QGraphicsScene()
scene.addItem(item)
print(scene.children())


print(QGraphicsItem.mro())
print(QGraphicsScene.mro())


In [ ]:
if 'x' in 'xxx':
    print('true')

In [ ]:
from PySide6.QtGui import QTransform, QPen, QFont , QDrag, QBrush
from PySide6.QtCore import QRectF, Qt, QMimeData
from PySide6.QtWidgets import QGraphicsItem, QApplication, QGraphicsView, QGraphicsScene, QWidget, QVBoxLayout, QGraphicsRectItem, QGroupBox, QLabel, QMainWindow, QPushButton
import sys 
import os 
# child widgets appear inside their parents. top level widgets appear floating in their own window. QDialogs are always top level widgets. Q: I kinda wanted a QWidget that would appear floating, like a toplevel, while being parented on another widget but I guess that doesn't exist/ thats what a QDialog is... 
# I could use aQDialog, or use QWidget at a top level(just can't be a function-local variable-- gotta return it) 

app = QApplication(sys.argv)
window = QMainWindow()
window.show()

def on_button_clicked():
    print('Button was clicked')
    print(window.layout())
    print(window)
    # window2 = QWidget() # Will create a invisible window-- not useful
    # window2.show() # Will show the window-- which is immediatedly then destroyed. ( Because button2 is local to this function, when the function is done, it will be garbage collected by python) 
    button3 = QPushButton('PushMe3', window)  # Have window take ownership of button3; tether button3 to window, such that button3 won't get garbage collected once the function has finished.
    button3.show() # Now that button3 is parented on window, we still need to .show() button3 or it'll be invisible. 
    # window.setCentralWidget(button3) # button3 is parented on a QMainWidget. It shows transparently through the .centralWidget. Q: Where is centralWidget placed on the layout()? 
    # window.setCentralWidget(button2)
    button2.setParent(window)
    button2.show() 
    print(window.children()) # Button3 is on top of button2 but we can see they're both present in .children(). 
    
button = QPushButton('PushMe')
button2 = QPushButton('PushMe2')
button.clicked.connect(on_button_clicked) # We can't really make use of a return value 

button.show()
button2.show() # No problems with showing two top-level, individual windows... 
# window.setCentralWidget(button)

# window.show()
sys.exit(app.exec())

In [ ]:
x = 1 
def func():
    x = 2  # I made a variable. But its inside a function, so it doesn't affect x outside the function
    
def func(): 
    x = 2 
    return x 
func() 
print(x) # The function may have returned 'x', but I did not assigned the return value to a variable, so it was 'lost'


    
x = func() # I overwrite x with the value returned by func(). 
print(x)


In [ ]:
from PySide6.QtGui import QTransform, QPen, QFont , QDrag, QBrush
from PySide6.QtCore import QRectF, Qt, QMimeData
from PySide6.QtWidgets import QGraphicsItem, QApplication, QGraphicsView, QGraphicsScene, QWidget, QVBoxLayout, QGraphicsRectItem, QGroupBox, QLabel
import sys 
# from MyCentralWidget import MyCentralWidget

class DragSource(QWidget):
    def __init__(self):
        super().__init__()
        self.start = None 
        
    def mousePressEvent(self, event):
        print('Source.mousePress')
        self.start = event.position().toPoint()
        
    def mouseMoveEvent(self, event):
        print('Source.mouseMoveEvent')
        drag = QDrag(self)
        mime= QMimeData()
        mime.setText("HiThere")
        drag.setMimeData(mime)
        drag.exec()

class DragTarget(QWidget):
    def __init__(self): 
        super().__init__()
        self.setAcceptDrops(True) # Allow this QWidget to accept drops
        self.scene = QGraphicsScene()
        self.scene.setBackgroundBrush(QBrush(Qt.red))
        self.view = QGraphicsView()
        self.view.setScene(self.scene)
        layout = QVBoxLayout() 
        layout.addWidget(self.view)
        self.setLayout(layout)
        self.label = QLabel('NothingYet')

        
    def dragEnterEvent(self, event):
        print('Target.DragEnterEvent')
        # print(event.proposedAction()) # DropAction.MoveAction
        # event.acceptProposedAction() 
        # To allow this widget to receive further DragMoveEvents and DropEvents, the DragEnterEvent MUST be accepted; if you like the event.proposedAction you can event.acceptProposedAction() or event.accept(). You may also pick a different action with event.setProposedAction() then event.accept(). I have no idea what the proposedactions... do ...? Note that QGraphicsScene and QGraphicsView default have a dragndrop implementation; they handle the event.accept() part, so their DragNDrop api looks a bit different
        event.acceptProposedAction() # 
        # event.accept()
        
    def dragMoveEvent(self, event):
        print('Target.dragMoveEvent')
        
    def dropEvent(self, event):
        layout = self.layout()
        if self.label: 
            layout.removeWidget(self.label)
        self.label = QLabel(event.mimeData().text())
        layout.addWidget(self.label)

app = QApplication(sys.argv)
# cw = MyCentralWidget()
source = DragSource()
source_layout = QVBoxLayout()
source_layout.addWidget(source)
source_gb = QGroupBox('Source')
source_gb.setLayout(source_layout)

target = DragTarget()
target_layout = QVBoxLayout()
target_layout.addWidget(target)
target_gb = QGroupBox('Target')
target_gb.setLayout(target_layout)

layout = QVBoxLayout()
layout.addWidget(source_gb, 1)
layout.addWidget(target_gb, 1)

window= QWidget()
window.setLayout(layout)
window.resize(600,600)
window.show()

sys.exit(app.exec())

In [ ]:
import json 

x = json.dumps({'a':1})
print(x)
json

In [ ]:
###QTRANSFORM### 
# For use in rendering 
#CONVENIENCE FUNCTIONS:#
# .ROTATE()
# .TRANSLATE()
# .SHEAR()
# .SCALE() 
#MATRIX OPERATIONS# 
from PySide6.QtGui import QTransform, QPen, QFont
from PySide6.QtCore import QRectF, Qt
from PySide6.QtWidgets import QGraphicsItem, QApplication, QGraphicsView, QGraphicsScene

class MyItem(QGraphicsItem):
    def __init__(self, parent=None):
        super().__init__(parent)
    def boundingRect(self):
        return QRectF(-50,-50,100,100)
    def paint(self, painter, option, widget): # This is where QTransform comes in handy(?)
        painter.drawRect(self.boundingRect())
        # transform = QTransform()
        # transform.translate(50,50) # Build your transform
        # transform.rotate(45)
        # transform.scale(.5,1)
        # painter.setTransform(transform)
        # transform.reset() # Reset your transform to Identiry matrix 

        painter.setFont(QFont('Helvetica', 24))
        painter.setPen(QPen(Qt.black, 1))
        painter.drawText(20,10, "QTransform")
        
import sys 
app = QApplication(sys.argv)
scene = QGraphicsScene()
# scene.addItem(MyItem())
view = QGraphicsView()
print('VIEW.VIEWPORT:', view.viewport() ) # A QWidget
print('VIEW.VIEWPORT.RECT():', view.viewport().rect()) # PySide6.QtCore.QRect(0, 0, 638, 478). Not there is a tiny border, (the frame?) which is good bc it makes my viewport rect visible
scene.addRect(view.viewport().rect())

# scene.addEllipse(QRectF(-2,-2,4,4))
view.setScene(scene)
view.show()
sys.exit(app.exec())

In [ ]:
l = [1,2] 
print(l*10)
print('hey') if -3 else print('No')
from PySide6.QtCore import QLineF
line = QLineF(20,20,20,20)
print(line.length())


In [ ]:
from PySide6.QtCore import QPointF , QPoint
from utils import Utils

p1 = QPointF(1,2)
p2=  QPointF(240.6, 305)

p3 = p1 + p2 
# print(p3)

p4 = p3.toPoint()
# print(p4)

def snap_to_grid(point: QPointF | QPoint):
    grid = grid
    return QPoint( round(point.x()/grid)*grid , round(point.y()/grid)*grid ) 

p5 = snap_to_grid(p3)
p6 = snap_to_grid(p4)

print(p5)
print(p6)


In [ ]:
# for i in range(4):
#     print(i)
# for i in range(0,4): 
#     print(i)
for i in range(0,4,2): # if using range()'s step parameter, have to give all three args 
    print(i)

In [ ]:
#Comparison of floats 
print(5.0 == 5.00000000000000) # True 
print(5.0 == 5.000000000000001) # False 
print(5.0 == 5.0000000000000001) # True  So there is a limit, 15 decimals out




In [ ]:
# super(): Returns an object representing the parent class 
class A: 
    def age(self):
        print("age is 10")    
            
class B:
    def age(self):
        print("age is 30")    
        
class C(A,B): 
    def age(self):
        super().age()
        
c = C()
print()
print(C.mro()) # C, A, B
c.age() # 10


class C(B,A):
    def age(self):
        super().age() # Equivalent to super(C, self).age()

c = C()
print()
print(C.mro())# C, B, A 
c.age()

In [ ]:
class Rectangle:
    def __init__(self, length, width):
        self.length = length
        self.width = width 
        
    def area(self):
        return self.length * self.width 
    
    def perimeter(self):
        return 2*self.length*2*self.width 
    
class Square(Rectangle):
    def __init__(self, length): 
        super().__init__(length, length) # Equivalent to super(Square, self) 
        
class Cube(Rectangle):
    def surface_area(self):
        face_area = super().area()  # Causes usage of RECTANGLE'S super().area method. Equivalent to super(Cube, self).area(). NOTE that fiddling with super()'s parameters is not reccomended and could indicate a design issue
        face_area = super(Square, self).area() #Causes usage of Square's super .area() method
        return face_area *6 
    def volume(self):
        return self.length^3
    
super(subclass, instanceOfSubclass)



        


In [ ]:
# https://stackoverflow.com/questions/37736412/set-qgraphicstextitem-text-contents-of-exact-height-and-width
from PySide6.QtGui import QFont, QFontMetricsF

font = QFont("times", 24)
fm = QFontMetricsF(font)
pixelsWide = fm.horizontalAdvance("What's the advance width of this text?")
pixelsHight = fm.height() # vs .capHeight(), which returns the height of flat captial letters like "H", curved tops like "O" may display overshoot


In [ ]:
# Qt's default font size is the system's default font size. Generally, thats 12 pt font. 
# What is a point size ? A point is 1/72 of an inch. There is a unit called 'em' too 

from PySide6.QtGui import QFont, QFontMetricsF
from PySide6.QtWidgets import QApplication, QGraphicsSimpleTextItem, QGraphicsScene, QGraphicsView, QGraphicsItem
import sys 
from MySchematicScene import MySchematicScene 
from MyView import MyView

app = QApplication(sys.argv)
scene= MySchematicScene()

text = QGraphicsSimpleTextItem('Default font text')
text.setFlags(QGraphicsItem.ItemIsMovable|QGraphicsItem.ItemIsSelectable)
scene.addItem(text)

text2 = QGraphicsSimpleTextItem('24 Pt Text arial ')
text2.setFlags(QGraphicsItem.ItemIsMovable|QGraphicsItem.ItemIsSelectable)
font2 = QFont('arial', 24)
text2.setFont(font2)
scene.addItem(text2)


text3 = QGraphicsSimpleTextItem('1 pt Text helvetica')
text3.setFlags(QGraphicsItem.ItemIsMovable|QGraphicsItem.ItemIsSelectable)
font = QFont("helvetica", 1)
text3.setFont(font)
scene.addItem(text3)

view = MyView()
view.setScene(scene)
view.show()

sys.exit(app.exec())

In [ ]:
s = 'hey'
print(s.upper())
print(s.capitalize())
class MyClass(str):
    s = 'hey'
    
c = MyClass()
print(type(c))
    
    
print(type(c) == str)
print(type(c) == MyClass)

def recurse(lst, desc=[]):
    for i in lst: 
        if isinstance(i, list):
            recurse(i, desc)
        else: 
            desc.append(i)
    return desc

l = recurse( [ [1,2, [4,5,6]] , 3] )
print(l)
    
    

In [ ]:
import lxml.etree as etree 

footprint = etree.Element('footprint')
footprint.text = 'Hi' # Hey why does text mess with pretty_print? 

child1 = etree.SubElement( footprint, 'child1', key='value')
print(etree.tostring(footprint, encoding=str, pretty_print = True))


<footprint>Hi<child1 key="value"/></footprint>

<Element footprint at 0x26ab7b62dc0>
<footprint>Hi<child1 key="value"/></footprint>



In [ ]:
# Run code in cmd line from jupyther with exclamation mark: ! pip install gerber_writer  ! pip install pygerber
from gerber_writer import DataLayer, Circle, RoundedRectangle
import os 

trace_width = 0.127
via_pad = Circle(0.508, 'ViaPad')
IC17_toe = RoundedRectangle(1.257, 2.286, 0.254, 'SMDPad,CuDef')
toe_point = (0, 2.54)
via_point = (5.08, 0)

top = DataLayer('Copper,L1,Top,Signal')

top.add_pad(IC17_toe, toe_point, angle=45)
top.add_trace_line(toe_point, (2.54, 0), trace_width, 'Conductor')
top.add_trace_line((2.54, 0), via_point, trace_width, 'Conductor')
top.add_pad(via_pad, via_point)

path = os.path.join('gerbers')# ,' gerber_writer_example_small.gbr') 
if not os.path.exists(path):
    os.makedirs(path)
    
with open('gerbers\gerber_writer_example_small.gbr', 'w') as outfile:
    top.dump_gerber(outfile)
    

In [ ]:
# # from MyKicadSymbolConverter import MyKicadSymbolConverter
# import os 
# from utils import Utils 
# from lxml import etree
# # converter = MyKicadSymbolConverter( os.path.join('third_party', 'kicad', 'symbols', 'LTST-C190GKT.kicad_sym') )
# # converter.symbol_to_xml()
# # converter.format_graphics()
# # converter.save()
# # # (third_party\kicad\symbols\LTST-C190GKT.kicad_sym

symbol = "LTST-C190GKT.sym"
# # with open( os.path.join(symbols_path, symbol) ) as fo: 
# #     lines = fo.readlines()
# #     print(''.join(lines))
    

tree = etree.parse( os.path.join(symbols_path, symbol))
print(type(tree) , etree.tostring(tree, encoding = str , pretty_print=True))
root = tree.getroot() # etree.parse -> ElementTree object(which has no .iterdescendants) BUT etree.ElementTree.getRoot() -> etree.Element object, which is the root of the ElementTree, and has .iterdescendants.
for desc in root.iterdescendants(): 
    # print(desc)
    if desc.tag == 'pin': 
        desc.getparent().remove(desc) # Remove elem from root. Note That (bc pointers) root is affected 

print()
print(etree.tostring(root, encoding = str , pretty_print=True)) # No more 'pin' elements!


In [ ]:

import datetime
import pandas

df = pandas.DataFrame([ ['a'] ], columns = ['symbol'])

print(df.iloc[0].to_dict())



In [ ]:
import pandas 
data = pandas.DataFrame([
[-1,-2,-3, -4], # invalid data; for testing
["123abc", 'digikey', 'kyocera', 'capacitor_unipolar'],
], columns = ['mpn', 'vendor', 'mfr','symbol'])#, index=['Row 1', 'Row 2'])

# print(data)
# print()
# print(data.columns.get_loc('symbol'))
# data = data.loc[:,['mpn', 'vendor', 'symbol', 'mfr']]
# print(data)

print(data['vendor'] , type(data['vendor'])) # is a series; basically a list. The df[] syntax takes columns as first arg (confusingly, df.loc[] syntax takes rows, in first arg)
print()
print(data['vendor'] == -2 , type(data['vendor'] == -2)) # Is a boolean series, a boolean list ( so not useful by itself(as returns no data) but can be used in .loc[] )
print()
print(data.loc[ (data['vendor'] == 'digikey') ]) # got the rows where 'vendor'== 'digikey' 
# if your primary key is (mpn vendor mfr): 
mpn = '123abc'
vendor = 'digikey'
mfr = 'kyocera'
result = data.loc[     (data['vendor'] == vendor)   &   (data['mfr'] == mfr)   &   (data['mpn'] == mpn)   ] # I need the row number out of this, tho 
row = result.iloc[0] # Get first row only. We just became a series; list, our df.columns just became our series.index; our df.index just became our series.name... confuzing
print()
print("ROW:")
print(row)
print("ROW.NAME: AKA ROW_IDX:")
print(row.name)
row_idx = result.iloc[0].name # THIS IS HOW YOU GET ROW IDX FROM A DATAFRAME , so as to use it in model.setData()... gotta be an easier way 




In [ ]:
# The truth value of a pandas Index is undefined; you CANNOT use 'if DataFrame.column:';  Use 'if DataFrame.column.empty' instead' 


In [ ]:

import sys, os 
from PySide6.QtCore import Qt, QDir, Signal, Slot, QAbstractTableModel, QModelIndex
from PySide6.QtWidgets import QApplication, QFileSystemModel, QSplitter,QTreeView, QListView, QLabel, QWidget, QVBoxLayout, QTableView, QMainWindow
from PySide6.QtGui import QColor, QIcon
import datetime
import pandas
from MyDatabase import MyDatabase 
from MyTableModel import MyTableModel
from MyTableView import MyTableView
from Spreadsheet import Spreadsheet

app=QApplication(sys.argv)

window=QMainWindow()
database = MyDatabase()
model = MyTableModel()
table=  MyTableView() 
model._table_name = 'integrated circuits (ics)_embedded_microcontrollers'
spreadsheet = Spreadsheet()

tables = database.metadata.tables.keys()
spreadsheet.combo_box_tables.insertItems(0, tables)

#Connections to update the table when user picks a new table 
spreadsheet.combo_box_tables.currentTextChanged.connect(database.create_model)
database.create_model_finished.connect(table.setModel)

# Connections to change the view & database when the model changes. The View protects against user changing model on a restricted column
model.update_database.connect(database.update)
model.dataChanged.connect(model.on_data_changed)

table.setModel(model)
window.setCentralWidget(table)

window.show()
sys.exit(app.exec())

In [ ]:
import sys, os 
from PySide6.QtCore import Qt, QDir, Signal, Slot, QAbstractTableModel, QModelIndex, QObject
from PySide6.QtWidgets import QApplication, QFileSystemModel, QSplitter,QTreeView, QListView, QLabel, QWidget, QVBoxLayout, QTableView, QMainWindow, QPushButton
from PySide6.QtGui import QColor, QIcon

###QOBJECT###
class MyObject(QWidget):
    
    int_sig = Signal(int)
    bool_sig = Signal(bool)
    def __init__(self, parent=None):
        super().__init__(parent)
        # super(QObject).__init__() 
        
        # super(QWidget, self).__init__(parent) # 'self' must be passed in second+ call to super
        self.int_sig.emit(10)
        
    def emit_int(self, num):
        self.int_sig.emit(num)
    
        
class MyWidget(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        
    @Slot(int)
    def int_slot(self, num):
        print('NUM:', num)
        
    @Slot(bool)
    def bool_slot(self, tf):
        print('TF:', tf)
        
app = QApplication()
o = MyObject()
window = QMainWindow()

widget = MyWidget()

o.emit_int(5)

btn = QPushButton('clickme')
window.setCentralWidget(btn)

btn.clicked.connect(o.bool_sig)

o.bool_sig.connect(widget.bool_slot)
o.int_sig.connect(widget.int_slot)
# btn.clicked.connect(o.int_sig) Can't connect, bc clicked emits a BOOL while this takes an INT
# You can connect two signals together. They must emit/take correct types
# btn.clicked.connect()
# Its just very confusing, practice nailing down signals/slots 

window.show()
app.exec_()




In [ ]:
###QAbstractItemView### 
##SIGNALS##
#.ACTIVATED(index)
#activated is when user chooses an item; basically when they double click on it or press enter key. 
#The activated signal emits the modelindex 
#.ACTIVATED, .CLICKED, .DOUBLECLICKED, .ENTERED, .PRESSED, all these signals emit the model index. pressed means mouse goes down, clicked means mouse goes down and up. 
#VIEWPORTENTERED() emits when the view is entered.
##SLOTS##
#.COMMITDATA(editor) Commit the data in the editor to the model
#.CLOSEEDITOR(editor, hint) Closes the given editor, and releases it. The hint is used to specify how the view should respond to the end of the editing operation. For example, the hint may indicate that the next item in the view should be opened for editing.
#.DATACHANGED(topLeft, bottomRight, roles) Slot called when model item is changed : called when items with the given roles are changed in the model. The changed items are those from topLeft to bottomRight inclusive. If just one item is changed topLeft == bottomRight.


In [ ]:
Where does data come from? Sql Database. 
# Put the sql database data into my table model 
User has ability to change 'symbols' data. 
User double clicks on 'symbols'. Enters new symbol. 
Enter that change into the database. 
How do I do this? Signals & Slots? 


In [ ]:
# https://www.pythonguis.com/tutorials/qtableview-modelviews-numpy-pandas/
# Make a model for a QTableView 
# The view asks the model for its data in a given role, most often Qt.DisplayRole, which -> str to print. But if we so choose, we can 
import sys, os 
from PySide6.QtCore import Qt, QDir, Signal, Slot, QAbstractTableModel
from PySide6.QtWidgets import QApplication, QFileSystemModel, QSplitter,QTreeView, QListView, QLabel, QWidget, QVBoxLayout, QTableView, QMainWindow
from PySide6.QtGui import QColor, QIcon
import datetime

class TableModel(QAbstractTableModel):
    def __init__(self, data):
        super().__init__()
        self._data = data 
        self.colors = ['#053061', '#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#f7f7f7', '#fddbc7', '#f4a582', '#d6604d', '#b2182b', '#67001f']
        
    def data(self, index, role):
        value = self._data[index.row()][index.column()]
        if role == Qt.DisplayRole: 
            # return self._data[index.row()][index.column()]
            if isinstance(value, int):
                return value
            if isinstance(value, float):
                return f"{value:2f}"
            if isinstance(value, str):
                return value
            if isinstance(value, datetime.datetime):
                return str(value)
        if role == Qt.TextAlignmentRole:
            if isinstance(value, int) or isinstance(value, float):
                return Qt.AlignVCenter + Qt.AlignRight
        if role == Qt.ForegroundRole:  # make all negative numbers in this model return a negative number
            if isinstance(value, (int, float)) and value<0: 
                return QColor('red')
        if role == Qt.BackgroundRole: 
            if isinstance(value, (int, float)):
                value = int(value)
                value = max(-5 , value)
                value = min(5,value)
                value = value +5 
                return QColor(self.colors[value])
        # if role == Qt.DecorationRole: 
        #     if isinstance(value, datetime):
        #         return QIcon('calendar.png')
            # if isinstance(value, bool):
            #     if value: 
            #         return QIcon('tick.png')
            #     return QIcon('cross.png')
        
    def rowCount(self, index):
        return len(self._data)
         
    def columnCount(self, index):
        return len(self._data[0])
    
    def headerData(self, section, orientation, role):
        if not role == Qt.DisplayRole: # Don't bother if the role isn't valid; we're only bothering with displayrole for now
            return None
        if orientation == Qt.Horizontal:
            return f"Column {section}"
        else:
            return f"Row {section}"
        
class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super().__init__(parent)
    
        data = [
            [4, 'hello', 2],
            [-1, 0, 0],
            [3, datetime.datetime(2017,5,3), 0],
            [3, 3, 3.1419265],
            [7, 8, 9],
        ]

        self.table = QTableView()
        self.model = TableModel(data)
        self.table.setModel(self.model)
        self.setCentralWidget(self.table)
        

app=QApplication(sys.argv)
window=MainWindow()
window.show()
app.exec_()

In [ ]:
model = index.model() # model indexes know their model
#model indexes should not be stored, bc they change-- use it one and done
index = model.index(row, col, parent)
index = model.index(row, col, QModelIndex()) # top level items in a model have QModelIndex() as their parent.

indexA = model.index(0,0,QModelIndex())
indexB = model.index(1,0, indexA)

We can ask the model for an item's data by passing it the model index and role
value = model.data(index, role)
models can provide hints to views and delegates, about how items should be presented to the user.
# DisplayRole is used to hold a string to display


In [ ]:

# QFileSystemModel

if __name__ == "__main__":
    app = QApplication(sys.argv)
    splitter = QSplitter()
    model = QFileSystemModel()
    model.setRootPath(QDir.currentPath()) # model is set up to use data from certain file system. .setRootPath tells the model which drive to expose to the views...?
    
    # create two views, so we can examine the items held in the model, in two ways
    tree = QTreeView(splitter)
    tree.setModel(model)
    tree.setRootIndex(model.index(QDir.currentPath())) # QFileSystem.index(directory) takes a directory, and it returs a model index
    
    list = QListView(splitter)
    list.setModel(model)
    list.setRootIndex(model.index(QDir.currentPath()))
    
    # make a view display the items in a model with setModel(model)
    
    splitter.setWindowTitle("Two views on the same file system model")
    splitter.show()


    sys.exit(app.exec())

In [ ]:
# Example deals with model indexes & getting data from a model. Not a 'normal' way to use a QFileSystemModel, just for example.
import sys, os 
from PySide6.QtCore import Qt, QDir, Signal, Slot
from PySide6.QtWidgets import QApplication, QFileSystemModel, QSplitter,QTreeView, QListView, QLabel, QWidget, QVBoxLayout
from PySide6.QtGui import QPen

if __name__ == "__main__":
    app = QApplication(sys.argv)
    splitter = QSplitter()
    model = QFileSystemModel()
    model.setRootPath(QDir.currentPath()) # model is set up to use data from certain file system. .setRootPath tells the model which drive to expose to the views...?



    window = QWidget()
    layout= QVBoxLayout()
    
    @Slot(str)
    def onDirectoryLoaded(directory): 
        parentIndex = model.index(directory)
        numRows = model.rowCount(parentIndex)
        for row in range(numRows):
            index = model.index(row, 0 , parentIndex)
            text = model.data(index)
            label = QLabel(text, window)
            layout.addWidget(label)
            
    model.directoryLoaded.connect(onDirectoryLoaded)
    window.setLayout(layout)
    window.show()

    sys.exit(app.exec())

In [ ]:
# The view renders the contents of a model, accessing data via the model's interface. When the user tries to edit an item, the view uses a default delegate to provide an editor widget.
import sys, os 
from PySide6.QtCore import Qt, QDir, Signal, Slot, QAbstractListModel, QModelIndex
from PySide6.QtWidgets import QApplication, QFileSystemModel, QSplitter,QTreeView, QListView, QLabel, QWidget, QVBoxLayout
from PySide6.QtGui import QPen

class StringListModel(QAbstractListModel): 
    # QAIM does not store any data itself; merely presents an interface, that the view uses to access data.
    def __init__(self, strings, parent=None):
        super().__init__(parent)
        self.stringList = strings
        
    def rowCount(self, parent = QModelIndex()):
        return len(self.stringList)
    
    def data(self, index, role):
        if not index.isValid():
            return None
        if index.row() >= len(self.stringList):
            return None
        if role == Qt.DisplayRole:
            return self.stringList[index.row()]
        else:
            return None
            
    def headerData(self, section, orientation, role):
        if not role == Qt.DisplayRole: # Don't bother if the role isn't valid; we're only bothering with displayrole for now
            return None
        if orientation == Qt.Horizontal:
            return f"Column {section}"
        else:
            return f"Row {section}"
    
    
if __name__ == "__main__":
    
    app = QApplication(sys.argv)
    numbers = [ "One" , "Two" , "Three" , "Four" ]
    model = StringListModel(numbers)
    
    view = QListView()
    view.setModel(model)
    view.show()
    sys.exit(app.exec())



In [ ]:
from PySide6.QtWidgets import QApplication, QTableView
from PySide6.QtCore import Qt, QAbstractTableModel, QModelIndex
import sys
# QAbstractTableModel requires three overrides .rowCount().columnCount().data()
class MyModel(QAbstractTableModel): 
    def __init__(self, parent=None):
        super().__init__(parent)
    def rowCount(self, parent= QModelIndex()) : # override
        return 2
    def columnCount(self, parent = QModelIndex()): # override
        return 3 
    def data(self, index:QModelIndex, role = Qt.DisplayRole): # override
        if role == Qt.DisplayRole:
            return f"Row{ index.row() +1 }, Column{ index.column() +1 }"
        return str()
    
if __name__ == "__main__":
    app = QApplication(sys.argv)
    tableView = QTableView()
    myModel = MyModel()
    tableView.setModel(myModel) # tableView will invoke myModel.rowCount(), .columnCount, .data(), to find out number of rows and columns, and the content to print in each cell. The model needs to reimplement these functions, to respond.
    tableView.show()
    sys.exit(app.exec())
    

    
    

In [ ]:
from PySide6.QtWidgets import QApplication, QTableView
from PySide6.QtGui import QFont, QBrush, QPen
from PySide6.QtCore import Qt, QAbstractTableModel, QModelIndex
import sys
# QAbstractTableModel requires three overrides .rowCount().columnCount().data()
class MyModel(QAbstractTableModel): 
    def __init__(self, parent=None):
        super().__init__(parent)
    def rowCount(self, parent= QModelIndex()) : # override
        return 2
    def columnCount(self, parent = QModelIndex()): # override
        return 3 
    def data(index:QModelIndex, role:int):# override. index: the index being requested. role: the item data rol being requested.
        row = index.row()
        col = index.col()
        if role == Qt.ItemDataRole.DisplayRole: 
            if row == 0 and col==1: 
                return ("<--left")
            if row==1 and col==1: 
                return("right-->")
            return f"Row{ index.row() }, Column{ index.column() }"
        elif role== Qt.ItemDataRole.FontRole:
            if row==0 and col==1: 
                boldFont=QFont()  
                boldFont.setBold(True)
                return boldFont     
            
# In this example, there are six cells, and seven ItemDataRoles, to populate the view, model.data() is called seven times on each cell, to get the display,font, background, textAlignment, &checkState data. That's why you should make sure data is available when .data() is called, and why you should cache expensive lookup operations. 

if __name__ == "__main__":
    app = QApplication(sys.argv)
    tableView = QTableView()
    myModel = MyModel()
    tableView.setModel(myModel) # tableView will invoke myModel.rowCount(), .columnCount, .data(), to find out number of rows and columns, and the content to print in each cell. The model needs to reimplement these functions, to respond.
    tableView.show()
    sys.exit(app.exec())

In [ ]:
from PySide6.QtWidgets import QApplication, QTableView
from PySide6.QtGui import QFont, QBrush, QPen
from PySide6.QtCore import Qt, QAbstractTableModel, QModelIndex, QTime, QTimer, Slot, Signal
import sys
# QAbstractTableModel requires three overrides .rowCount().columnCount().data()
class MyModel(QAbstractTableModel): 
    def __init__(self, parent=None):
        super().__init__(parent)
        
###This will make the timer tick every second###
        timer = QTimer(self) 
        timer.setInterval(1000)
        timer.timeout.connect(self.timerHit)
        timer.start()
    @Slot()
    def timerHit(self):
        topLeft = self.createIndex(0,0) # identify the topLeft cell
        self.dataChanged.emit(topLeft, topLeft, Qt.DisplayRole ) # emit a signal, to make the view reread the identified data. 
###.DATACHANGED(topLeft,bottomRight,role(s))###
#This slot(signal?) emites when items from topLeft to bottomRight, with role(s), are changed in the model. To edit a cell, topLeft==bottomRight
#roles default is None, in which case all roles will update, or provide specific roles to just update those roles.
#################################################
    def rowCount(self, parent= QModelIndex()) : # override
        return 2
    
    def columnCount(self, parent = QModelIndex()): # override
        return 3 
    
    def data(self, index:QModelIndex, role:int):# override. index: the index being requested. role: the item data rol being requested.
        row = index.row()
        col = index.column()
        if role == Qt.ItemDataRole.DisplayRole and row ==0 and col ==0: 
            return QTime.currentTime().toString()
        return "" 
    # See how the clock only ticks when user interacts with cell, bc thats when the view ueries the model 

if __name__ == "__main__":
    app = QApplication(sys.argv)
    tableView = QTableView()
    myModel = MyModel()
    tableView.setModel(myModel) # tableView will invoke myModel.rowCount(), .columnCount, .data(), to find out number of rows and columns, and the content to print in each cell. The model needs to reimplement these functions, to respond.
    tableView.show()
    sys.exit(app.exec())

In [ ]:
from PySide6.QtWidgets import QApplication, QTableView
from PySide6.QtGui import QFont, QBrush, QPen
from PySide6.QtCore import Qt, QAbstractTableModel, QModelIndex, QTime, QTimer, Slot, Signal
import sys
# QAbstractTableModel requires three overrides .rowCount().columnCount().data()
class MyModel(QAbstractTableModel): 
    def __init__(self, parent=None):
        super().__init__(parent)
    #Header content is set via model, but, hidden with tableView.verticalHeader.hide()
    
    def headerData(self, section:int , orientation:Qt.Orientation, role:int): #override
        if role ==Qt.DisplayRole and orientation == Qt.Horizontal:
            print('header_data')
            if section == 0: 
                print('section==0')
                return "first"
            if section == 1: 
                return "second"
            if section == 2 :
                return 'third'
        return None
    
  
    def rowCount(self, parent= QModelIndex()) : # override
        return 2

    def columnCount(self, parent = QModelIndex()): # override
        return 3 

    def data(self, index:QModelIndex, role:int):# override. index: the index being requested. role: the item data rol being requested.
        row = index.row()
        col = index.column()
        if role == Qt.ItemDataRole.DisplayRole and row ==0 and col ==0: 
            return QTime.currentTime().toString()
        return None
    # See how the clock only ticks when user interacts with cell, bc thats when the view ueries the model 

if __name__ == "__main__":
    app = QApplication(sys.argv)
    tableView = QTableView()
    myModel = MyModel()
    tableView.setModel(myModel) # tableView will invoke myModel.rowCount(), .columnCount, .data(), to find out number of rows and columns, and the content to print in each cell. The model needs to reimplement these functions, to respond.
    tableView.show()
    sys.exit(app.exec())
    
#Hmm headers not showing, athis one missing something 

In [ ]:
from PySide6.QtWidgets import QApplication, QTableView
from PySide6.QtGui import QFont, QBrush, QPen
from PySide6.QtCore import Qt, QAbstractTableModel, QModelIndex, QTime, QTimer, Slot, Signal
import sys
# QAbstractTableModel requires three overrides .rowCount().columnCount().data()
class MyModel(QAbstractTableModel): 
    # editCompleted = Signal(str) # Transfers the modified text, to window title.
    
    def __init__(self, parent=None):
        super().__init__(parent)
        self.m_gridData = [] # 2D array holds text entered into QTableView; core of model.
    #Header content is set via model, but, hidden with tableView.verticalHeader.hide()
    
    def setData(self,index, value, role): # override
        if role == Qt.EditRole:
            if not self.checkIndex(index): 
                return False 
            self.m_gridData[index.row()][index.column()] = value
            result = "" #Aesthetics: build & emit string
            # for row_idx, row in enumerate(self.m_gridData):
            #     for col_idx, _ in row: 
            #         result += str(self.m_gridData[row_idx][col_idx])
            # self.editCompleted.emit(result)
            return True
        return False
            

    def flags(self, index): # override. Various properties ofa cell can be adjusted with .flags().
        return Qt.ItemIsEditable|QAbstractTableModel.flags(index)

    def rowCount(self, parent= QModelIndex()) : # override
        return 2
    
    def columnCount(self, parent = QModelIndex()): # override
        return 3 
    
    def data(self, index:QModelIndex, role:int):# override. index: the index being requested. role: the item data rol being requested.
        row = index.row()
        col = index.column()
        if role == Qt.ItemDataRole.DisplayRole and row ==0 and col ==0: 
            return str(QTime.currentTime())
        return None
    # See how the clock only ticks when user interacts with cell, bc thats when the view ueries the model 

if __name__ == "__main__":
    app = QApplication(sys.argv)
    tableView = QTableView()
    myModel = MyModel()
    tableView.setModel(myModel) # tableView will invoke myModel.rowCount(), .columnCount, .data(), to find out number of rows and columns, and the content to print in each cell. The model needs to reimplement these functions, to respond.
    tableView.show()
    sys.exit(app.exec())
    
#Kernel Crashes...

In [ ]:
import sys 
import os 
from PySide6.QtWidgets import *
app = QApplication(sys.argv)
window = QWidget()

# test = os.path.abspath('test/') # -> c:\Users\robby\OneDrive\part_database\test. Relative paths aren't best practice for QFileDialogs.
# file_path, _ = QFileDialog.getOpenFileName(
#     parent=window,
#     caption="Select a file",
#     dir=test, 
#     filter="All Files (*)"
# )

# if file_path:
#     print(f"Selected file: {file_path}")
# else:
#     print("No file selected.")
    
start_dir = os.path.abspath('parts') # c:\Users\robby\OneDrive\part_database\parts
print("START_DIR",start_dir) 
selected_file = QFileDialog.getOpenFileName(parent=None, caption="SelectFileToOpen", dir=start_dir, filter="All Files (*)")
print('selected_file',selected_file)
# library = selected_file 
# library_id = #GoIntoFileAndExtractAllFileNames.->Selected fileName
# def get_library_ids(library): 


# file_path = QFileDialog.getOpenFileName()
### QFILEDIALOG ###
# 'modal' means 'popup'. modal means 'blocking popup': Those popups tha
t you have to click 'ok'/'cancel' on to make it go away, and prevent user from clicking on other windows until you deal with the popup 
### .GETOPENFILENAME -- launchs a 'open file' modal. -> filename
    # QFileDialog.getOpenFileName(parent=None, caption="", dir="",filter=""): # convenience function. -> existing file selected by user. If user presses Cancel, -> null string ''. 
    # QFileDialog.getOpenFileName(parent=None, caption="Open File", dir="/home",filter="All Files (*)") # filter = "Images (*.png *.xpm *.jpg)"
    # The function creates a modal(popout) file dialog on 'parent', in 'dir'. If 'dir' includes a file name, the file is selected. 
    # Only files matching 'filter' are shown. Separate multiple filters with double semicolon ;;. Ex: "Images (*.png *.xpm *.jpg);;Text files (*.txt);;XML files (*.xml)"
    # See QFileDialog.Option enum, for info on 'options', which holds options about how to run the dialog
    # Set dialog's caption with 'caption'
    # Slight behavior differences between file dialogs on different OS's. On Windows, and macOS, this static function uses the native file dialog and not a QFileDialog. Note that the macOS native file dialog does not show a title bar.On Windows the dialog spins a blocking modal event loop that does not dispatch any QTimers, and if parent is not nullptr then it positions the dialog just below the parent's title bar. On Unix/X11, the normal behavior of the file dialog is to resolve and follow symlinks. For example, if /usr/tmp is a symlink to /var/tmp, the file dialog changes to /var/tmp after entering /usr/tmp. If options includes DontResolveSymlinks, the file dialog treats symlinks as regular directories.
### .GETSAVEFILENAME -- launches a 'save file' modal. -> 
### .GETEXISTINGDIRECTORY -- launches a 'select folder' modal 



In [ ]:
if [[]]:
    print('[[]] Evaluates True')
    
if [] == False:
    print('[]==False evaluates True')
    
if [1]: 
    print('true')

In [ ]:
import sys
import os
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout,
    QPushButton, QFileDialog, QLabel
)

class MyApp(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Symbol Selector")
        self.setMinimumWidth(300)

        layout = QVBoxLayout()

        # Label to show the result
        self.label = QLabel("No action yet.")
        layout.addWidget(self.label)

        # Button 1: ClickMe
        self.click_button = QPushButton("ClickMe")
        self.click_button.clicked.connect(self.on_clickme)
        layout.addWidget(self.click_button)

        # Button 2: Open from symbols folder
        self.file_button = QPushButton("Select Symbol File")
        self.file_button.clicked.connect(self.select_file)
        layout.addWidget(self.file_button)

        self.setLayout(layout)

    def on_clickme(self):
        self.label.setText("ClickMe button was clicked!")

    def select_file(self):
        folder = os.path.join( "symbols")
        if not os.path.exists(folder):
            os.makedirs(folder)  # Create the folder if it doesn't exist

        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Select a Symbol File",
            folder,
            "All Files (*.*)"
        )

        if file_path:
            self.label.setText(f"Selected: {os.path.basename(file_path)}")
        else:
            self.label.setText("No file selected.")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = MyApp()
    window.show()
    sys.exit(app.exec())



In [ ]:
import webbrowser
webbrowser.open('google.com')

In [ ]:
btn = QPushButton()
btn = QPushButton('hey')

if btn:
    print('true')

In [ ]:

# Implementing drags: 
# mousePressEvent, mouseDoubleClickEvent, QAbstractItemView.DragOnly.ItemIsDragEnabled.supportedDragActions.mimeData.moveAction.CopyAction

In [ ]:
columns = [5,4]
df = [1,2,3,4,5]
other_columns = [c for c in df if not c in columns ]
print()
print(other_columns)
columns.extend(other_columns)
print(columns)

In [ ]:
###SQLALCHEMY###
# https://soogoonsoogoonpythonists.github.io/sqlalchemy-for-pythonist/en/tutorial/2.%20Setting%20Up%20a%20Connection.html#connecting-to-a-database  A good sqlalchemy tutorial ( the official sqlalchemy docs are soo wordy and dilute. I recommend following soogoonsoogoon's tutorial)
from sqlalchemy import *
invoice_item = Table('invoice_item', metadata, 
                    Column('item_id',Integer, primary_key=True),
                    Column('item_name', String, nullable=False),
                    Column('invoice_id', Integer, nullable=False),
                    Column('ref_num', Integer, nullable = False),
                    ForeignKeyConstraint(
                        ['invoice_id', 'ref_num'],
                        ['invoice.invoice_id','invoice.ref_num'], 
                        onupdate=  "CASCADE",
                        ondelete = "SET NULL"),
                    UniqueConstraint('item_name', 'invoice_id', name = 'unique1'),
                    CheckConstraint("item_id > 0", name = 'check1')
)
"""
CREATE TABLE invoice_item (
item_id INTEGER NOT NULL, 
item_name VARCHAR NOT NULL, 
invoice_id INTEGER NOT NULL, 
ref_num INTEGER NOT NULL, 
PRIMARY KEY (item_id), 
FOREIGN KEY(invoice_id, ref_num) REFERENCES invoice (invoice_id, ref_num) ON DELETE SET NULL ON UPDATE CASCADE, 
CONSTRAINT unique1 UNIQUE (item_name, invoice_id), 
CONSTRAINT check1 CHECK (item_id > 0)
)
"""
# Note you cant move the type defines down below: has to be id INTEGER cant be id alone

In [ ]:
###PANDAS.DATAFRAME###
import pandas as pd 
df = pd.DataFrame([[1, 2], [4, 5], [7, 8]],
                  index=['cobra', 'viper', 'sidewinder'],
                  columns=['max_speed', 'shield'])
print('DF:')
print(df)
a = df.loc['viper'] 
b = df.loc[ ['viper', 'sidewinder'] ]
c = df.loc[ ['viper', 'sidewinder'], ['shield']]
d = df.loc[ ['viper'] , ['shield' , 'max_speed'] ] # A weird one: this locates everything: all the rows, all the columns

print()
print('a:')
print(a)
print()
print('b:')
print(b)
print()
print('c:')
print(c)
print()
print('d:')
print(d)

In [ ]:
def func(func):
    print(func)
    
func('hey')


In [ ]:
list_widget = QListWidget()
list_widget.setDragDropMode(QAbstractItemView.DragDropMode.InternalMove) # Reorder items (within the list)

list_widget.setDragEnabled(True) # QAbstractItemView.setDragEnabled(bool): set whether view supports dragging of its own items


In [ ]:
    # clicked = Signal(QGraphicsObject) #NOTE: We don't HAVE to use signals.  QWidget has a .clicked signal, but QGraphicsItem does not, so let's add one-- we still need to clicked.emit(self) in the MousePressEvent, and self.clicked.connect(slot), in our constructor ( Is this OK to do? It works, but is it good practice?)

    # def mousePressEvent(self, event):
    #     QGraphicsItem.mousePressEvent(self, event)  # keep the base implementation, which calls mousePressEvent()
    #     # super().mousePressEvent(event)            # This works too. Note super() calls the next class by MethodResolutionOrder MRO, which is QGraphicsItem in this case. And, 'self' is bound to 'super()', so you won't need to do super(self); super() binds self for you; but if you reference QGraphicsItem.mousePressEvent() by name, you'll have to include self(Why ?)
    #     # QGraphicsItem.mousePressEvent(event)      # This dnw : 'mousePressEvent' for 'PySide6.QtWidgets.QGraphicsItem' objects doesn't apply to a 'PySide6.QtWidgets.QGraphicsSceneMouseEvent' object
    #             this_items_layout.removeWidget(widget) # Takes widget out of layout; widget becomes a unattached, floating window; Does not hide nor destroy widget
    #             widget.setParent(None) # hide widget, & let it be destroyed
# QWidget.hide() effectively removes the widget from it's layout until QWidget.show()    
# Nonselectable or nonmovable items Do not receive single or double click events 


In [ ]:
# File Browser App which returns 
from PySide6.QtWidgets import (
    QApplication, QWidget, QVBoxLayout,
    QTreeView, QFileSystemModel
)
from PySide6.QtCore import QDir
from utils import * 
import os 
from MyGraphicsItem import MyItem

class FileBrowser(QWidget):
    def __init__(self):
        super().__init__()
        path = '/'.join(['C:', 'Users', 'robby','OneDrive', 'part_database', 'symbols']) 
        print('PATH:', path)
        print("QDIR.ROOTPATH():", QDir.rootPath()) # C:/
        print("QDIR.HOMEPATH()", QDir.homePath()) # C:/Users/robby 
        
        self.setWindowTitle("Custom File Browser")
        layout = QVBoxLayout(self)

        self.model = QFileSystemModel()
        self.model.setRootPath(QDir.rootPath())
        # self.model.setRootPath(path)

        self.view = QTreeView()
        self.view.setModel(self.model)
        # self.view.setRootIndex(self.model.index(QDir.homePath()))
        self.view.setRootIndex(self.model.index(path)) # Providing .index(wrongPath) -> 0
        self.view.doubleClicked.connect(self.on_double_click)

        layout.addWidget(self.view)

    def on_double_click(self, index):
        selected_path = self.model.filePath(index)
        print(f"SELECTED_PATH:", selected_path)
        root, extension = os.path.splitext(selected_path)
        print()
        print('EXTENSION:', extension)
        if extension == '.xml':
            item = MyItem(selected_path)
            scene=QGraphicsScene(item.boundingRect())
            scene.addItem(item)
            view= QGraphicsView(scene)
            print('SELF.LAYOUT():', self.layout())
            self.layout().addWidget(view)
            
            
        # It does not close, behaves like a persistent file browser

if __name__ == "__main__":
    app = QApplication([])
    fb = FileBrowser()
    fb.resize(600, 400)
    fb.show()
    app.exec()

In [ ]:
from lxml import etree

def prettyprint(element, **kwargs):
    xml = etree.tostring(element, pretty_print= True, **kwargs) 
    print(xml.decode(), end='')
    
root = etree.Element("root", k1='v1')
root.text= 'hey'

etree.SubElement(root, "child1", key = 'value').text = "Child 1"
p =etree.SubElement(root, "child2")
print()
print(p)
prettyprint(root)

for element in root.iter():
    print(f"{element.tag} - {element.text}")
# iteration yields ALL nodes(ProcessingInstructions, Comments, and Entity instances included). Pass 'Element' to specify only return element instances.

sym = etree.Element(root.tag,root.attrib)
sym.text = root.text
for elem in root.iter():
    text = elem.text if elem.text else ""
    x = etree.SubElement(sym, elem.tag ,elem.attrib)
    print(x.text)

print()
print("SYM:", sym.tag, sym.attrib, sym.text ,sym)
prettyprint(sym)


In [ ]:
from utils import * 

class MyFileDialog(QDialog):
    finished = Signal(int) # This signal is emitted when the dialog's result code has been set
    
    def __init__(self, parent=None):
        super().__init__(parent)
        layout = QVBoxLayout()
        self.setLayout(layout)
        self.setWindowModality(Qt.WindowModality.NonModal) # Choose QT.NonModal(nonBlocking) or Qt.WindowModal(kindaBlocking) or Qt.ApplicationModal(all blocking) Default nonModal
        self.add_contents()
        self.add_button_box()
        self.finished.connect(self.on_finished)
        
    def add_contents(self):
        container = QWidget()
        layout=  QGridLayout()
        container.setLayout(layout)
        
        title= 'Create Symbol - Download Symbol'
        container.setWindowTitle(title)

        label1 = QLabel("Download from Url:")
        edit1 = QTextEdit()
        label2 = QLabel('OR')
        label3 = QLabel("DoubleClick on part in table with a snapmagic or ultralibrarian link")
        
        layout.addWidget(label1, 0,0)
        layout.addWidget(edit1 , 0,1)
        layout.addWidget(label2, 1,1)
        layout.addWidget(label3, 2,0)
        
        self.layout().addWidget(container)

    def add_button_box(self):        
        self.button_box = QDialogButtonBox(QDialogButtonBox.Ok|QDialogButtonBox.Cancel)
        self.button_box.accepted.connect(self.accept)
        self.button_box.rejected.connect(self.reject)
        self.layout().addWidget(self.button_box)
        
    # def mousePressEvent(self, event):
    def accept(self):
        print()
        print('QDIALOG.ACCEPT: Hides the modal dialog and sets the result code to Accepted.')
        super().accept()
        
    def reject(self):
        print()
        print('QDIALOG.REJECT: Hides the modal dialog and sets the result code to Rejected.')
        super().reject()
        
    @Slot(int) # takes int r: result_code
    def on_finished(self, result_code ):
        print()
        print(f"QDIALOG.ON_FINISHED got result code: {result_code}")
        
        
        
app = QApplication(sys.argv)
dialog = MyFileDialog()
dialog.open() # Shows the dialog as a window modal dialog, returning immediately. Connect to the 'finished' signal to know when the dialog has been QDialog.Accepted or QDialog.rejected


sys.exit(app.exec())

In [ ]:
### QDIR ###
# Analogous to python's 'os' module 
# QDir("Documents/Letters/Applications").dirName() // "Applications"
# QDir().dirName()                                 // "."


# QDir directory("Documents/Letters");
# QString path = directory.filePath("contents.txt");
# QString absolutePath = directory.absoluteFilePath("contents.txt");


# Remove a file: remove(file)
# Remove a dir: rmdir(file)

# Filters: 
# name filter (match pattern)(.setNameFilters)
# attribute filter( match properties)( .setFilter(bitwise OR filters)) (.setSorting(SortFlags) 
                                       

In [ ]:
import json 
x = json.dumps({'a':1})
print(x)


In [ ]:

# Python Overloading does not actually exist( @typing.overload decorator?) In python, much the functionality of overloads can be accomplished with: 
# @classmethod decorator
# multipledispatch.dispatch module
# functools.singledispatch module
    
# Use @classmethod to make instances from different variables
class MyClass():
    def __init__(self, i):
        self.i = i
        
    @classmethod 
    def from_int(cls, i):
        return cls(i)

    @classmethod 
    def from_str(cls, s):
        return cls(int(s))
    
class MyPart:
    def __init__(self, mpn = None):
        self.mpn = mpn
        
    @classmethod
    def from_record(cls, record):
        return cls(record.get('mpn', None))
    
part=MyPart('STM32C0')
record = {'mpn': 'STM32F6'}
part=MyPart.from_record(record)
part = MyPart(record)
    

#Overload, method overloads, function overloading 

# from QtWidgets.pyi
import typing
@typing.overload
def addItem(self, text: str, userData: typing.Any = ...) -> None: ...
@typing.overload
def addItem(self, icon: PySide6.QtGui.QIcon | PySide6.QtGui.QPixmap, text: str, userData: typing.Any = ...) -> None: ...

# Use python's typing.overload decorator to define function calls with different parameters.



### Multimethods, multiple dispatch, same function, different arguments.

from multipledispatch import dispatch # pip install multipledispatch 

@dispatch(int, int)
def product(a,b):
    return a*b

@dispatch(str,str)
def product(a,b):
    return float(a)*float(b)
print(product(3,4))
print(product('3.0', '4.9'))

# Funfact: multiple dispatch is important in the programming language Julia


### Q: explain some differences between an overloaded function(functions of same name, take different number of args/ argument types ), default arguments, and keyword arguments.

In [ ]:
s = """(symbol "CSQG703BP_0_0"
      (polyline
        (pts (xy -0.635 1.27) (xy 0.0 1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.0 1.27) (xy 0.635 1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.635 1.27) (xy 0.635 -1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.635 -1.27) (xy 0.0 -1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.0 -1.27) (xy -0.635 -1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy -0.635 -1.27) (xy -0.635 1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.635 1.27) (xy 1.905 3.175)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 1.905 3.175) (xy 1.905 -3.175)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 1.905 -3.175) (xy 0.635 -1.27)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.0 1.27) (xy 0.0 2.54)) (stroke (width 0.254))
      )
      (polyline
        (pts (xy 0.0 -1.27) (xy 0.0 -2.54)) (stroke (width 0.254))
      )
      (text "+" (at -1.9109 1.27393 0)
        (effects (font (size 1.78351 1.78351)) (justify bottom left))
      )
      (text "-" (at -1.91394 -3.8126 0)
        (effects (font (size 1.78634 1.78634)) (justify bottom left))
      )
      (pin passive line (at 0.0 5.08 270.0) (length 2.54)
        (name "~"
          (effects (font (size 1.016 1.016)))
        )
        (number "P"
          (effects (font (size 1.016 1.016)))
        )
      )
      (pin passive line (at 0.0 -5.08 90.0) (length 2.54)
        (name "~"
          (effects (font (size 1.016 1.016)))
        )
        (number "N"
          (effects (font (size 1.016 1.016)))
        )
      )
    )"""
import sexpdata
import json
s = s.replace('(','[').replace(')',']')
x = json.loads(s, object_hook = list)
print(x)
# print(s)
print(sexpdata.loads(s))



print(len(s))

print(list(s))

In [ ]:
# A 'simple' app featuring panning, zooming, for mouse AND trackpad( HEYY I THOUGHT YOU HAD TO IMPLEMENT PINCHGESTURE FOR TRACKPAD DRAGGING, guess not) 
from utils import *

class SchematicScene(QGraphicsScene):
    def __init__(self, *args, **kwargs):
        super().__init__(*args , **kwargs)
        
class SchematicView(QGraphicsView):
    ZOOM_FACTOR = 1.05
    # def __init__(self, *args, **kwargs):
    #     super().__init__(*args, **kwargs) # Tricky : constructor should pass args and kwargs to QGraphicsView, so include *args and *kwargs to super and constructor. But the super() does not get self-- super() merely returns the parent class so it takes no self 
    def __init__(self, *args, **kwargs ):
        super().__init__(*args, **kwargs ) # NO SELF IN SUPER().INIT() but do pass args, kwargs along
        
    def wheelEvent(self, event):
        print("WHEEL EVENT OCCURRED")
        # delta = event.delta() AttributeError: 'PySide6.QtGui.QWheelEvent' object has no attribute 'delta'
        delta = event.angleDelta().y()
        print(delta)
        if delta > 0: 
            zoom_factor = self.ZOOM_FACTOR
            self.scale(zoom_factor, zoom_factor)
        elif delta < 0: 
            zoom_factor = 1/self.ZOOM_FACTOR
            self.scale(zoom_factor, zoom_factor)
        else: 
            print(f"expected delta to be nonzero integer but got {type(delta)}, {delta}")
        
        
        
if __name__ == "__main__":
    app = QApplication(sys.argv)
    
    item = QGraphicsRectItem(QRect(-50,-50,100,100))
    scene = QGraphicsScene()
    # scene.setSceneRect(-2000,-2000,4000,4000)
    scene.addItem(item)
    scene.addItem(QGraphicsRectItem(-2000,-2000,4000,4000))
    
    view = SchematicView(scene) #dnw
    view.setTransformationAnchor(QGraphicsView.ViewportAnchor.AnchorUnderMouse)
    view.setDragMode(QGraphicsView.DragMode.ScrollHandDrag)
    
    # view = QGraphicsView(scene) # Works
    view.show()
    sys.exit(app.exec())

In [ ]:
# Dict keys are often strings, but can be any immutable( int and tuple )
# Dicts with tuples as keys 
# tuple order matters 




In [ ]:
# Python IS : returns True if the memory addresses of two things are the same 
print(None is None)
print('y' is 'y')
print('asdf' is 'asdf')
print([5] is [5])
L = [] 
P= [] 
Q = [1] 
L.append(Q)
P.append(Q)
print(L[0] is P[0])

In [ ]:

import sys

from PyQt6.QtCore import Qt, QRectF
from PyQt6.QtGui import QBrush, QPainter, QPen
from PyQt6.QtWidgets import (
    QApplication,
    QGraphicsEllipseItem,
    QGraphicsItem,
    QGraphicsRectItem,
    QGraphicsScene,
    QGraphicsView,
    QHBoxLayout,
    QPushButton,
    QSlider,
    QDial,
    QVBoxLayout,
    QWidget,
)

class Window(QWidget):
    def __init__(self):
        super().__init__()
        
        up = QPushButton('Up')
        down = QPushButton('Down')
        moveLeft = QPushButton('Left')
        dial = QDial()
        
        vBoxLayout = QVBoxLayout()
        vBoxLayout.addWidget(up)
        vBoxLayout.addWidget(down)
        vBoxLayout.addWidget(moveLeft)
        vBoxLayout.addWidget(dial)
        
        down.clicked.connect(self.down)
        up.clicked.connect(self.up)
        moveLeft.clicked.connect(self.moveLeft)
        dial.valueChanged.connect(self.dial)
        
        # QGraphicsRect.moveTo(top, moveLeft) # Convenience classes have ez 'move' methods. Plain QGraphicsItems have setX, setY, setPos(x,y) do not, having instead 'setTransform()
        # setTransformOriginPoint() Sets the origin point for the transformation in item coordinates.
        vBoxLayout=QVBoxLayout()
        vBoxLayout.addWidget(up)
        vBoxLayout.addWidget(down)
        vBoxLayout.addWidget(moveLeft)
        vBoxLayout.addWidget(dial)
        
        self.scene = QGraphicsScene(0,0,600,400) # self.scene must be a instance attribute because we want access to the scene wherever we are in clas 'Window'. (if scene was used instead of self.scene, scene would be local to init, thus unaccessible outside init)
        view = QGraphicsView(self.scene)

        hBoxLayout = QHBoxLayout()
        hBoxLayout.addLayout(vBoxLayout)
        hBoxLayout.addWidget(view)
        
        self.setLayout(hBoxLayout)
        
    # Slider(linear knob) vs Dials(rotary knob). Default no difference in their behavior-- just visuals. .sliderPressed()Released()Moved(). .setValue(). QDial.wrapping() false default

    def up(self):
        items = self.scene.selectedItems()
        for item in items: 
            item.setZValue(item.ZValue()+1)
    def down(self):
        items = self.scene.selectedItems()
        for item in items:
            item.setZValue(item.zValue()-1)
    def moveLeft(self):
        for item in self.scene.selectedItems():
            print(item.x())
            print()
            item.setX(item.x()+2)
            
    def dial(self, value):
        for item in self.scene.selectedItems():
            item.setRotation(value) # Note rotation is about origin. Thats why its smart to put your shape centroid @origin, not the top moveLeft, as in QRectF(5,5, -10, -10) not (0,0 , )
            
app = QApplication(sys.argv)

window=Window()

ellipse = QGraphicsEllipseItem(20,200, 200,50)
ellipse.setBrush(QBrush(Qt.GlobalColor.black))
ellipse.setPen(QPen(Qt.GlobalColor.red, 10))
ellipse.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsMovable | QGraphicsItem.GraphicsItemFlag.ItemIsSelectable) # 

rect=  QGraphicsRectItem(20,20, 200,50)
rect.setBrush(QBrush(Qt.GlobalColor.black))
rect.setPen(QPen(Qt.GlobalColor.red, 10))
rect.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsMovable | QGraphicsItem.GraphicsItemFlag.ItemIsSelectable) # 

window.scene.addItem(ellipse)
window.scene.addItem(rect)

window.show()
app.exec()


In [ ]:
# REGEX MODULE regex essentials 
strWithNewlineChars = """ hey jude " \
"don't take it bad " \
"just takes a " \
"better " \
""""
import re 
line = re.sub(r'\s+', '', line).strip()


In [ ]:
#CORE PRACTICE ESSENTIALS 
###########################################################################################################################################
# Why might 
# lst, L, l, list_ 
# make acceptable variable names for lists, while "list" is an unacceptable variable name? 
###
L=[]
def func(x,y,z): 
    L.extend([x,y,z])
    # L.extend(x,y,z) Does this line generate an error? why? 
    
def func(x,y,z):
    L = [] 
    L.extend([x,y,z])
    return L 
###
    
# Is "L" a local variable in the first function? 
# In the second function, is "L" a local variable? 

# In the first function, "return L" is nonexistent. Is L still available outside the function? 
###########################################################################################################################################

def func(): # This function takes no arguemnts. Is that allowed? 
    print('Hello There')
    
print(func()) # What does this print? 
print(func) # What does this print? Does this print the same output as the above line? Why does this line print a bunch of numbers, and what do the numbers represent? 
###########################################################################################################################################
# Data structtures 
L = [1,2,3] 
D = {'1': 2}
# D = {'1', 2} Why is the above line a dictionary, yet this is a set? 
L = [[1,2,3], ['a','b','c']]
D = {1:'a' , 2:'b' , 3:'c'}
# Compare and contrast the above L and D. Identify one or more similarity between the data they carry . 
# Turn L into a Dictionary equal to D. Turn D into a List equal to L. 
# Turn the below data into 1 or more lists. Then turn the below data into one or more dictionaries. 
# name: jill age: 40 
# name: jack age: 38 
# # Turn the below data into 1 or more lists. Then turn the below data into one or more dictionaries. Comment on the implications of 'hill' posessing an entry which is not shared by 'jack' nor 'jill' 
# name: jill age: 40 
# name: jack age: 38 
# name: hill mass: 50000 kg 


# Mutables behavior
d = {'a':[0]}
d.get('a').append(6)
print(d)




In [ ]:
# CRASH COURSE DICTIONARIES

# dictionaries are bracketed with curly braces {}. 
# dictionaries are key/value pairs; pairs of information, a key, and a value. 
# keys and values are colon-separated; key:value.
# A dictionary can have many key:value pairs. key:value pairs are separated by , commas , while the key and the value are separated by : colon.

# Inevitably, you will type INCORRECT comma separted key:value pairs, and get an err.
# Dictionaries are also called a hash table( equal to or similar to?). Dictionaries are also pretty compatible with the JSON file format: json example: "{'A':0}"

# create a empty dictionary with  reserved keyword dict() or {}. Mostly , people use {}, so we will too.
d = {}
d = dict()

# Initialize a dict with {key:value }.  values can be any type. keys can be integers, floats, or strings. keys can also be tuples. 
# keys can be any 'hashable' type. Lists are not hashable; lists can't be dict keys. 
s = "A"
f = 0.0
t = (1,1) 
i = 6

d = { s : t } # string key, string value, all good. 
d = {f : i} # float key, int value, all good
d = {t : i} # tuple key, int value, all good. 

l = []
# d = { l : 0 } # list key, BAD: dict keys must be a 'hashable' type, which excludes lists : TypeError: unhashable type: 'list'

# Iterating  over a dictinary, ONLY iterates over the keys, ignores values 
for i in d: 
    print(i) 
    
# Iterate over just the keys in the dictionary, with .keys(). ( This accomplishes the same thing as iterating over the dictionary ) 
for k in d.keys():
    print(k)
    
# Iterate over just the values in the dictionary, ignoring keys, with .values(). 
for v in d.values(): 
    print(v)
    
# To iterate over everything in the dictionary, keys and values, use the .items() method, which returns a weird 'dict_items' type: Which basically just a list of 2-tuples (The tuple (3,4) is a 2-tuple because it has 2 data inside it). 
# print(d.items()) # Not that useful-- Helpfully, a dict_items object is a sequence. It can be iterated over as-is, just as a list could be iterated over. You could cast dict_items to a list, but that's not strictly necessary
# print(list(d.items())) # You could cast they dict_items type to a list, but you don't have to, dict_items is already an iterable; a sequence, just like a list.
# for i in d.items(): # 
#     print(i) 
# Because the .items() method returns a list of 2-tuples, its common to 'unpack' that tuple in the for loop: Rather than "for i in d.items()" we use "for key, value in d.items()". key becomes the first item in the tuple, value becomes the second item in the tuple.
    
for key, value in d.items() : 
    print(key)
    print(value)
    
    
# .items() and .values() and .keys() 
    

# CHAINING DICTIONARIES: 
# Dictionaries can have dictionaries as values.
d = { 0 : {'position': (1,1)  }, 1 : {'position': (2,3) } } # Values of d are more dictionaries

for i in d: 
    print(i)
    
print(d)
print()

for k,v in d.items():
    print(k) 
    print(v)
    v['position'] = 'asdf' # DICTIoNARY CHAINGING
    
print(d)
    


    
    


In [ ]:
# TIPSNTRICKS
False == 0 # True
True == 1 # True
[0,1,2][True] # -> 1
[0,1,2][False] # -> 0 

### SLICING ### 

In [ ]:
# lists list iterables advanced algorithm wrap around wrap-around shoelace 
# p2 = polygon[(i + 1) % len(polygon)]

# Find centroid shoelace algorithm

# from PySide6.QtWidgets import QGraphicsItem
# from PySide6.QtGui import QPainterPath
# from PySide6.QtCore import QPointF
# def get_centroid(item: QGraphicsItem) -> QPointF:
#     path: QPainterPath = item.shape()
#     polygon = path.toFillPolygon()  # returns QPolygonF

#     if polygon.isEmpty():
#         return item.pos()  # fallback to item's position if shape is empty

#     area = 0.0
#     cx = 0.0
#     cy = 0.0

#     # Shoelace formula for centroid
#     for i in range(len(polygon)):
#         p1 = polygon[i]
#         p2 = polygon[(i + 1) % len(polygon)]
#         cross = p1.x() * p2.y() - p2.x() * p1.y()  
#         area += cross
#         cx += (p1.x() + p2.x()) * cross
#         cy += (p1.y() + p2.y()) * cross

#     area *= 0.5
#     if area == 0:
#         return item.pos()  # fallback if degenerate shape

#     cx /= (6.0 * area)
#     cy /= (6.0 * area)

#     return QPointF(cx, cy)





# Remember python modulus aka remainder operator:
print('PYTHON MODULUS OPERATOR % REMAINDER ')
print('1%4 =', 1%4)
print(1%6) # = 0 r 1
print(6%6) # = 1 r 0
print(7%6) # = 1 r 1
print(22%3) # = 7 r1
print(100% 92) # = 1 r 4

# Remember the Cross Product: aka 
# xb read 'a cross b'. axb = |a|*|b|*sin(theta), where theta is the angle between a and b in the plane shared by a and b. (When theta = 90, bc sin(90deg) = 1 , axb = |a|*|b|). CP is a vector orthagonal to both a and b, and thus normal to the plane connecting a and b. 
# axb = -bxa # anti commutative, the cross product is 

# now we have the centroid, we probably want to set the origin of the item to said centroid. In Qt, we can use self.setTransformOriginPoint(centroid)




In [ ]:
l = [1]*2
print(l)

In [ ]:
#

In [ ]:
x = {1,'2'}

In [ ]:
# https://stackoverflow.com/questions/33746224/in-xml-is-the-attribute-order-important  
#Sort xml attributes 

import xml.etree.ElementTree as ET

tree = ET.fromstring(xml_str)

def getkey(elem):
    # Used for sorting elements by @LIN.
    # returns a tuple of ints from the exploded @LIN value
    # '1.0' -> (1,0)
    # '1.0.1' -> (1,0,1)
    return float(elem.get('LIN'))

root = etree.fromstring(xml_str)
lines = root.find("PurchaseOrder/LineItems")
lines[:] = sorted(lines, key=getkey)

lines = root.find("")

In [ ]:
x = 10 
def func():
    print(x) # Global variable x is readable from wihtin functions, no problem. 
    # x = x + 1# BUT global var x is NOT available for writing. UnboundLocalError 
    # You should pass x as an argument to make x available within the function. 
func()

In [ ]:
def recurse( lst, res = []): 
    for idx in lst: 
        if isinstance(idx, list):
            recurse(idx, res)
        elif isinstance(idx, (str, float, int)):
            res.append(idx)
    return res 
L = [1,2,3,[4]]
print(recurse(L))

In [ ]:
L = [1,2,3]
def func(lst, res = []):
    L.append(4)
    res.extend(L)
    if len(L) == 4: 
        parent = func(L, res)
        
    return res 

res = func(L)
print(res)
print(L)

In [ ]:
def func(d):
    d['new'] = 'value'
    print("In func", d)

my_dict=  {'ogkey': 'ogvalue'}
print('my_dict:', my_dict)
print()


func(my_dict)
print('Outside func', my_dict)
print("We didn't return, yet the change persisted outside the function, becasue my_dict and d refer to the same object in memeory." )
print('Python objects until it cannot; as in the case when the variable is reassigned, or the object is immutable')
print()


def func2(d):
    d = {'a':1}
    print("Inside func2", d)

func2(my_dict)
print("Outside func2: ",  my_dict)
print("func2 reassigned variable 'd', so d was updated to refer to a new object in memory")
print()


class MyClass:
    value = 10

def func3(obj): 
    obj.value +=1 
    print('Inside func3:', obj.value)

ins = MyClass()
print('Value at start: ', ins.value)
func3(ins)
print(f"Outside func3: " ,ins.value)
print("func3 modified obj.value, which is the same memory as ins, so the obj.value is modified outside the function, without a return value")
print()


def func4(obj):
    obj = MyClass() # This new instance is only known inside func4; local to func4
    obj.value = 100
    print('inside func4:', obj.value)
ins = MyClass()
func4(ins)
print("Outside func4:", ins.value)

# Any (mutable) variable, modified inside a function, will persist outside the function, this is because a variable points to the same memory, (as long as you do not reassign the variable). Thus a 'return' statement is not needed.
# local variables, created inside a function, are known only within the function


In [ ]:
num = 10
lst=[]

def func(num, lst):
    num = num**3 # Ints are mutable. You have to reassign them, to change them-- so the 'change applies outside the function' is bogus for immutables. 
    lst.append(12) # modifies lst, same memory as lst, outside the function
    lst = lst.append(12) # .append() returns None. lst = reassigns variable 'lst' to a new memory-- remaking 'lst' into a local variable -- and local 'lst' becomes None, since .append() returns None.  Yet, lst is also outside the function, and it has TWO '12's in it-- from the two appends!
    print('Inside func:', num, lst)
    
func(num,lst) 
print('Outside func', num, lst)

In [ ]:
# Simple app to practice concepts. (Needs all the imports) 
class MainWindow(QWidget):
    
    
    def __init__(self, parent=None):
        super().__init__()
        layout = QHBoxLayout()
        btnA = QPushButton('A')
        layout.addWidget(btnA)
        
        self.setLayout(layout)
        
if __name__=="__main__":
    app=QApplication([])
    w=MainWindow()
    w.show()
    sys.exit(app.exec())
    
v

In [ ]:
Modifying attributes inside a function affects the original object, unless the attribute itself is immutable.
✅ If you reassign the instance inside the function, it creates a new object and does not change the original.
class MyClass:
    def __init__(self):
        self.data = []

def modify_list_attribute(obj):
    obj.data.append(42)  # Modify the list inside the instance
    print(f"Inside function: {obj.data}")

instance = MyClass()
modify_list_attribute(instance)
print(f"Outside function: {instance.data}")

class MyClass:
    def __init__(self, value):
        self.value = value

def modify_instance(obj):
    obj.value += 1  # Modify an attribute of the instance
    print(f"Inside function: {obj.value}")

instance = MyClass(10)
modify_instance(instance)
print(f"Outside function: {instance.value}")

# Output

# Inside function: 11
# Outside function: 11

#     The function modified obj.value, which changed the instance.value outside the function because both obj and instance point to the same object in memory.












def reassign_instance(obj):
    obj = MyClass(100)  # Creating a NEW instance
    print(f"Inside function: {obj.value}")

instance = MyClass(10)
reassign_instance(instance)
print(f"Outside function: {instance.value}")


In [ ]:
from PySide6.QtWidgets import QApplication, QGraphicsView, QGraphicsScene, QGraphicsPixmapItem,QGraphicsItemGroup, QGraphicsItem
from PySide6.QtSvg import QSvgRenderer
from PySide6.QtSvgWidgets import QGraphicsSvgItem, QSvgWidget
from PySide6.QtGui import QPixmap, QPainter
from PySide6.QtCore import QByteArray, Qt
from lxml import etree as ET 
shape_tags = {"line", "circle", "rect", "ellipse", "polygon", "polyline", "path"}

svg_line ="""<svg xmlns="http://www.w3.org/2000/svg">
<ellipse id ='id1' cx="230" cy="90" rx="30" ry="15" fill="none" stroke="black" stroke-width="2"/>
</svg>
"""
#funfact: set renderer on svg w/ single ellipse. Then create seven SvgItems.sharedRenderer() and get seven ellipses

app = QApplication([])
scene = QGraphicsScene()
view = QGraphicsView()
view.setScene(scene)

# renderer = QSvgRenderer("svgA.svg") # No point in making a renderer for a file, since files may not have id valueibutes for all their elements-- making them useless for QGraphicsSvgItem rendering-- so first, parse the svg with etree, ADD ids if missing, THEN create a renderer, THEN create .setSharedRenderer() on the renderer w/ ids. Woof.
# renderer = QSvgRenderer(QByteArray(svg_line.encode('utf-8')))  
# print(renderer.isValid())# prints an object, but how do I see renderer contents? via its properties 
# print(renderer.dumpObjectInfo())
# print(renderer.dumpObjectTree())
# print(renderer.viewBox())
# print(renderer.elementExists(elemId))

tree = ET.parse('svgA.svg')
print(tree)                             # <lxml.etree._ElementTree object at 0x00000235CEE27A00>
root = tree.getroot()
print('root: ', type(root), root)       # <class 'lxml.etree._Element'>
elems = root.findall('.//')             
print('Findall: ', type(elems), elems)  #  <Element {http://www.w3.org/2000/svg}svg at 0x235cee27ac0>

counter = 0
for elem in elems: 
    tag = ET.QName(elem.tag).localname
    id = elem.get('id')
    text = (elem.text or '').strip('\n')
    valueid=  elem.valueid
    # print('Tag: ',tag, "ID: ",id, 'Text: ',text, 'valueib: ',valueib)
    if tag in shape_tags: 
        # print(ET.tostring(elem, encoding='unicode'))
        if id==None: 
            id = f"auto_id{counter}"
            elem.set('id', id) # Note: doesn't back-modify the file -- just the python variables -- to save these changes back into the file, rewrite the file(not done here) 
            counter += 1 
        # print('did ID update? : ', elem.get('id') )
        # print(id)
        #Yeah, the ElementTree.Element.IDvalue updated-- but renderer is still seeing the origional file. Thus item.setElementId(id) sees no id. 
        # I need to update, or make new, the renderer, use .load(), or better(?), save changes to the file itself. 
        
# renderer = QSvgRenderer(QByteArray(uft-8EncodedStringbinaryString))  
print(elems)
print(type(elems))
print()
renderer = QSvgRenderer(QByteArray(ET.tostring(root, encoding = 'utf-8')))
print(renderer.isValid())
print(renderer.viewBox())
# print(renderer.elementExists(elemId))
# So load my mutated tree into renderer:

# May reuse mutated tree. Iterate thru it, but create SvgItems on this second loop-around, with a renderer loaded with ids
for elem in elems:
    tag = ET.QName(elem.tag).localname
    id = elem.get('id')
    text = (elem.text or '').strip('\n')
    valueib =  elem.valueib
    print('Tag: ',tag, "ID: ",id, 'Text: ',text, 'valueib: ',valueib)

    if tag in shape_tags: 
#Create SvgItems based on my new renderer:
        item = QGraphicsSvgItem()
        # item = makeItem(elem, tag)
        item.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsMovable | QGraphicsItem.GraphicsItemFlag.ItemIsSelectable)
        item.setSharedRenderer(renderer)
        # item.setElementId(None) # .setElementId(None) renders whole svg. But .setElementId('nonexistentId') will render a blank svgItem-- it will be selectable, flags depending, but invisible; nasty to debug.
        # item.setElementId('hey') # 7 'invisible' images... bc 'hey' is not an id...
        item.setElementId(id) #If setElementId() method is called, only the SVG element (and its children) with the passed id will be renderer. This provides a convenient way of selectively rendering large SVG files that contain a number of discrete elements
        scene.addItem(item)
        print()
        
def makeGraphicsItem(elem, tag = None):
    if not tag: 
        tag = ET.QName(elem.tag).localname
    


#         #Oops creating 7 superimposed svgItems atm, bc no elements have an id. Give auto-IDs. Ok, now nothing renders... uncomment setElementId(),and it renders 7 SI items again... switch to QBteArray its good, but setElementID again, its bad again...
# Ok... <MARKER> gets its 'id' but the nested shapes, get auto-generated ids... wait-- markers have to have ids because they are reused by id... I need to set a renderer, which has id's for each shape to be rendered, which entails opening the file, parsing as an ETree, then adding ids, then passing that mutated ETree to a renderer, THEN setting shared renderer on graphics items, with that mutated renderer. Ugh. 

# All Svg elements are now individually rendered... with a total loss of position information... I thought QSVg supported markers... 
# The ONLY point of useing QSvgItems  over QGraphicsItmes was its builtin ability to use the position information from the svg... I thought QtSvg has a parser that could handle this. B
# But of course, Each QSvgItem I create, is its own item, with a .pos() which evidently does NOT detect its svg elements' xy...
# The annoying thing about svg is each analogous coordinate is named different iter-element. xy for rectangle is cx cy for circle ( as circles are based on a center point, while rect xy is a leftmost/topmost point...
# just a note that if every svg element was cast into a svg <path> element, I could parse  all elements with a svg <path> parser, then scene.addItem(item)
# Damn. QT doesn't have support for mapping svg coordinates to SvgItem coordinates. '  
# Also, when I move one discrete svg item, sometimes others move too, which is bad. 
# So I guess after 2 days of learning how to use svg renderers I'm just gonna go back to to making a svg parser 

# STOP should switch to making a Kicad parser-- I'll be ripping .kicad_sym &.fp files for parts I download-- SVGs never come into play, now that I know QTSVG doesn't handle position as I wanted it to.
# Also, all parts need to go to a QGraphics[Rect,Circle,etc]Item(), so forget about svg and build for that... 

# Eventually I'll have a little symbol editor. And my own no BS filetype -- just xml w/o all the typing. '
# circle,ellipse, rect, lrwh 
# pin x1y1x2y2, marker1 marker2 name
# Oh-- I cannot parse kicad files-- bc they don't hold pinRegister info! 



x = float(valueib.get("x", valueib.get("cx", "0")))
y = float(valueib.get("y", valueib.get("cy", "0")))
item.setPos(x, y)

view.show()
app.exec()

In [ ]:
# return exits the entire function, optionally returning a value. 
# Break exits its' while/for loop

In [ ]:
s = set([1,2,3])
print(s) 
print(s[0])

In [ ]:


# QGraphcisItem Parent items propagate their position, transformation, and visibility to all childrenby default. Child items basically treated same as parent.
# Child items use PARENT coordinates( Which is good ) # Child items appear at the parent's origin unless otherwise specified

In [ ]:


import lxml.etree as ET 
# You can't iterate over ElementTree objects -- use .findall(xpath), or iterdescendants, or iterchildren-- create an iterator, then iterate over that  

# etree.parse() returns and ElementTree, representing a complete document from a file -- Thats including namespaces & more.
# ElementTree .iter, .iterfind, .find, .findall.
# Element .iterchildren, .descendants, 

# root = etree.XML('<root><a><b/></a></root>')  
# root = etree.fromstring('<root><a><b/></a></root>')
# root=  etree.parse
# print(type(root))

# some ways to load xml into lxml. .parse() .fromstring() .ElementTree()
tree = ET.parse('eda_symbol.svg')  # -> ElementTree obj
# tree = ET.fromstring(eda_string) 
tree = ET.ElementTree(file = 'eda_symbol.svg')# .parse a xml file into a ElementTree object (same as above)


t = tree.findall('.//')

"ET.QName(elem.tag).localname"
for i in t: 
    print(f'TAG: {ET.QName(i.tag).localname} TYPE: {type(i)} ATTR: {i.attrib} ')
    print(f'HasChild: {i.haschild()}')
    
# Tree perusal
.find/findall() : find the first/all matching subelement(s)
getchildren():-> all (direct) children # Note: lxml no 'hasChildren'
getiterator(): -> an iterator for the subtree( but does not actually iterate on it) 
addnext(self, element) : add elem as sibling after this element
append(self, elem): append elem as a subelem to this element 
#Iterate over the elements! 
ET.Element.iter(self, tag=None, * tags)
    Iterate over this element and all subelements
iterdescendants(): Iterate over all subelements ( like .iter but skips this element)
ET.Elment.iterancestors(self, tag=None, *tags) 
    Iterate over the ancestors of this element(from parent to parent) 
ET.Element.iterchildren(iterate over the descendants ofthis element, in documnt order) 
iterfind(self, path, namespaces) 
itersiblings(preceding=False): iterate over the following or preceding siblings of this element
itertext(): Iterates over the text content of a subtree


# lxml etree xml ET element parsing module  MyFav 
pin_at.getparent().remove(pin_at) Remove elmeent'pin_at' by .gettingParent() then calling .remove('pin_at') on the parent. 
elem.getchildren('tag')
elem.iterdesc('tag') 


In [ ]:
def recursive(lst, res=[]):
    # res = [] Pro question: why does this line break the program? A: Bc it sets res=[] every time we call recursive()--which happens a lot 
    for i in lst: 
        if isinstance(i, (list, tuple)):
            recursive(i,res)
        elif isinstance(i, (str, float, int)):
            res.append(i)
        elif isinstance(i, dict):
            res.append([v for v in i.values()])
    return res


def recursive_search(iter, flat = []):
    for i in iter :
        if isinstance(i, (list, tuple)):
            recursive_search(i, flat)
        else: 
            flat.append(i)
    return flat 

res = recursive([1,[2,3,4,[5,6,7]]])
print(res)

In [ ]:
from sexpdata import loads, dumps  # loadString load a string into sexp; dumpString dump a  sexpression into a string
sexpr = loads('(a "b" "c" "d" (e (f 0 1) ( g( 3 5 h (i j k))) ("f")))') # A python list-- but () not [], and separator is spacebar not comma , and bad documentation 
sexpr = loads('(a "b" "c")')
print(sexpr)
print(type(sexpr))
print(sexpr[0])
print(type(sexpr[0]))
print(sexpr[0].value()) # Alt cast as string, or access [0]
print()

#Strings become 'symbol', if unquoted . 
sexpr = loads('(ThisASymbolBcItsFirstStringNoQuotes ("notSymbol" 1 3 ayo) )')
print(sexpr)

print(type(sexpr[0]))
import sexpdata
if type(sexpr[0]) ==sexpdata.Symbol: 
    print('True')
if isinstance(sexpr[0], sexpdata.Symbol):
    print('True')
    
# # # How the fuck am i supposed to handle these non-standardized-ass pices of shit? 
# #     [(extends "LIBRARY_ID")]            
# #     SYMBOL_PROPERTIES...                                        
# #     "LIBRARY_ID" | "UNIT_ID"                                    
# #     [(pin_names [(offset OFFSET)] hide)]      





In [ ]:


ET.Element().getparent .getnext .getprevious .getroottree .index .insert .keys 
.set 
.tag .text

In [ ]:
if not y: 
    print('3')

In [ ]:
d = {'a':1, 'b':2}
d[0]

In [ ]:
print('a' in ('b', 'c'))

In [ ]:
import lxml.etree as ET
print([i for i in dir(ET.Element('root')) if not '__' in i])
['_init', 'addnext', 'addprevious', 'append', 'attrib', 'base', 'clear', 'cssselect', 'extend', 'find', 'findall', 'findtext', 'get', 'getchildren', 'getiterator', 'getnext', 'getparent', 'getprevious', 'getroottree', 'index', 'insert', 'items', 'iter', 'iterancestors', 'iterchildren', 'iterdescendants', 'iterfind', 'itersiblings', 'itertext', 'keys', 'makeelement', 'nsmap', 'prefix', 'remove', 'replace', 'set', 'sourceline', 'tag', 'tail', 'text', 'values', 'xpath']


In [ ]:
print('a' == 'a' or 'b')
l=[1,2,3]
print(id(l[1:]) == id(l[1:])) # True 

x = l[1:]
y = l[1:]
print(id(x) == id(y)) # False 

x = ["UNIT_ID"]
print('|' in x[1:])
print(x[2:]) # Its ok to slice past a string's index 
print(x[1]) # but you cannot access an index past a string index 


In [ ]:
import sexpdata 
from sexpdata import loads, dumps # loadString load a string into sexp; dumpString dump a  sexpression into a string
import lxml.etree as ET

# x = sexpr[0]
# print(x)
# print()

def recurse_doc_keys(token): # token is a official kicad token. 'symbol' 'at' etc. Fx recurses through the sexp docs, mutating it into a list, intended to be zipped with corresponding values from a kicad sexpr file. 

    def get_token_docs(token):
        docs= documentation.get(token, ()) #loads(None) fails; use loads(()) -> [] . Don't default to [] as loads([]) creates a sexpdata.Brackets object
        print(docs)
        docs = loads(docs) 
        # print(docs)
        return docs

    
    def recurse_docs(docs, keys = [], brack_idx=None):
        for k in docs: 
            if isinstance(k , sexpdata.Symbol): # Note, the docs will often be type sexpdata.Symbol or brackets, less often a string, 
                if '|' not in k.value(): # '|' becomes Symbol('|'), so we gotta use .value() to get string
                    keys.append(k.value()) # [Symbol('at'), Symbol('X'), Symbol('Y'), Brackets(I=[Symbol('ANGLE')])]
                else: 
                    # bad keys.append ([_ for _ in map(str, k) if _ != '|']) #-> [ ['LIBRARY_ID', 'UNIT_ID'] ]
                    index = docs.index(k)
                    next = str(docs.pop(index+1)) 
                    prev = keys.pop(-1) #delete the last key, to be replaced it with the or'd 
                    keys.append([prev, next])
                
            elif isinstance(k, sexpdata.Brackets): # KICAD: sqBrackets denote optional fields-- they may be missing! sexpdata.Brackets behave similar as a list but we must access brack[0] to get the actual data AND there's no sexpdata.Symbol types within Brackets-- [asdf] is just a string
                # IF a bracket, recurse on brack[0] to get the keys(whose values may not be present-- sqBrackets mean optional in kicad) Then, IF that index is present in the kicad file, we have the ability to kvPair it up.
                # ALSO, we need the index of the bracket-- bc its gonna become a key:value pair-- and the value needs to know the name of the nested sexpression
                if not k[0]: 
                    print("Umm why is this Brackets object blank? fxrecurse_docs")
                else:
                    # k[0][0] assumes kiDocs only have brackets with one entry
                    if not brack_idx: 
                        brack_idx = {}
                    if isinstance(k[0][0], str):
                        brack_idx.update({k[0][0].value() : docs.index(k)})
                        recurse_docs(k[0]) # brackets = Brackets(I=[Symbol('ANGLE')])  # brackets[0] = [Symbol('ANGLE')] # brackets[0][0] = symbol('ANGLE') brackets[0][0].value() = ANGLE
                # does 'ANGLE' have docs? 
            elif isinstance(k, (list, tuple)):
                recurse_docs(k, keys=keys) 
            else: 
                keys.append(k)
        return keys, brack_idx
    

    docs = get_token_docs(token)
    print('DOCS: ', docs)
    keys, brack_idx = recurse_docs(docs)
    print('KEYS: ', keys)
    print('Brack_idx: ', brack_idx)
    return keys, brack_idx

def recurse_values(sexpr, values=[], brack_idx = None): #returns values from an sexpression, indended to later get zipped with keys from kicad docs.
    
    for val in sexpr[1:]:  # skip the zeroeth -- actially we should grab it as token & use it to cehck/verify agains the docs 
        if isinstance(val ,str):
            values.append(val)
        elif isinstance(val, (sexpdata.Symbol, sexpdata.Brackets, sexpdata.Parens)):
            print(f"In kicad file, weird type: {type(val)} for val {val} at sexpr index {sexpr.index(val)}")    
        elif isinstance(val, (list, tuple)): 
            if brack_idx: 
                if sexpr.index(val) in brack_idx.values(): # {'ANGLE': 3} 
                    print(sexpr.index(val)) 
                    print((sexpr.index(val) in brack_idx.values()))
                    values.append(val)
                #NOTE that this won't work, if kicad allows some fields to be blank in the kicad_sym files, which I think its does allow. 
                # KICAD : sqBrackets code for optional settings: Ex (paper A0 [portrait]) the page portrait setting is optional.
            # print(f"kicad File has list/tuple @ idx {expr.index(val)}. at idx   )
    print('VALUES: ', values)
    return values

# I expect only the first entry to be a symbol.
def recursive_sexpr(sexpr, keys=[], values =[], ):
    first = sexpr[0]
    if type(first) == sexpdata.Symbol : 
        token = str(first)
        print('Token: ', token, type(token), end='')
        keys, brack_idx = recurse_doc_keys(token)  #only return keys of one sexpression-- at pt pt 
        # I've parsed the documentation(?) for this sexp into a list of keys... time to create some values...
        # print("KEYS: ", keys)
        values = recurse_values(sexpr, brack_idx = brack_idx) # returns values down to last nest -- the massive tail end just won't appear in the zip
    res = dict({keys[0] : dict(zip(keys[1:], values))})
    # print('RES: ', res)
    return res 



sexpr = loads(symbol_string)
print('sexpr: ', sexpr)
print('RES: ', recursive_sexpr(sexpr))
        # Convert the 'at' kicad sexpression into a dict 'at'

    #  How the fuck am i supposed to handle these non-standardized-ass pices of shit? recursivesly!
    #     [(extends "LIBRARY_ID")]            
    #     SYMBOL_PROPERTIES...                                        
    #     "LIBRARY_ID" | "UNIT_ID"                                    
    #     [(pin_names [(offset OFFSET)] hide)]   

# TypeError: unhashable type: 'list' occurs because token is a list, yet you are trying to set it as a key in a dictionary. dict keys must be hashable, and lists are not, because lists are mutable 



In [ ]:
# behavior of .pop() while iterating over a list: Its allowed, it will mutate your list, so be careful 

l = [1,2,3,4,5,6,7]

for i in l: 
    print(i)
    l.pop(-1)
    print(l)
    print()
    
print(l)

# for i in l[1:]: # 
#     print('i:',i)
#     l.pop(-1)
#     print(l)
#     print()
# does slicing create a copy of the list? 


In [ ]:
.pop() method .pop method  pop method

l = [1, 2, 3, 4, 5]

# Create a function to remove items from the list
while l:
    element = l.pop()
    print(f"Removed element: {element}")
print("List is now empty:", l)
# Good to do tasks one by one

In [ ]:
l = [0,1,2,3,4,5]
for i in l: 
    print(l, 'popping: ', i)
    l.pop(i)
    # You can mutate a list while iterating over it. (dicts disallow this). But the behavior can be hard to predict. Consider saving to a second list, then iterating over that copy
    

In [ ]:
#Python Iterators iterables 

#In Python, an iterator is an object which supports the methods __iter__() and __next__()
it = iter(l)
print(it) # <list_iterator object at 0x00000282A1790D30>
print(next(it))
print(next(it))
print(next(it))
print(next(it))
print(next(it))
print(next(it))
# print(next(it)) StopIteration
# 
# The for loop creates an iterator object and executes the next() method for each loop.
for i in l: 
    print(i)
    
#Implement the __iter__ and __next__ methods in your classes to make your instance objects iterable 


In [ ]:
L = [1,2,3]
print(L[:-1])
print(L[0:2])
L.pop(-1)

In [ ]:
for

In [ ]:
file="parts/KiCADv6/KiCADv6/STM32C011J4M6.kicad_sym"

with open(file) as fo: 
    lines=fo.readlines()
s = ''.join(lines)
s = loads(s)
print('Len: ', len(s), s )
for i in s: 
    print(i)

for sexpr in s: 



In [ ]:
def recurse(lst, res = []): # provide 'res' argument, to pass the results we've already gotten, if any. 
    for i in lst: 
        if isinstance(i, list):
            res = recurse(i , res) # We set res equal to the return of our function, so that we can access the latest res. 
        else: 
            res.append(i)
        return res  #we return res, so that we can access res outside the function
    
def recurse(L, res=[]):
    for i in L: 
        if isinstance(i, list): 
            recurse(i, res) 
            # res = recurse(i, res)  # How come these are euqal? BC res.append(i) mutates res in-place, and bc res never goes outside this function, till its 
            
            
        else:
            res.append(i)
    return res 
    
L = [1,[2,[3, [4], [5, 6 ,7 ]]]]
print(recurse(L))
print()

def recurse(lst, res=[]):
    for i in lst: 
        if isinstance(i,list):
            recurse(i,res)
        else: 
            res.append(i)
    return res 
print(recurse(L))

In [ ]:
def recurse(L, res=[]):
    for i in L: 
        if isinstance(i, list): 
            recurse(i, res)  
            # res = recurse(i, res)  # This does the same thing!
        else:
            res.append(i)
    return res

lst = [1, [2, [3, 4], 5], 6]
result = recurse(lst)
print(result)  # Output: [1, 2, 3, 4, 5, 6]

result1 = recurse([1, 2])
result2 = recurse([3, 4])
print(result1)  # Output: [1, 2, 3, 4]  (potentially confuing!)
print(result2)  # Output: [1, 2, 3, 4]  (potentially confuing!)

# Confuze about local variables, in/out of their functions,  mutable objects, copies, mutating in-place vs oin a copy, 
#  Python tries to reuse the same object, when it can. If you reuse variable name, python will let you. 
# immutables (like int, tuple) get copies made of them, bc the OG is immutable tho.

In [ ]:

def modify_dict(d):
    d["new_key"] = "new_value"
    print(f"Inside function: {d}")

my_dict = {"original_key": "original_value"}
modify_dict(my_dict)
print(f"Outside function: {my_dict}")
# Inside function: {'original_key': 'original_value', 'new_key': 'new_value'}
# Outside function: {'original_key': 'original_value', 'new_key': 'new_value'}
# The dictionary my_dict was modified inside the function.

# The change persisted outside the function because d and my_dict refer to the same dictionary object in memory.

def reassign_dict(d):
    d = {"new_key": "new_value"}  # This creates a NEW dictionary
    print(f"Inside function: {d}")

my_dict = {"original_key": "original_value"}
reassign_dict(my_dict)
print(f"Outside function: {my_dict}")

# Output

# Inside function: {'new_key': 'new_value'}
# Outside function: {'original_key': 'original_value'}

#     The function created a new dictionary and assigned d to it.

#     But this did not affect my_dict outside the function, because d was now pointing to a different object in memory.

In [ ]:
# l = [0,1,[2,[3]]]
# print(3 in l)
# print(1 in l)

# print(l[-1])

In [ ]:
a = [1, 3, 5, 7, 9, 11]
val = 8

for i in a:
    if i == val:
        print(f"Found at {i}!")
        break
    
else:
    print(f"not found")
    
print('done')

In [ ]:
print(loads("[]")) # Brackets(I=[])
print(loads("()")) # []
# print(loads(None)) Fails 
# print(loads("")) Fails 
print(())

In [ ]:
s =     """
("LIBRARY_ID" | "UNIT_ID" )
"""         
s=loads(s)
print(s)

k=[]
k.append ([_ for _ in map(str, s) if _ != '|'])
print(k)



In [ ]:
from sexpdata import Brackets, dumps, loads
x = ['a', 1, 2, 3, 4, [5,[6]]]
brack = Brackets('a',1,2,3,4,Brackets(5,Brackets(Brackets(6))))
x = dumps(x)
brack = dumps(brack)
print(x)
print(brack)
print()

x = loads(x)
brack = loads(brack)
print(x)
print(brack)
print()
print(x[0])
print(brack[0])
print()
for i in brack[0]: 
    print(i) # OK so it gives a Brackets() wrapper; still behaves as a list.
print(isinstance(brack, Brackets))
print(isinstance(brack, list))
# keys.append(k[0][0].value()) # brackets = Brackets(I=[Symbol('ANGLE')])  # brackets[0] = [Symbol('ANGLE')] # brackets[0][0] = symbol('ANGLE') brackets[0][0].value() = ANGLE




In [ ]:
import sexpdata 
import lxml.etree as etree






                                                 





In [ ]:
sexp_str = "(at X Y (Brackets (I ANGLE)))"

# Parse the S-expression
parsed = sexpdata.loads(sexp_str)
print(parsed)

In [ ]:
    # API functions:
    'load', 'loads', 'dump', 'dumps', 'parse',
    # Utility functions:
    'car', 'cdr',
    # S-expression classes:
    'Symbol', 'String', 'Quoted', 'Brackets', 'Parens',
]

In [ ]:
from sexpdata import loads, dumps # loadString load a string into sexp; dumpString dump a  sexpression into a string
from collections.abc import Iterable

# def recursive_search_This_Stack_Overflows( iter, flat = []): 
#     for i in iter: 
#         if isinstance(i, Iterable): 
#             recursive_search(i, flat)
#         else: 
#             flat.append(i)            
#     return flat # SO -- I need to EXCLUDE string iterables. 

def recursive_search(iter, flat = []): 
    for i in iter:
        if isinstance(i, Iterable) and not isinstance(i, (str, bytes)):  # Check for iterables (excluding strings and bytes)
            recursive_search(i, flat)  # Recurse if it's an iterable
        else:
            flat.append(i)  # Append non-iterables directly
    return flat

def recursive_search(iter, flat = []):
    for i in iter :
        if isinstance(i, (list, tuple)):
            recursive_search(i, flat)
        else: 
            flat.append(i)
    return flat 



In [ ]:
kicad = """(kicad_symbol_lib (version 20211014) (generator kicad_symbol_editor)
  (symbol "STM32C011J4M6" (pin_names (offset 0.254)) (in_bom yes) (on_board yes)
    (property "Reference" "U" (id 0) (at 63.5 10.16 0)
      (effects (font (size 1.524 1.524)))
    )
    (entry "Value" "STM32C011J4M6" (id 1) (at 63.5 7.62 0)
      (effects (font (size 1.524 1.524)))
    )
    (entry "Footprint" "SO-8_STM" (id 2) (at 0 0 0)
      (effects (font (size 1.27 1.27) italic) hide)
    )
    (property "Datasheet" "STM32C011J4M6" (id 3) (at 0 0 0)
      (effects (font (size 1.27 1.27) italic) hide)
    )
    (property "ki_keywords" "STM32C011J4M6" (id 4) (at 0 0 0)
      (effects (font (size 1.27 1.27)) hide)
    )
    (property "ki_locked" "" (id 5) (at 0 0 0)
      (effects (font (size 1.27 1.27)) hide)
    )
    (property "ki_fp_filters" "SO-8_STM SO-8_STM-M SO-8_STM-L" (id 6) (at 0 0 0)
      (effects (font (size 1.27 1.27)) hide)
    )
    (symbol "STM32C011J4M6_0_1"
      (polyline
        (pts
          (xy 7.62 5.08)
          (xy 7.62 -17.78)
        )
        (stroke (width 0.127) (type default) (color 0 0 0 0))
        (fill (type none))
      )
      (polyline
        (pts
          (xy 7.62 -17.78)
          (xy 119.38 -17.78)
        )
        (stroke (width 0.127) (type default) (color 0 0 0 0))
        (fill (type none))
      )
      (polyline
        (pts
          (xy 119.38 -17.78)
          (xy 119.38 5.08)
        )
        (stroke (width 0.127) (type default) (color 0 0 0 0))
        (fill (type none))
      )
      (polyline
        (pts
          (xy 119.38 5.08)
          (xy 7.62 5.08)
        )
        (stroke (width 0.127) (type default) (color 0 0 0 0))
        (fill (type none))
      )
      (pin bidirectional line (at 127 -2.54 180) (length 7.62)
        (name "PB7/PC14-OSCX_IN" (effects (font (size 1.27 1.27))))
        (number "1" (effects (font (size 1.27 1.27))))
      )
      (pin power_in line (at 127 -7.62 180) (length 7.62)
        (name "VDD/VDDA" (effects (font (size 1.27 1.27))))
        (number "2" (effects (font (size 1.27 1.27))))
      )
      (pin power_in line (at 0 -12.7 0) (length 7.62)
        (name "VSS/VSSA" (effects (font (size 1.27 1.27))))
        (number "3" (effects (font (size 1.27 1.27))))
      )
      (pin bidirectional line (at 0 0 0) (length 7.62)
        (name "PA0/PA1/PA2/PF2-NRST" (effects (font (size 1.27 1.27))))
        (number "4" (effects (font (size 1.27 1.27))))
      )
      (pin bidirectional line (at 0 -2.54 0) (length 7.62)
        (name "PA11[PA9]/PA8" (effects (font (size 1.27 1.27))))
        (number "5" (effects (font (size 1.27 1.27))))
      )
      (pin bidirectional line (at 0 -5.08 0) (length 7.62)
        (name "PA12[PA10]" (effects (font (size 1.27 1.27))))
        (number "6" (effects (font (size 1.27 1.27))))
      )
      (pin bidirectional line (at 0 -7.62 0) (length 7.62)
        (name "PA13" (effects (font (size 1.27 1.27))))
        (number "7" (effects (font (size 1.27 1.27))))
      )
      (pin bidirectional line (at 127 0 180) (length 7.62)
        (name "PB6/PA14-BOOT0/PC15-OSCX_OUT" (effects (font (size 1.27 1.27))))
        (number "8" (effects (font (size 1.27 1.27))))
      )
    )
  )
  )"""
kicad_short= """(kicad_symbol_lib (version 20211014) (generator kicad_symbol_editor)
  (symbol "STM32C011J4M6" (pin_names (offset 0.254)) (in_bom yes) (on_board yes)
    (property "Reference" "U" (id 0) (at 63.5 10.16 0)
      (effects (font (size 1.524 1.524)))
    )
    (property "Value" "STM32C011J4M6" (id 1) (at 63.5 7.62 0)
      (effects (font (size 1.524 1.524)))
)))"""
from collections import defaultdict


from sexpdata import loads, dumps
import sexpdata
#Unquoted string becomes symbol
# kicad = kicad_short
kicad = loads(kicad)


symbols_list = kicad[3:]

properties = [] 
symbols = []                  # RobsRobots symbols list 
parts =[]                     # RobsRobots part list
other = {}

def parse_graphic(sexpr, part):
  print(part)
  kicad_shapes = (
      "polyline", "arc", "circle", "curve",
      "rect", "polygon", "filled_polygon",
      "text", "text_box",
      "bitmap", "bezier"
  )
  kicad_shapes = dict(zip(kicad_shapes, [ [] for i in range(len(kicad_shapes)) ] )) # set all values in dict to empty list []. Alt use from collections import defaultdict
  print(kicad_shapes)
  
  graphic = sexpr[2:]
  # pprint(graphic, indent = 2, width = 400)
  # print()
  
  for item in graphic: 
      token = str(item[0])
      print('item: ', item)  #item:  [Symbol('polyline'), ...
      if token == 'pin':
          # item:  [Symbol('pin'), Symbol('power_in'), Symbol('line'), [Symbol('at'), 127, -7.62, 180], ...
          pin_electrical_type = str(item[1])
          # print(pin_electrical_type)
          pin_shape = str(item[2])
          pin_at = item[3]
          other_pin_info = item[4:]
          # print('other pin_info: ', pin_info)
          for pin_info in other_pin_info:
              if str(pin_info[0]) =='length': 
                  pin_length = str(pin_info[1])
              if str(pin_info[0]) == 'name': 
                  pin_name=str(pin_info[1])
                  # print(pin_name)
              elif str(pin_info[0]) =='number':
                  pin_number= str(pin_info[1])
                  # print(pin_number)
          part['pins'].append( {'pin_name': pin_name, 'pin_electrical_type': pin_electrical_type, 'pin_shape':pin_shape, 'pin_at':pin_at, 'pin_number': pin_number, 'pin_length': pin_length} ) # add pin as a record
          
      if token in kicad_shapes: 
        print(part)
        print()
        part['symbol'].get(token, []) # Initialize symbol[token] to a list, if it DNE. 
        if token =='polyline': 
          polyline = []
          COORDINATE_POINT_LIST, STROKE_DEFINITION, FILL_DEFINITION = item[1:4]
          #  [Symbol('pts'), [Symbol('xy'), 7.62, 5.08], [Symbol('xy'), 7.62, -17.78]]
          for pts in COORDINATE_POINT_LIST: 
            polyline.append( ( pts[1], pts[2] ) ) 
          data = polyline  
      part['symbol'][token].append(data) # symbol['Polyline'].append( ((1,2), (4,6), ...) ) 
  return part


In [ ]:
# python any 


# Check if any of the items in a list are True:
print(any([False, False, False]))
print(any([False, False, True]))
print()
# Check if any items in a list are strings: 


lst = [1,2,'3']
res=[]
for item in lst: 
    res.append(isinstance(item, str))
print(res)
print(any(res))




In [ ]:
for symbol_list in symbols_list: 
  
  if not isinstance(symbol_list[0], sexpdata.Symbol) and str(symbol_list[0]) == 'symbol': # [Symbol('symbol'), 'STM32C011J4M6', [Symbol('pin_names'),
    print('Something wrong')

  part_name = symbol_list[1]
  print('part_name: ', part_name)
  print()
  part = {}
  part['Name'] = part_name  # {'Name': 'STM32CO46}
  part['symbol_name']= part_name  # part['symbol'] -> STM32C046 or 'C_Small' -- just the name of the symbol
  part['properties'] = {} # initialize a properties dict
  part['symbol'] = {}
  part['pins'] = []
    
  for entry in symbol_list[2:]: 
    token = str(entry[0])
    print(token)
    if token == 'symbol': # codes for drawing entry
      part = parse_graphic(entry, part)  
    if token == 'property':
      part['properties'].update({entry[1]: entry[2]})
    else: 
      other[token] = str(entry[1])
    

parts.append(part)
symbols.append(part['symbol'])


print('symbol: ', part['symbol'])
print('properties: ', properties)
print('pins: ', pins)
print('parts: ', parts)
print('other: ', other)

  

In [ ]:
import sexpdata
from pprint import pprint
s = """(symbol "STM32C011J4M6_0_1"
    (polyline
    (pts
        (xy 7.62 5.08)
        (xy 7.62 -17.78)
    )
    (stroke (width 0.127) (type default) (color 0 0 0 0))
    (fill (type none))
    )
    (stroke (width 0.127) (type default) (color 0 0 0 0))
    (fill (type none))
    (pin bidirectional line (at 127 -2.54 180) (length 7.62)
    (name "PB7/PC14-OSCX_IN" (effects (font (size 1.27 1.27))))
    (number "1" (effects (font (size 1.27 1.27))))
    )
    (pin power_in line (at 127 -7.62 180) (length 7.62)
    (name "VDD/VDDA" (effects (font (size 1.27 1.27))))
    (number "2" (effects (font (size 1.27 1.27))))
    )

)"""
s = sexpdata.loads(s) 

graphic = s[2:]
# pprint(graphic, indent = 2, width = 400)
# print()
    
symbol = {}
pins = []
stroke = {}
brush = {}

for item in graphic: 
    print('item: ', item)  #item:  [Symbol('polyline'), ...
    token = str(item[0]) 
    if token == 'pin':
        # item:  [Symbol('pin'), Symbol('power_in'), Symbol('line'), [Symbol('at'), 127, -7.62, 180], ...
        pin_electrical_type = str(item[1])
        # print(pin_electrical_type)
        pin_shape = str(item[2])
        pin_at = item[3]
        other_pin_info = item[4:]
        # print('other pin_info: ', pin_info)
        for pin_info in other_pin_info:
            if str(pin_info[0]) == 'name': 
                pin_name=str(pin_info[1])
                # print(pin_name)
            elif str(pin_info[0]) =='number':
                pin_number= str(pin_info[1])
                # print(pin_number)
            elif str(pin_info[0]) =='length': 
                pin_length = str(pin_info[1])
                
pins.append( {'pin_name': pin_name, 'pin_electrical_type': pin_electrical_type, 'pin_shape':pin_shape, 'pin_at':pin_at, 'pin_number': pin_number, 'pin_length': pin_length} ) # add pin as a record
print(pins)

# [Symbol('pin'),
#  Symbol('power_in'),
#  Symbol('line'),
#  [Symbol('at'), 127, -7.62, 180],
#  [Symbol('length'), 7.62],
#  [Symbol('name'),
#   'VDD/VDDA',
#   [Symbol('effects'), [Symbol('font'), [Symbol('size'), 1.27, 1.27]]]],
#  [Symbol('number'),
#   '2',
#   [Symbol('effects'), [Symbol('font'), [Symbol('size'), 1.27, 1.27]]]]]


In [ ]:
``

In [ ]:
def parse_symbols(data):
    if isinstance(data, list):
        return [parse_symbols(item) for item in data]
    elif isinstance(data, sexpdata.Symbol):
        token = str(data)
        
        return str(data)  # Convert Symbols to strings
    else:
        return data  # Return numbers or other values unchanged

parse = parse_symbols(kicad)
print(parse)
kicad = parse[3:]
# print(kicad)
from pprint import pprint 
pprint(kicad,indent = 1, width = 2000)

# import json 
# print(json.dumps(kicad, indent = 2))




In [ ]:
symbol 
property 
pin_names 
pin_numbers
in_bom
on_board

within graphic_items: 
polyline, arc, ...



 


#https://dev-docs.kicad.org/en/file-formats/sexpr-intro/index.html#_position_identifier
if token == 'at': 
    x = sexpr[1] 
    y = sexpr[2]
    ANGLE = sexpr[3] 
    #  Symbol text ANGLEs are stored in tenth’s of a degree. All other ANGLEs are stored in degrees. 
#   (at
#     X                                                           
#     Y                                                           
#     [ANGLE]                                                     
#   )
if token =='pts': 
    points =[]
    for p in sexpr: 
        points.append(sexpr[1])
        points.append(sexpr[2])
 	# The xy token defines a single X and Y coordinate pair. The number of points is determined by the object type.
#   (pts
#     (xy X Y)                                                    
#     ...
#     (xy X Y)
#   )
if token == 'stroke': 
    for sexpr in token: 
        if sexpr[0] == 'width':
            width == sexpr[1]
        elif sexpr[1] ==' type':
            type = sexpr[1]
        elif sexpr[0] =='color':
            r,g,b,a = [sexpr[i] for i in [1,2,3,4]] 
#   (stroke
#     (width WIDTH)                                               
#     (type TYPE)                                                 
#     (color R G B A)                                             
#   )
 	# The width token attribute defines the line width of the graphic object.
	# The type token attribute defines the line style of the graphic object. Valid stroke line styles are:

    # dash

    # dash_dot

    # dash_dot_dot (from version 7)

    # dot

    # default

    # solid

	# The color token attributes define the line red, green, blue, and alpha color settings.

#Text objects optional effects 
if token == 'effects': 

 effects size = 1.5 if 'property' == 'Reference' or 'Value'
else:
effects size = 1.3 and hide = True
#  (effects
#     (font                                                       
#       [(face FACE_NAME)]                                        
#       (size HEIGHT WIDTH)                                       
#       [(thickness THICKNESS)]                                   
#       [bold]                                                    
#       [italic]                                                  
#       [(line_spacing LINE_SPACING)]                             
#     )
#     [(justify [left | right] [top | bottom] [mirror])]          
#     [hide]                                                      
#   )

 #SKIP
#   (paper
#     PAPER_SIZE | WIDTH HEIGHT                                   
#     [portrait]                                                  
#   )

 #SKIP
#   (title_block
#     (title "TITLE")                                             
#     (date "DATE")                                               
#     (rev "REVISION")                                            
#     (company "COMPANY_NAME")                                    
#     (comment N "COMMENT")                                       
#   )

In [ ]:
import sexpdata
s = "(xy 0 1 )"
s = sexpdata.loads(s)
print(s)

            

In [ ]:

# print(kicad)
import pandas as pd 
df= pd.DataFrame(kicad)

print(df.head) 
display(df) 
df.pop(0) # column 0 is just the 'symbol' 
display(df)

# print(df[0])
# df.pop(item)# item: Label of column to be popped.
# df.pop
# df.setindex() Set the DataFrame index(row labels) using existing columns.
# df.set_axis() Set row (0 or 'index') or col(1 or 'columns') labels.
# df.set_axis(['I', 'II'], axis='columns')
#            I  II
#         0  1   4
#         1  2   5
#         2  3   6

# create columns, the names of the dataframe's columns, for the purpose of naming the df's column index. 
columns = []
graphical_object = []
c = 0
all_graphics = []
for i in df:
    cell = df[i][0]
    print(cell)
    if isinstance(cell, str) and i>0: 
        columns.append('part_name')  # cell='symbol' if i==0, and I don't care about that. Note that 'symbol' if i!=0 is ok
    else: 
        # I expect lists form now on
        if isinstance(cell, list):
            colNm = str(cell.pop(0)) # rm & return first item
            print()
            # print(colNm)
            if colNm == 'symbol':
                # Kicad Graphical object
                columns.append(f'graphics_{c}')
                all_graphics.append(cell)
                c+=1
            else:
                columns.append(colNm)
        

# print(columns)
df = df.set_axis(columns, axis = 'columns')
display(df)
print(df.columns)
print(df.head)
print()
# for i in 
        

In [ ]:
print(all_graphics)
df= pd.

In [ ]:
from lxml import etree 
elem =   '<pad name="1" type="smd" shape="rect" width="0.4318" height="0.2794" layers="F.Cu , F.Mask, F.SilkS" left="-1.6159" top="-1.6397" c_x="-1.4" c_y="-1.5"/>'
    
t = etree.ElementTree(etree.fromstring(elem)) # Get an ElementTree from a string
print(t)

In [ ]:
import lxml.etree as ET

def list_to_xml(parent, data):
    """Recursively convert list structure to XML."""
    if isinstance(data, list):
        if isinstance(data[0], str):  # First element as tag
            tag = data[0]
            elem = ET.SubElement(parent, tag)
            for item in data[1:]:
                list_to_xml(elem, item)  # Recurse into children
        else:  # Nested lists (multiple elements)
            for item in data:
                list_to_xml(parent, item)
    elif isinstance(data, (int, float, str, sexpdata.Symbol)):
        parent.text = str(data)

def convert_to_xml(data):
    root = ET.Element("root")
    list_to_xml(root, data)
    return ET.tostring(root, encoding="unicode", pretty_print=True)

# Example usage
structured_data = [['STM32C011J4M6_0_1', ['polyline', ['pts', ['xy', 7.62, 5.08], ['xy', 7.62, -17.78]], ['stroke', ['width', 0.127], ['type', 'default'], ['color', 0, 0, 0, 0]], ['fill', ['type', 'none']]], ['pin', 'bidirectional', 'line', ['at', 127, -2.54, 180], ['length', 7.62], ['name', 'PB7/PC14-OSCX_IN', ['effects', ['font', ['size', 1.27, 1.27]]]], ['number', '1', ['effects', ['font', ['size', 1.27, 1.27]]]]]]]

xml_output = convert_to_xml(structured_data)
print(xml_output)

In [ ]:
l = [1,2,3]
l.pop(0)
print(l)


In [ ]:
d = {'symbol':[0,0,1], 'symbol': [0,4,5]}
print(d)
df = pd.DataFrame(d)
display(df)

In [ ]:

# svg renderer constructed from file: 
renderer = QSvgRenderer("svgA.svg") 
# svg renderer constructed from bytes/binary/bytes object/bytes string/binary string: encode in utf-8:
renderer = QSvgRenderer(QByteArray(svg_line.encode('utf-8')))  
print(renderer.isValid())# prints an object, but how do I see renderer contents? via its properties 
print(renderer.dumpObjectInfo())
print(renderer.dumpObjectTree())
print(renderer.viewBox())
print(renderer.elementExists(elemId))

# Some of the most used encodings are utf-8, asci, latin-1 and utf-32

In [ ]:
# MyFav Enums Qt.Enums 

from PySide6.QtCore import Qt
Qt.MouseButton.LeftButton
Qt.CursorShape.ClosedHandCursor

In [ ]:
svg= """<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" width="100" height="100" viewBox="0 0 100 100">
<path fill-rule="nonzero" fill="rgb(100%, 0%, 0%)" fill-opacity="1" d="M 90 50 C 90 72.089844 72.089844 90 50 90 C 27.910156 90 10 72.089844 10 50 C 10 27.910156 27.910156 10 50 10 C 72.089844 10 90 27.910156 90 50 "/>
</svg>"""


In [ ]:
# Minimalist Example QGraphicsItem Subclass, draggable. Mime data, dragphoto effect, paint override, boundingRectOverride.

from PySide6.QtCore import (QEasingCurve, 
                            QLineF,
                            QMimeData,
                            QByteArray,
                            QPoint,
                            QPointF,
                            QRandomGenerator,
                            QRectF,
                            QTimeLine,
                            Qt,
                            )
from PySide6.QtGui import (QBrush,
                           QPen, QColor,
                           QImage, QPixmap,
                           QPainter,
                           QTransform,
                           QDrag,
                           
                           )
from PySide6.QtWidgets import (QApplication,
                               QGraphicsView, 
                               QGraphicsScene, 
                               QGraphicsItem,
                               QGraphicsItemAnimation,
                               )
import sys 

class MyQGraphicsItem(QGraphicsItem):
    bounding_rect = (-100, -100, 200, 200)
    rect = (0,0,100,100)
    
    def __init__(self, parent = None):
        super().__init__()
        self.setCursor(Qt.CursorShape.OpenHandCursor)
        
        
    def boundingRect(self):
        return QRectF(*self.bounding_rect)
    def paint(self, painter, option, widget):
        painter.setPen(QColor(255, 10, 10))
        painter.setBrush(Qt.GlobalColor.green)
        painter.drawEllipse(*self.rect)
        
    def mousePressEvent(self, event):
        self.setCursor(Qt.CursorShape.ClosedHandCursor)
    def mouseMoveEvent(self, event):

        drag = QDrag(event.widget())
        mime = QMimeData()
        drag.setMimeData(mime)
        
        drag.exec()
        
    def mouseReleaseEvent(self, event):
        self.setCursor(Qt.CursorShape.OpenHandCursor)
    
# I actually dont want to xfer any data with my drag-- I want to be able to reposition the Graphics item. 
# May not ~need~ a a QDrag() to do this. I could 
    def mouseMoveEvent(self, event): 
        st = event.buttonDownScenePos()
        cur = event.scenePos()
        
        event.setpos
        
             class MainWindow(QWidget):
    
    def __init__(self, parent=None):
        layout = QHBoxLayout()
        btnA = QPushButton('A')
        layout.addWidget(btnA)
        
        self.setLayout(layout)
        

if __name__ == "__main__": 
    app = QApplication([])
    w = MainWindow()
    w.show()
    app.exex(sys.exit())
        
        
if __name__ == '__main__':
    app = QApplication(sys.argv)

    scene = QGraphicsScene(-200, -200, 400, 400)
    item = MyQGraphicsItem()
    
    scene.addItem(item)

    view = QGraphicsView(scene)
    view.setRenderHint(QPainter.RenderHint.Antialiasing)
    view.setViewportUpdateMode(QGraphicsView.ViewportUpdateMode.BoundingRectViewportUpdate)
    view.setBackgroundBrush(QColor(230, 200, 167))
    view.setWindowTitle("Window Title")
    view.show()

    sys.exit(app.exec())
        

        
        

In [ ]:
divide = 10/3   # divide operator aka normal division 
floorDivide = 10//3         # floorDivide aka round down 
rem = 10%3      #'modulo' operator aka remainder

print(floorDivide)
print(divide)
print(rem)
print(divmod(10,3))

def

In [ ]:
l= [1,2,3]
l.extend(l)
print(l)
x = [0] + [1]
print(x)


In [ ]:
# Mouse Event Data:

# # pos(), scenePos(), screenPos(), for mouse cursor positions, realtime.
# # buttonDownPos(), buttonDownScenePos(), buttonDownScreenPos(), for mousecursor positions, when mouse button was pressed; @ start of event. 
#         event.buttonDownScreenPos(Qt.MouseButton.btn)           # ->                      
#         event.buttonDownPos(Qt.MouseButton.btn)                 # ->                  
#         event.buttonDownScenePos(Qt.MouseButton.btn)            # ->                      
#         event.pos()                                             # ->      
#         event.screenPos()                                       # ->              
#         event.scenePos()                                        # -> 

# BUT you can make a QGraphics item draggable across it's scene, but setting it's selectable&movable flags. 




In [ ]:
#Crash Course QGraphicsItem -- Must #Reimplement methods .paint(self,painter,option,widget) and .boundingRect()->QRectF in QGraphicsItem subclasses. .paint() renders whatever you make your painter draw. .boundingRect() is used as 'where the object is' -- Ensure the boundingRect matches up correctly to the painting. For mouse interaction, set QGraphicsRectItem.ItemIsMovable and QGraphicsRectItem. ItemIsSelectable). To do your own code upon mouse interactions, reimplement the event handlers mouse(Press,Release,Move,DoubleClick)Event() to automatcally .accept() the event, then write your code, then call base implementation. Reimplement mousePressEvent() to become the 'mouse grabber'. Only mouse grabbers can receive further mouse events (Move,Release,DoubleClick). 

# Note that dragging an object around the scene is not a dragNDrop operation(which has mime, QDrag, is very complicated, etc.) 
# I thought I would have to do some mafs but I do not. 
from PySide6.QtWidgets import QApplication, QGraphicsScene, QGraphicsView, QGraphicsRectItem, QGraphicsEllipseItem, QGraphicsItemGroup
from PySide6.QtCore import Qt

#QGraphicsItemGroup.addToGroup()
#QGraphicsItemGroup.setFlags(move|select)

class DraggableGroup(QGraphicsItemGroup):
    def __init__(self, rect, circle):
        super().__init__()
        self.addToGroup(rect)
        self.addToGroup(circle)
        self.setFlags(QGraphicsItemGroup.ItemIsMovable | QGraphicsItemGroup.ItemIsSelectable)

app = QApplication([])
scene = QGraphicsScene()

rect = QGraphicsRectItem(50, 50, 100, 100)
rect.setBrush(Qt.blue)
circle = QGraphicsEllipseItem(80, 80, 50, 50)
circle.setBrush(Qt.red)

group = DraggableGroup(rect, circle)
scene.addItem(group)

view = QGraphicsView(scene)
view.setRenderHint(view.renderHints())
view.show()
app.exec()


In [ ]:
# Hardly working grouped button 
from PySide6.QtWidgets import QApplication, QGraphicsScene, QGraphicsView, QGraphicsRectItem, QGraphicsEllipseItem, QGraphicsItemGroup, QPushButton, QVBoxLayout, QWidget
from PySide6.QtCore import Qt

class DraggableGroup(QGraphicsItemGroup):
    def __init__(self, rect, circle):
        super().__init__()
        self.rect = rect
        self.circle = circle
        self.setGroup(True)
    
    def setGroup(self, group):
        if group:
            self.addToGroup(self.rect)
            self.addToGroup(self.circle)
            self.setFlags(QGraphicsItemGroup.ItemIsMovable | QGraphicsItemGroup.ItemIsSelectable)
        else:
            self.scene().removeItem(self)
            self.scene().addItem(self.rect)
            self.scene().addItem(self.circle)
            self.rect.setFlags(QGraphicsRectItem.ItemIsMovable | QGraphicsRectItem.ItemIsSelectable)
            self.circle.setFlags(QGraphicsEllipseItem.ItemIsMovable | QGraphicsEllipseItem.ItemIsSelectable)

def toggle_group():
    global grouped
    if grouped:
        scene.removeItem(group)
        scene.addItem(rect)
        scene.addItem(circle)
        rect.setFlags(QGraphicsRectItem.ItemIsMovable | QGraphicsRectItem.ItemIsSelectable)
        circle.setFlags(QGraphicsEllipseItem.ItemIsMovable | QGraphicsEllipseItem.ItemIsSelectable)
        button.setText("Group")
    else:
        scene.removeItem(rect)
        scene.removeItem(circle)
        scene.addItem(group)
        button.setText("Ungroup")
    grouped = not grouped

app = QApplication([])
scene = QGraphicsScene()

rect = QGraphicsRectItem(50, 50, 100, 100)
rect.setBrush(Qt.blue)
circle = QGraphicsEllipseItem(80, 80, 50, 50)
circle.setBrush(Qt.red)

group = DraggableGroup(rect, circle)
grouped = True
scene.addItem(group)

view = QGraphicsView(scene)
view.setRenderHint(view.renderHints())

button = QPushButton("Ungroup")
button.clicked.connect(toggle_group)

layout = QVBoxLayout()
layout.addWidget(view)
layout.addWidget(button)

container = QWidget()
container.setLayout(layout)
container.show()

app.exec()


In [ ]:
# etree module lxml module python 
import lxml 

In [ ]:

# XML NAMESPACE 
# An xml namespace : xmlns="http://www.w3.org/2000/svg">. Namespaces are often websites, though they need not be; this is common practice because websites, which are globally registered, are guaranteed unique. 
# Give the alias 'svg' to a namespace inline with its xml: <componenet xmlns:svg="http://www.w3.org/2000/svg">. (in python, ElementTree also lets you alias with a mapper alias dict)
"""
<component xmlns="http://example.com/electronics">  this xmlns is like one folder
  <name>Resistor</name>
  <value>10kΩ</value>
</component>

<svg xmlns="http://www.w3.org/2000/svg">  this other xmlns is like another folder
  <rect x="10" y="10" width="50" height="20" fill="blue"/>
</svg>
"""
# xml namespaces are used to avoid naming conflicts. namespaces are like having different folders in your xml file. But if you have them, you have to use them in your xpaths, with the ns:element syntax.
# but if you have namespaces, you have to use namespaces in your xpath. Give your wordy namespaces an alias; a mapper dict
# root= ET.fromstring(svg)                            # load xml string, into a ET object.
# namespace = {"svg": "http://www.w3.org/2000/svg"}   # dict alias mapper  
# # Referece the 'svg' alias in our xpath with the : syntax, everytime we reference an element. alias:element
# defs = root.find('.//svg:defs', namespace)
# print(defs)
# for i in defs: 
#     print(i.tag) # {xmlns}marker

# SADLY, ns clutter output, since XML relies on the association of ns with elements.
# SADLY, xml.etree.ElementTree has no support for removing the namespace. But you could use string slicing .split("}")[-1] to remove the namespace, OR, preferred, extend ElementTree with the lxml module. Then, use "ET.QName(elem.tag).localname" to print just the localname of the xmlns. 
# Libxml2 is a XML parser written in C. It has a python binding, lxml. 
# lxml builds right on xml.etree.ElementTree, usually lxml.etree is a drop-in replacement for ElementTree. Just use "from lxml import etree as ET" 
# lxml.QName().localname and .namespace can be used to split xmlsn from the name.
# lxml fully supports XPath, rather than just mostly supporting XPath.
# Use QName(element).localname and .namespace to separate the ns

# print("XML needs ns+tag to know what is what")
# for i in defs: 
#     print(i.text, i.tag)
# print()
# print("If you want to discard ns, use lxml.etree.QName().localname and .namespace")
# for i in defs: 
#     print(i.text, ET.QName(i.tag).localname)
    

# AttributeError: 'lxml.etree._Element' object has no attribute 'QName'

In [ ]:
import xml.etree.ElementTree as ET
country_data = """<?xml version="1.0"?>
<data>
    <country name="Liechtenstein">
        <rank>1</rank>
        <year>2008</year>
        <gdppc>141100</gdppc>
        <neighbor name="Austria" direction="E"/>
        <neighbor name="Switzerland" direction="W"/>
    </country>
    <country name="Singapore">
        <rank>4</rank>
        <year>2011</year>
        <gdppc>59900</gdppc>
        <neighbor name="Malaysia" direction="N"/>
    </country>
    <country name="Panama">
        <rank>68</rank>
        <year>2011</year>
        <gdppc>13600</gdppc>
        <neighbor name="Costa Rica" direction="W"/>
        <neighbor name="Colombia" direction="E"/>
    </country>
</data>"""

# root = ET.parse(_FileRead)

root = ET.fromstring(country_data)
root.findall(".")

#xpath '.' current directory
root.findall(".")
# All 'neighbor' grand-children of 'country' children of the top-level
# elements
root.findall("./country/neighbor")
# Nodes with name='Singapore' that have a 'year' child
root.findall(".//year/..[@name='Singapore']")
root.findall(".//year/..[@name='Singapore']")
# 'year' nodes that are children of nodes with name='Singapore'
root.findall(".//*[@name='Singapore']/year")
# Second 'neighbor' elements 
root.findall(".//neighbor[2]")

# ElementTree supports most xpaths. Must not be 'floating'. Must be tied back to the 'current node' with a  dot . to reference the rootmost path. 

# kinda the worst thing about python is that it wraps everything up in python bindings-- SQLite, XPaths, as two examples. BUT, you still have to master Sqlite or Xpaths to understand how to use the bindings-- I would almost just rather use the raw stuff a lot of the time. 





# Note: ElementTree can't fully utilize XPaths. It does most of it, except to use absolute paths, you just use .findall(./
# See https://docs.python.org/3/library/xml.etree.elementtree.html#supported-xpath-syntax
# [] and paths which don't start at root dnw. only / and // utility included. 

In [ ]:
# XPATHS : Navigating XML trees. Shorthand XML paths. 
# xml.etree.ElementTree module does much the same stuff

bookstore = """
<?xml version="1.0" encoding="UTF-8"?>

<bookstore>

<book>
  <title lang="en">Harry Potter</title>
  <price>29.99</price>
</book>

<book>
  <title lang="en">Learning XML</title>
  <price>39.95</price>
</book>

</bookstore>"""

# lxml .find(), .findall(), and .xpath() all use xpaths to search for subelements. 

# children/parent : implies one level down/up 
# descendant/ancestor: implies all levels down/up

# / fwdSlash: absolute path. Select Children(one level below)
# // dFwdslash: relative path. Select Descendants(all levels below)
# [] sqBr: slicing. 
# .     currentnode
# ..    parentnode
# node:
# element: 
# 
body            # select all nodes with name "body" 
//body          # select all 'body' elements 
body/p          # select all children 'p' elements of 'body' element
body//p         # select all descendants 'p' of 'body'
//@href         # select all "href" attributes 
//a[@href]      # select all 'a' elements with 'href' attribute
/html           # absolute path to 'html' root element

# predicates are embedded in sqBr[]
/bookstore/book[1]          # fist book element which is child of bookstore
/bookstore/book[last()]     # select last book element which is child of bookstore
/bookstore/book[position()<3] # select first&second book elements which are child of bookstore
//title[@lang]                  # select all title elements which have 'lang' attribute
//title[@lang='en']             # select all title elements which have'lang' attribute equal to 'en'
/bookstore/book[price>35.00]    # selects all book elements child of bookstore with price element > 35
/bookstore/book[price>35.00]/title  # selects all title elements of book elements child of bookstore with 'price' element > 35
/bookstore/book[]

# Xpath wildcards 
*               match any  element node 
@*              match any attribute node 
node()          match any element or attribute node 

/bookstrore/* # match any elements children of bookstore 
//*             # match all descendant elements , of root node, so all elements 
//title[@*]     # match all 'title' elements with one+ attributes 

# Many XPaths 
//book/title | //book/price # Select title elements of any 'book' elements AND 'price' element of any 'book' element
//title | //price       # select all title elements and all price elements 
/bookstore/book/title | //price # select all title elements of the book elements of the bookstore element AND all price elements 





In [ ]:
# Examples to learn: 
# colliding mice 
# dragdroprobot
# Then continue with rendering my capacitor svgs 


In [ ]:

        QPainterPath QGraphicsItem::shape() const 
        ReturnType   Class::fxName          # :: is called the 'scope resolution operator'. Its how you add methods to classes, its syntax is:  className::fxName 


In [ ]:

if 'string' == 'TruthyString' or 'string' # The or clause here will ALWAYS evaluate True, since non empty strings evaluate TRUE. 'a' or 'b' -> True. 
# Try instead: 
if 'string' in ('TruthyString' , 'truthystring')

In [ ]:
# Copyright (C) 2024 The Qt Company Ltd.
# SPDX-License-Identifier: LicenseRef-Qt-Commercial OR BSD-3-Clause

from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import os
import platform
from pathlib import Path

Base = declarative_base()


class Finance(Base):
    __tablename__ = 'finances'
    id = Column(Integer, primary_key=True)
    item_name = Column(String)
    category = Column(String)
    cost = Column(Float)
    date = Column(String)


# Check for an environment variable for the database path
env_db_path = os.getenv('FINANCE_MANAGER_DB_PATH')

if env_db_path:
    db_path = Path(env_db_path)
else:
    # Determine the application data directory based on the operating system using pathlib
    if platform.system() == 'Windows':
        app_data_location = Path(os.getenv('APPDATA')) / 'FinanceManager'
    elif platform.system() == 'Darwin':  # macOS
        app_data_location = Path.home() / 'Library' / 'Application Support' / 'FinanceManager'
    else:  # Linux and other Unix-like systems
        app_data_location = Path.home() / '.local' / 'share' / 'FinanceManager'

    db_path = app_data_location / 'finances.db'

DATABASE_URL = f'sqlite:///{db_path}'
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)

# Default data to be added to the database
default_data = [
    {"item_name": "Mobile Prepaid", "category": "Electronics", "cost": 20.00, "date": "15-02-2024"},
    {"item_name": "Groceries-Feb-Week1", "category": "Groceries", "cost": 60.75,
     "date": "16-01-2024"},
    {"item_name": "Bus Ticket", "category": "Transport", "cost": 5.50, "date": "17-01-2024"},
    {"item_name": "Book", "category": "Education", "cost": 25.00, "date": "18-01-2024"},
]


def initialize_database():
    if db_path.exists():
        print(f"Database '{db_path}' already exists.")
        return

    app_data_location.mkdir(parents=True, exist_ok=True)
    Base.metadata.create_all(engine)
    print(f"Database '{db_path}' created successfully.")
    session = Session()

    for data in default_data:
        finance = Finance(**data) # Equivalent to passing keyword arguemnts. Bc kwargs specify col:value pairs, don't have to provide values for nullable cols.
# finance = Finance(
#     item_name="Mobile Prepaid",
#     category="Electronics",
#     cost=20.00,
#     date="15-02-2024"
# )
        session.add(finance)

    session.commit()key
    print("Default data has been added to the database.")

# To dynamically create a sqlAl class for my unknown part_attributes data: Must begin with a __tablename__. Then use __setattr__(). 
tablename = 'ayo'
class NewPart(Base):
    __tablename__ =  tablename
    id = Column(Integer, primary_key = True)

for attr in part.attributes: 
    NewPart.__setattr__(self, attr, Column(String))# Voltage-Rated = Column(String) 
    
newpart = NewPart(**part.attributes) # NewPart(Voltage-Rated='16V', ...) 
session.add(newpart) # INSERT INTO __tablename__ ('Voltage-Rated', ...) VALUES ('16V', ...)


In [ ]:
# Open a browser in python using  builtin module 'webbrowser' 
import webbrowser
url = "https://www.w3schools.com/"
webbrowser.open(url)
webbrowser.open_new_tab(url)
webbrowser.open_new(url) #new window

# Open a browser with PySide6 using QUrl and QDesktopServices: 
from PySide6.QtGui import QDesktopServices 
from PySide6.QtCore import QUrl
QDesktopServices.openUrl(QUrl(url))


In [ ]:
PySide6 Python QT 


vocab: 
rubber band : That selection tool when you like right click and drag a box over a bunch of items and select all the items inside the box at once.

# .metaObject() 


In [ ]:
keys= 's','a' # Py lets you shorthand initialize a tuple w/o the ()...
keys = 0,1,2
print(type(keys))
print(keys)

In [ ]:
1️⃣ How Does Qt Know My Default Browser?
Qt uses native OS functions to query the system registry (on Windows), read user settings (on Linux/macOS), and determine which application is assigned to handle HTTP/HTTPS URLs.

🔹 Windows
Qt calls the Windows ShellExecute API (ShellExecuteW or ShellExecuteEx) to open URLs.
This function checks file associations in the Windows Registry, under:
arduino
Copy
Edit
HKEY_CLASSES_ROOT\http\shell\open\command
The default browser (e.g., Chrome, Edge, Firefox) is registered here.
When you run:
python
Copy
Edit
QDesktopServices.openUrl(QUrl("https://www.python.org"))
Qt essentially does:
cpp
Copy
Edit
ShellExecuteW(NULL, L"open", L"https://www.python.org", NULL, NULL, SW_SHOWNORMAL);
This executes the system-assigned browser with the given URL.
🔹 macOS
Uses Launch Services API (LSOpenURLsWithRole).
This API consults ~/Library/Preferences/com.apple.LaunchServices.plist, which stores user-defined default apps.
Equivalent system command:
sh
Copy
Edit
open "https://www.python.org"
This runs the default web browser.
🔹 Linux (X11/Wayland)
Uses xdg-open, which checks:
~/.config/mimeapps.list
/usr/share/applications/mimeinfo.cache
/etc/xdg/mimeapps.list
Runs the equivalent of:
sh
Copy
Edit
xdg-open "https://www.python.org"
This launches the system’s default web browser.
2️⃣ How Does Qt Get Permission to Open the Browser?
Qt doesn’t need elevated permissions to open a browser. Here’s why:

Qt is not accessing system files or modifying settings, just invoking a standard OS feature.
The browser itself is responsible for handling security (sandboxing, user permissions, etc.).
OSes treat launching a browser the same as opening a file—no admin rights required.
🔹 When Would Permissions Be Needed?
If your app is in a restricted environment (e.g., kiosk mode, sandboxed apps).
If opening a browser is explicitly blocked by system policies.
If your app is running as a different user (e.g., root trying to open a user-space browser).
3️⃣ How Qt Abstracts This in QDesktopServices.openUrl()
Qt checks the OS (Windows, macOS, Linux).
It calls the appropriate native API (ShellExecute, xdg-open, LSOpenURLsWithRole).
The OS then forwards the request to the default handler for HTTP/HTTPS.
That’s why it "just works" across platforms. 🚀

Want to dig deeper into how QDesktopServices.openUrl() works internally in Qt source code? 

In [ ]:
QTableWidget : Your standard table. Made up of QTableWidgetItems. Default model. 
tableWidget = QTableWidget(row_num, col_num, par
QTableView: 


In [ ]:
Foreign key cheatsheet crash course 
many to many: requires its own table
one to many:  foreign key into the *many* side of the relationship
one to one: foreign key into either table  

CONTAINSTABLE is used in FROM clauses and acts as if it were a normal table name 



recursive relationships 

when song cover of another song 

ER model Entity relationship 
EER model Extended entity relationship

"disjoint subsets total participation" --no null values 
unions are an alternative to disjoint subsets 
artist = U(band, musician)
# 
# https://www.youtube.com/watch?v=S3PeFXuzpAE

In [ ]:

# https://www.youtube.com/watch?v=o0DgKSA0YeE

#"scehmas" or "schemata" are schema pl.
# Representation of a 'table' schema: 
TableName(Attr1, Attr2, ... )

# Schemata Below: 
Artist(ArtistID)
Band(ArtistID, BandName)
Member(MusicianArtistID, BandArtistID)
Musician(ArtistID, BirthDate, FirstName, LastName)
Album(AlbumID, AlbumName)
Contains(AlbumID, SongID, TrackNumber)
Song(SongID, SongName, Duration, ArtistID, CoveredSongID)
Features(SongID, ArtistID)
# You'll also see graphical drawings of schemata, with arrows to indicate FK/PK relationships:
entity names in rectangles      Song
attributes in circles           Title, Album, artist
primary keys underlined         SongID 

referential integrity: FKs are subject to referential integrity: FK value must be NULL, or equal to a value in the PK


In [ ]:
CREATE TABLE Orders (
    OrderID INTEGER NOT NULL,
    OrderNumber INTEGER NOT NULL,
    PersonID INTEGER,
    PRIMARY KEY (OrderID),
    FOREIGN KEY (PersonID) REFERENCES Persons(PersonID)
);


In [ ]:
CREATE TABLE Persons (
    PersonID INTEGER NOT NULL,
    LastName VARCHAR NOT NULL,
    FirstName VARCHAR NOT NULL, 
    Age INTEGER,
    City VARCHAR DEFAULT 'NYC' # set default value for 'City'
    CurrentDate date DEFAULT GETDATE() # Use DEFAULT constraint to insert system values
    CHECK (Age>=18)
    
);
    # CONSTRAINT CHK_Person CHECK (Age>=18 AND City = 'Sadnes') Must use CONSTRAINT syntax for composite CHECKs and/or to name the CHECK

In [ ]:
class myClass():
    a = 1 
    b = 2 
    def __init__(self, name): 
        self.name = name 
        
    @classmethod
    def nickname(cls, nickname, arg2 ):
        bs = nickname +str(arg2)
        return cls(bs)
# classmethod
    
a = myClass('Jill')
b = myClass.nickname('Jill', 'yolo')

print(a.name)
print(b.name)

In [ ]:
# Defaultdict: useful for counter variables 
from collections import defaultdict 
# Normal dict throws error on attempt to .get() keys which dne. defaultdict adds a key:defaultValue pair, on attempt to .get() keys which dne. You must specify a type for the default value. int->0 str->'' list->[], etc. defaultdicts handy for tracking counter variables. 

    # Initialize defaultdict with int, so it returns default 0. If a key which dne is accessed, a k:v pair will be added w/ default value, instead of throwing an keyError, like a dict would.
    counters= defaultdict(int)
    L = [1,2,3,2]

    for i in L: 
        counters[i] += 1 
        
    print(counters)
    
    counters = defaultdict(int) 



In [ ]:
# Code is voltages at memory addresses. 1s and 0s baby! Transistors blah blah 
# Everything is an object
# class is an object 
# strings are an object, functions, everything 
# classes are bundled objects. Often we like to associate many objects together; classes are a bunch of associated objects  
# classes can make 'instance objects'; objects that represent one member of a class  (?)

class C(): 
    ca = 'a' 
    ca2 = 'b'
    
    def __init__(self, ia = 1, ia2 = 2):
        self.ia =1 
        self.ia2 = 2 
        self.ia3 = ia+ia2
        
    def add(self): # bind function object 'add' to instance object 'self': resulting in a ... method! (a function, bound to an object) 
        return sum([self.ia, self.ia2])
    
    @classmethod
    def configure_ca(cls, value): 
        # ca = value BAD 'ca' only exists in scope of C.ca
        C.ca = value
        
    @classmethod
    def from_csv(cls, csv): 
        return cls(ia=  int(csv[0]))
    
ca = 'a' 



In [ ]:
### ###

class OAuth(): # .authenticate() -> token
    secret_key  = ... # TODO: keep sensitive info secure while using in python? Is config in class OK? 
    private_key= ...

    def __init__(self):
        self.authenticate()
       

    def authenticate(self):
        OAuth.secret_key #access class variables 
        self.token = token
        return token

# New user here. Needs to store his own private/secret keys: use a @classmethod. Alternative: make private/secret keys instance attributes
    @classmethod
    def SetSecretKey(cls, secretKey):
        cls.secretKey = secretKey
        #instance objects have the @property decorator for attribute getters/setters. Class getter/setters I guess use @classmethos and cls?  
    def GetSecretKey(cls):
        return cls.secretKey
# usage: OAuth.SetSecretKey('mykey')

    
        
# Where shoudl I keep OAuth? If external , how to use OAuth within class API (simply token = OAuth().token)

        
class DigikeyAPI(): 
    endpoints = ['product_details', 'media']

    def __init__(self):
        pass        

    def fetch(self, mpn: str = 'randoMpn'): 
        self._token = OAuth().authenticate  # run oauth BTS
        for ep in self.endpoints:
            requests.request(ep)... # make request of dk api endpoint(s)
        return self.response# requests.request object
    # dk = DigikeyApi()
    # dk.fetch(mpn, ) 
    

dk          = DigikeyApi() 
response    = dk.fetch(mpn)

part = Part().parse(response)








class Part():
    ...
    def __init__(self): 
        pass
    
    def parse(self, response):
        #parse response into json part object 

    def store(self, database):
        ...
    @classmethod
    def load_csv(cls, category_folder, mpn):
        #load 'part' json object from relational csv(s) 
        return cls.parse()




        Schema: 
    folder: 
Diodes/RF Diodes 
    tables: 
RF Diodes_attribute_groups.csv      general_attributes,part_attributes,eda_sw_specific_attributes,pricing_attributes ,meta_attributes, media_attributes(Different columns for every part ) 
RF Diodes.csv                       All data fields, in one big row. (media columns differ between parts, so IF they are stored here-- I must have ALTER TABLE capability, to add columns-- wait NO bc new parts would FAIL to insert(w/o further processing), bc their schema would be differnt. Oh, could pass a (cols) tuple : INSERT INTO "RF DIODES" (col1, ... , coln) VALUES (val1, ... valn) 
# to put all in one table: check if insert has columns which DNE in table.tables
# If yes, ALTER TABLE, to add columns 
# Requuires I always INSERT with named columns. Which is fine. 

RF Diodes_media (?)                 media data, to be relational, needs its own table, bc its schema always changes . 
RF Diodes_other                     IF there's any new data which I haven't anticipated, and thus doesn't fit into schema, just shove it here. Probably won't ever need it

 # I want 'DS', '3d', & others from 'media' endpoint to be included in 'general_attributes'
# store 'media', with always diff columns, in json or other. Let user peruse this media, if primary ds isn't working.


# want my 'part' object so I can render, interact with it. 
# or do I want a 'part' object NOT 100% loaded... so I only fetch part attributes relevant to NOW? 
# Would start with 'part' 100% loaded... at a kB or so, 1k parts not significant
class RenderArea(QWidget): 
    _part = Part(
# 
# part.graphic = 'blah.svg'
# part. 
folder Diodes/RFDiodes has tables table1, table2, tablen
for elligible_files in folder: 
    
SELECT * FROM table1, table2, ..., tablen ( SQL needs to see it all typed out ) 
f"SELECT * FROM {RFDiodes.tables}"
# PROBLEM: This don't do what you think it will: you want "UNION ALL" clause-- as is, this query does a permutation of the rows, returning every possible row, resulting in a massive number of returned rows. This is called a 'row'cartesian product' or 'cross join' 
# Do instead: # SELECT * FROM table1 UNION ALL SELECT * FROM table2 UNION ALL SELECT * FROM tablen;
tables = ["table1", "table2", "tablen"]
query = " UNION ALL ".join([f"SELECT * FROM {table}" for table in tables])
print(query)

No duplication of rows (unless duplicates exist in the original tables.) (unless UNION is used instead of UNION ALL).
Requires tables to have the same structure.
tables must have same column schema( same col order and number)


In [ ]:
class A:
    pass

B = type("B", (), {})  # Creates a new class dynamically

print(A is B)  # False → A and B are different class objects
# instance objects vs class objects. 
# instance objects: easy to make more 
# class objects: not as 'easy' to make more ( like)


In [ ]:
# Cartesioan product/ cross join in sql: the result of "SELECT * FROM table1, table2: "
table1:
id	name
1	Alice
2	Bob

table2:
product
Apples
Bananas

Result of Cartesian Product:

id	name	product
1	Alice	Apples
1	Alice	Bananas
2	Bob	Apples
2	Bob	Bananas

In [ ]:
Singleton: a class may create many instance objects. A singleton class may create only one instance object. This limitation can be useful, depending. Say 

In [ ]:

class MyClass():
    pass

MyClass() 
# We do not strictly NEED a constructor to make a class instance. But the constructor is often used to 'set up' the class. Note that the __new__ method is what makes the instance in memory, and the constructor, __init__, works on the instance __new__ made.



In [ ]:
class MyClass():
    def __init__(self, a):
        self.a = a 
        
    @classmethod
    def from_str(cls, s):
        return cls(s)
    
MyClass.from_str('hey')
    
# Using @classmethod , I can make 'alternative constructors'... But this isn't quite the same as overloads; for one, each altCon has to have different name



In [ ]:
import typing

class MyClass:
    def __init__(self, a):
        self.a = a 
        
    @typing.overload

In [ ]:

import typing 

    def __init__(self, a):
        self.a = a
        
    @typing.overload
    def addItem(s)
#Overload, method overloads, function overloading 

# from QtWidgets.pyi
import typing
@typing.overload
def addItem(self, text: str, userData: typing.Any = ...) -> None: ...
@typing.overload
def addItem(self, icon: PySide6.QtGui.QIcon | PySide6.QtGui.QPixmap, text: str, userData: typing.Any = ...) -> None: ...

# Use python's typing.overload decorator to define function calls with different parameters.



In [ ]:
class ClassTemplate():
    class_attribute = 1             # Shared by all instances
    s = "Class attribute objects belong to the class. ClassTemplate.class_attribute->1"
    radius = 4

    def __init__(self, constructor_arg): # Constructor -> instance objects. 
        self.instance_variable = constructor_arg # instance variable, belongs to instance object; tied to instance object through use of self. Unique to each instance
        local_variable = 'local_variable, often not useful, only exists inside init'

    def local_function_in_class_lacks_self():
        print("local function (bc lacks that first arg, 'self'). Bc a 'method' is a function which 'belongs' to an object, this isn't a method, bc it does not 'belong' to an instance object, bc it lacks a first arguement, which is called 'self' by convention, because py always passes the instance object to the first arg of functions inside classes. You will inevitably make this type of non-method function a lot, as you'll forget to make 'self' the first arguement sometimes, and you will get an err, as python passes the instance object as the first parameter, despite the function specifying 0 arguments: "takes 0 positional arguments but 1 was given" But this type of function often not so useful. Its not accessible outside of class object, yet doesn't 'belong' to an instance object. Its not a @classmethod nor @staticmethod nor instance method." Q: Does anyone use this type of func anyway? ) # Attempted call by instance: myIns.local_function_in_class_lacks_self() 
        # Note static methods do not take self as arg in their definition, but you can call it with self.staticMethod() it just won't be passed self as first arg
        
    # instance methods must have self as a first argument
    # A method is a function which 'belongs' to an object. instance methods 'belong' to instance objects
    # You use instance methods via dot syntax "instance_object.instance_method()"
    def instance_method(self):
        print("For all function objects inside a class, python passes the instance object as the first argument. By convention, everyone refers to the instance object as 'self'. A 'method' is a function which 'belongs' to an object. Here, our function becomes a method when its made to belong to our instance object.")
        
    @classmethod # Use the @classmethod decorator to make a class method. Alternative constructors, __init__() being the 'main' constructor, @classmethods work on the class object, 'cls'. Some uses include: 
    # alter class attributes
    # create instance objects with obj = cls.__new__(cls)
    # create instance objects with return cls(arg1)
    # and more 
    def classMethod(cls, arg1, arg2): # classmethod does the constructor, with modifications. May create instance object, may modify class attributes 
        arg1 = arg1+arg2
        return cls(arg1) #-> an instance object made like C.classMethod(1,2). Compare to instance objects made by constructor like C(1)
    # Q: why are @classmethods called such (Bc BTS, py passes the class object to the first argument.)
    # Q2: what does a class method do (Trick, they do whatever coded to do. Often, @classmethods modify class attributes, often @classmethods return instance objects, doing what init does, just different.)
    
    # CONFUZE ALERT: 
    @classmethod # ClassTemplate.classMethod2(10) sets class_attribute
    def classMethod2(cls, arg1):
        cls.class_attribute = arg1  
        # Note -> None. ClassTemplate.classMethod2(10) -> None but will change class's class_attribute
        
    @classmethod
    def from_string(cls, str):
        obj = cls.__new__(cls) # create an instance 
        obj.lines = str.splitlines()
        
    @property #  @property makes a getter function for a instance attribute. @property may as well be @getter. @property lets you call methods like they’re attributes. Useful for calculated values.
    def area(self):
        return 3.14*self.radius**2
    #usage: myclass = MyClass(4) myclass.area #-> 16 NO parentheses 
     
    @readOnly.setter 
    def readOnly(self, value): #Not strictly needed. @name.setter is invoked when you assign obj.readOnly, as in obj.readOnly = 'readOnly'. The setter can do things like validate the entry is proper before updating the private attribute. Alternative : may also set instance attribute like:  ins_obj.attribute = 'hey'
        self.__readOnly = value
        

    
    @staticmethod
    def staticMethod(L):
        print("staticmethods are methods with some relevance to the class yet don't take class object or instance object")
        return sum(L)
    
F = ClassTemplate
C = ClassTemplate
# F and C both refer to the SAME OBJECT
# print(F, C) <class '__main__.ClassTemplate'>
# print(F is C) -> True
# print(id(F), id(C)) Same memAddr

print(C)                    # A class object # <class '__main__.ClassTemplate'> 
D = C.classMethod(2,2)      # An instance object # <__main__.ClassTemplate object at 0x000001374FEFA780> # an instance object, just as a constructor makes.
print(D.class_attribute)    # -> 1, Did not modify class_attribute 
print(D.instance_variable)
print(C.classMethod2(10))   # sets a class_attribute =10
print(C.class_attribute)    # ->10
# C.classMethod2(10) -> None so = assignment bad.

C.classMethod2(20)          # sets class_attribute =20
print(C.class_attribute) # -> 20
print(D.class_attribute) # -> 20. class_attribute changed for all instance objects 
print(F.class_attribute) # -> 20. There is only one class object, which C and F both refer to





# D is a new instance object. class_attribute is a class-level attribute.

# Receives the class as first arg automatically,, just as an instance method auto receives the instance.
#   class C:
#       @classmethod def f(cls, arg1, arg2, argN):
# Call with C.f() or C().f(). The instance is ignored, except for its class.
# In C++/JS, you can 'overload' functions. an overloaded function is a function with the same name, but different parameters. 
# Python doesn't have overloaded functions.But there are ways to achieve much the same effect. 

# In python, if we try to write two functions with the same name, but different parameters, the second function overwrites the first. 
def func(a,b):
    return a+B
def func(a,b,c): # func() has same name but has different parameters from first func. In python, the older function gets overwritten
    return a+b+c 

# In Python, there are a bunch of ways we can mimic parts of the functionality of an overload.
# positonal arguments (x, y, z) , default arguments ( x=0,y=0,z=0), *args, **kwargs, @classmethod decorator, typing.overload, are all some ways that we can mimic the functionality of overloads.
# C++/JS 'overloading' is analogous to python @classmethod, since both create differently 'flavored' constructors.
# C++ or Java static methods are analogous to python's builtin staticmethod 

In [ ]:
# I am confused about the purpose of @staticmethod, because you can achieve the same, with an instance method, that merely does not make use of self... (Hmm This observation is correct; here staticmethod accomplishes same as instance method. Yet while instance method CHOSE to not make use of the self argument, its WAS STILL passed self as a first parameter, while the staticmethod was not passed self as a first parameter. The utility of this, is not demonstrated here, is there a situation where an instance method can't cut it and we HAVE to use a staticmethod? )  See @staticmethod funcA and instancemethod funcB:
class ExampleClass():
    def __init__(self):
        pass
    
    @staticmethod
    def funcA(): # static method is not passed self as first parameter; thus staticmethod does not have self as first parameter
        print('Hey')
#ExampleClass.funcA() -> 'Hey'
#ExampleClass().funcA() -> 'Hey'

    def funcB(self):
        print('Hey')
#ExampleClass().funcB() -> 'Hey'
#ExampleClass.funcB() THROWS ERROR

e = ExampleClass() # e is an instance object; ExampleClass() is an instance object. ExampleClass is a class object.
ExampleClass().funcA()
ExampleClass().funcB()
e.funcA()
e.funcB()
ExampleClass.funcA()
# ExampleClass.funcB() TypeError: ExampleClass.funcB() missing 1 required positional argument: 'self'


# @staticmethods are bound to the class object; cls. Instance methods are bound the instance object; self. 
# Thus staticmethod can be called by ExampleClass; the class , or ExampleClass(); the instance. 
# But instancemethod has to be called with ExampleClass().funcB(); the instance,  not ExampleClass.funcB; the class. 






In [ ]:
#Python Raise error throw exception

# Types of errors you can raise: 
# Exception: 
# TypeError: 
#Thats pretty much all you need 

x = -1
if x < 0: 
    raise Exception("No numbers below 0")


x = "string"
if not type(x) is int:
    raise TypeError("x should be int but got:", type(x), x)


In [ ]:
#Python try except blocks try except else finally 

# try: test for errors 
# except: Handle exception 
# else: execute code when no error 
# finally: execute code regardless 
# Try: except blocks can intercept errors, stopping them from stopping your program. 
# f = open(file) 
# f.write('asdf')

try: 
    file = 'file.txt'
    msg = "asdf"
    f = open(file)
    try: # Try to do this 
        f.write(msg)
    # except: # If exception, do this
    #     print("Could not write to file", file)
    except Exception as e: # Access the thrown error so we can see what it was
        print("An error occurred:", e)
        
    else: 
        print(f"Wrote {msg} to file" ,file)
    finally: # Do this regardless
        f.close()
except:
    print("Could not open file" , file)
    


In [ ]:
# CONFUZE ALERT 
# To avoid confuze, I always refer to instances as 'instances', or 'instance objects'
# class 'classTemplate' is a class. An object, too(As everything is python is an object) a class...object...
# 'instance_object' is an instance of classTemplate. An object, too. An object of class. a class...object... 
# THE CONFUZE POTIONTIAL IS TOO DAMN HIGH : the term 'class object' is ambiguous, don't use it. Is 'class object' the class(which is an object) or an instance( which is an object of a class)
# To avoid confuze, I always refer to instances as 'instances', or 'instance objects'


In [ ]:
# simple class 
class classTemplate():
    class_attribute = 1

# some stuff you can do: 
print(classTemplate)
print(classTemplate.__class__) #__class__ : doubUScore name mangling. dot __class__ returns the object's class. DWAI but classTemplate is a class, and the class of classTemplate,  is type--as type is the default for all pythonclasses, which is why type() works on all objects; all objects can be type()'d. 
print(classTemplate.__name__) #__name__: gets name of object, 'classTemplate'

# May add class attributes with dot notation obj.attr
classTemplate.class_attribute_2 = 2
print(classTemplate.class_attribute_2)

# May create instance objects: use () after class
instance_object = classTemplate()
print(classTemplate)    # a class   <class '__main__.classTemplate'>
print(instance_object)  # a instance object  <__main__.classTemplate object at 0x0000019229124260>

# May add instance attributes with dot notation ins_obj.attr
instance_object.instance_attribute = 'a'
print(instance_object.instance_attribute)
# Class attributes are always there for instance objects to access 
print(instance_object.class_attribute)
print(instance_object.class_attribute_2)
# change class_attribute to 100. All    
instance_object.class_attribute = 100
print(instance_object.class_attribute)


C = classTemplate   # a class 
instance_object = classTemplate() # an instance of class

# Recap: 

# Make class
class classTemplate(): 
    class_attribute = 1 
#class done

#assign more class attributes to class: 
classTemplate.class_attribute_2 = 2 
#make instance object
instance_object = classTemplate() 
#assign an attribute to instance object 
instance_object.instance_attribute = 'a'

# Recap Done

In [ ]:
class A: 
    def __init__(self, a): 
        with open(a) as fo: 
            self.lines = fo.readlines()

    @classmethod
    def from_string(cls, str):
        # self.lines = str.split() # NO BAD 'self' DNE in @classmethod, only cls
        # cls(str) # NO BAD 
        obj = object.__new__(cls) # In python, there is a buitin class, 'object', from which all classes inherit. 
        obj = super().__new__(cls) # Since no parent was specified, class A will inherit directly from object; it is equivalent to call object.__new__(cls)
        obj = cls.__new__(cls) # Does this work?  I think this would recurse 4eva.
        obj.lines = str.split()   
        
a = A.from_string('its a hard')
print(a.lines)

In [ ]:

#concept: getter, setter, deleter functions. @property(auto implements getter) 
# set an instance object attribute:
instance_object.newAttribute = 10 
# get an instance object attribute: 
x = instance_object.newAttribute

# Python has special getter, setter, deleter support, for instance objects. @property, @property.setter and @property.deleter decorators define getter, setter, and deleters. 
# @property decorator makes value immutable(read-only) except through the setter/deleter
# Syntax is odd. Each decorator must be followed by a function define. The name of that function define must be used in the setter/deleter decorators.Plus, must name mangle getter's property-- else, you're trying to make self.name both a gettter function, as well as a variable, but python picks the last one so it thinkgs its a function, and that function calls itself forever; infinte recursion/stackoverflow on attempts to 'return "getter_function"' 
class GSDeleterPractice():
    _name = 'asdf' # _name because we want "name" to be the name of the getter-- "name" will return '_name'
    @property 
    def name(self):
        return self._name # DNW; RecursionError: maximum recursion depth exceeded; self.name is a getter function, which returns the gettern function.... which gets the getter_function... forever.  return self.__name instead, name mangled, or return self._name, not name mangled, yet not the same name as the getter function.
    
x= GSDeleterPractice()
print(x.name)


In [ ]:
#Some useful name mangled builtins: 
# These are always used in the dot context; object.__class__
__main__ __name__ __class__ __self__ 
# __main__ is the name of the module where your Python script is being executed. When you run a script directly, Python assigns "__main__" to the special built-in variable __name__. If you import a script as a module in another Python script, its __name__ will be the filename instead of __main__.
# __name__ is the name of the object 
#  __class__ is the class of the object 
# __self__ is the 

In [ ]:
class Foo: 
    a= 10
    b= 20 
    def __init__(self, c):
        self.c = c
    def instanceMethod(self):
        self.c +=10

    @classmethod 
    def twiceAsBig(cls, arg1):
        arg1 = arg1*2
        return cls(arg1)
    @classmethod 
    def setA(cls, value):
        cls.a = value # we can set class attributes outside the class, this isn't that special
        
    @staticmethod
    def openFile(file):
        print(f'file is {file}')

    @property # properties are  accessed with dot notation: object.property Foo.bar
    def name(self):
        return self.__name 
    
    @name.setter # go though setter when we say Foo.name = 'Rico'. That means "Rico" will go through the setter 
    def name(self, value):
        self.__name = value
        
    @name.deleter
    def name(self):
        del self.__name
            
    @property
    def name(self):
        return self.__name
    @name.setter
    def name(self, value):
        if value>100:
            raise Exception("The name is too large")
        else: 
            self.__name = value


F= Foo
print(Foo.a)
print(Foo.b)
# Foo.name = 200 # oops, we overwrote Foo.name @setter with '200'... @property is meant as a setter for instance objects, not class objects( see how it takes 'self')


f = Foo(30)
print(f.b)
#class and instance attributes may share the same name Foo.b and Foo(30).b
f.name = 20
# f.name = 200 Bc setter scouting for name > 100, attempt name =200 err.

f = Foo(1)
print(f.c)
f.instanceMethod()
print(f.c)
g = f.twiceAsBig(10)
print(g.a)
g.setA(33)
print(g.a)
f.openFile('ayo')

f.name =10 
print(f.name)
f.name =30
f.name

In [ ]:
# def f(x, y=0, *z, **a) 
# x is a positional arg. x has to go first
# y is a kwarg. y is supplied as a keyvaluepair
# z is any number of args 
# a is any number of kwargs 

# an attribute is a variable which 'belongs' to an object
# a method is a function which 'belongs' to an object
# Often, attribute:variable and function:method can be synonyms.

In [ ]:
class MyClass:
    x = 1 # class variable x  : accessible by all instances of the class (self.x) (self.x does MyClass.x)
    
    def __init__(self): 
        self.y = 2 #Instance variable 
        print('x: ', self.x) #
        
    def func(self): #Instance method
        print(self.y)
        print(self.x)
    

print(MyClass().x)



In [ ]:
class Foo(): 
    x = 1 
    
    # f is a function defined inside the class scope without self-- since f() does not take self, it is not an instance method.
    def f():# By definition, all attributes of a class that are function objects define corresponding methods of its instances.(Oh-- but you must define a first argument, self, bc python will pass the instance object as the first argument. )
    # Similarly, all attributes of a class that are variable objects define corresponsing variables of its instances... self.x ->1
         # f() not f(self), instance object not passed. This is a local function. Not a class function, instance function, or static function. Scoped as Foo.f()
        y = 1
        print(y)
        return y
# class functions often should have (self) rather than (). 
    
    def g(self): 
        y = 2 
        print(y)
        return y
    
    def __init__(self):
        z=3
        print(z)
        # print(x) access class variable x by passing it as a parameter. (A class is never used as a global scope.)
        
foo = Foo()
print(foo.f)     # foo.f is NOT bound to instance object foo. foo.f() won't have access to self, because f() isn't an instance method. 
print(foo.g()) 
Foo.f()

In [ ]:
class Yoshi():
    egg = 'speckled'
    
    def __init__(self):
        pass
    
y = Yoshi()
print(y.egg) # Tricky: 'y' used only for its class here. 'egg' exists at the class level

y.egg = 'throwable' # Tricky: instance object just created 'egg' as an instance variable. 'egg' is also a class variable, so its kinda 'covered up' by 'egg' the instance variable 
print(y.egg) # 
print(Yoshi.egg) # the class attribute did NOT change but the instance attribute DID change
print()
Yoshi.egg = 'hatchable'
print(Yoshi.egg) # the class attribute did change
print(y.egg)     # bc 'egg' both class and instance variable, instance 'egg' obscured class 'egg' 

print()
z = Yoshi() # expect class variable 'egg' to be 'hatchable'.
print(z.egg) 

class Yoshi():
    egg = 'speckled'
    
    
    def __init__(self, tongue = 'blewmp'):
        self.tongue = tongue
        
y = Yoshi()
y.tongue


In [ ]:
def scope_test():
    def do_local():
        spam = "local spam" 
        
    def do_nonlocal():
        nonlocal spam
        spam = "nonlocal spam"
    
    def do_global():
        global spam 
        spam = "global spam" 
    
    spam = "test spam" 
    do_local()
    print("After local: ", spam)
    do_nonlocal()
    print("After nonlocal: ", spam)
    do_global()
    print("After global: ", spam)

scope_test()
print("In global scope: ", spam)

In [ ]:
The only operations understood by instance objects are attribute references. There are two kinds of valid attribute names: data attributes and methods.

Valid method names of an instance object depend on its class. By definition, all attributes of a class that are function objects define corresponding methods of its instances. Since MyClass.f is a function, x.f is a valid method reference. But x.f is not the same thing as MyClass.f — it is a method object, not a function object. And, if you do not pass a first parameter 'self', then the function is a local function within MyClass, 

Instance objects only understand references to their attributes and methods 
All class methods are instance method. MyClass.func makes x.func a valid method reference. But MyClass.f !=x.f -- x.f is a method object, not a function object. 

A method is a function that “belongs to” an object. Usually, a method is called () right after it is bound: x.f().

x.f is a method object. It may be called at a later time: 
xf = x.f
while True: 
    print(xf())
    
methods, being a function which 'belongs' to an object, pass the instance object as the first argument of the function.

x.f() equals MyClass.f(x)

In general, methods work as follows. When a non-data attribute of an instance is referenced, the instance’s class is searched. If the name denotes a valid class attribute that is a function object, references to both the instance object and the function object are packed into a method object. When the method object is called with an argument list, a new argument list is constructed from the instance object and the argument list, and the function object is called with this new argument list.


class Dog: 
    tricks = [] # mistaken use of a class variable. mutable objects such as lists can have possibly surprising effects  when shared: d.add_trick('roll over') also adds 'roll over' to instance e. Correct class design should use an instance variable self.tricks =[]
    kind='canine' # class variable shared by all instances 
    def __init__(self, name): 
        self.name = name #instance variable unique to each instance 
d=  Dog('Fido')
e = Dog('Buddy')
d.kind
e.kind
d.name
e.name




In [ ]:
If the same attribute name occurs in both an instance and in a class, then attribute lookup prioritizes the instance:


class Warehouse: 
    purpose = 'storage' 
    region = 'west' 
    
w1= Warehouse()
print(w1.purpose, w1.region)

w2 = Warehouse()
w2.region = 'east' 
print(w2.purpose, w2.region)

In [ ]:
class Foo: 
    x = 1 
    def classmethod():
        y=2
        return y
    def __init__(self)

In [ ]:
# Any function object that is a class attribute defines a method for instances of that class.
# Class methods are instance methods. Foo().func() and Foo.func()
def f1(self, x, y): 
    return min(x, x+y)

class C: 
    f = f1
    def g(self):
        return 'hello world'
    
    h=g 
# f, g, h  class attributes that refer to function object. All methods of instances of C

#Note that this practice usually only serves to confuse the reader of a program.

In [ ]:
# Methods may call other methods, by using method attributes of the self argument
class Bag: 
    def __init__(self):
        self.data= []
        
    def add(self, x):
        self.data.append(x)
        
    def addtwice(self, x):
        self.add(x)
        self.add(x)
        
Methods may reference global names, as all functions do. The global scope of a method is its module. (A class is never used as a global scope.) While its bad practice to use global data in a method, there are legit uses of the global scope: functions and modules imported into the global scope can be used by methods

Each value is an object, and therefore has a class(also called its type) stored in object.__class__ 



In [ ]:
# Code uses memory (RAM) 
# memory has addresses. 
# code objects live in memory; at addresses.
# Anything in code is an object: variables x=10 functions def func():, dictionary D= {'a':1} a tuple, a list, a lambda function, a class, everything is an object, everything lives in memory at an address, at the lowest level, your variables are all stored in memory addresses.
# C++ manipulates code objects at their addresses. Python is written in C++, so python does this behind the scenes for you 
# Typing "func" is way different than typing "func()". func() calls the function; executes the function.  func is the function; its a function object; where func lives in memory. 
# class MyClass(): ... 
# Typing "MyClass()" calls the class; returns a class instance, class instance is another object, at its own address in memory. Typing "MyClass" returns the class; will return where the class lives in memory.




In [ ]:
# Python Classes : 

In [ ]:
#initialize variables inside tuple! 
(x,y,z,a) = range(4)
print(x,y,z,a)

In [ ]:
# floor divide, modulo remainder, divmod
divide = 11/10
floor_divide = 11//10 
modulo_divide = 11%10 
div_mod = divmod(1, 10)

print(divide, floor_divide, modulo_divide, div_mod)

In [ ]:
... ellipses is a builtin python object. 
Indicates "finish this code later" and has legit uses in enums, and slicing

in enums, it can represent integer values which will be replaced at runtime, or left unused if they were'nt called for. Comments should hint at what their values will be. From QTCore.Qt: 

    class WhiteSpaceMode(enum.Enum):

        WhiteSpaceModeUndefined   = ...  # -1
        WhiteSpaceNormal          = ...  # 0x0
        WhiteSpacePre             = ...  # 0x1
        WhiteSpaceNoWrap          = ...  # 0x2
        
in list slicing, (often numPy arrays with multiple dimensions): 
arr[0, ..., -1]  # Equivalent to arr[0, :, :, -1]

#Ussage of ... seems more confusing to me than explicit defines. I will blindly believe there is a higher reason for using ... 

In [ ]:
# args, keyword args, *args, **kwargs 
#keyword args need to go After positional args.
L = [1,2]
def addUp(L):
    print(sum(L))
addUp(L)

def addUp(*L):
    print(sum(L))
addUp(1,2)
addUp(*L)


In [ ]:
#bitwise operators & | << >> ~ ^ 
# complement ~ switch values of all 0s/1s. Equivalent to -x -1
# Exclusiveor ^ XOR 

# Note except for sets, the sets have specific meanings for | ^ & 


In [ ]:
# String formatting 
# f-strings are the bomb .com
x = 10
s = f'put variable here {x}'

# formatting codes : for when you want to mutate a string to another form. Another form like with leading zeroes, trailing zeroes, binary, octal, hexadecimal, floating, string, ASCII char,  etc. Uses colon : followed by its formatting code
# d	Decimal (base-10 integer).	f"{42:d}" → 42
# b	Binary (base-2 integer).	f"{42:b}" → 101010
# o	Octal (base-8 integer).	f"{42:o}" → 52
# x	Hexadecimal (base-16, lowercase).	f"{42:x}" → 2a
# X	Hexadecimal (base-16, uppercase).	f"{42:X}" → 2A
# n	Decimal, localized based on locale.	f"{12345:n}" → 12,345 (if locale is US)

In [ ]:

# CRASHCOURSE  SQL
#Vocab: 
# "Schema": The structure of a database; the tables, columns, data types, yes, and then also the relationships, and rules/constraints in a database. So... everything about the database, minus the actual data... Like a skeleton without the meat

# Concept of having tons of broken up tables, then joining them in clever ways the draw out their data. 

# "relational" : # sql revolves around breaking your data into tiny tables such that each table has columns which are designed to be set in stone. SQL column names need to be known before your create tables. 

# 'Nullable' - Able to be NULL. (quicker to say 'this column is nullable' than 'this column is able to be Null')            To make relationship()s nullable, use |Null in their typing. Ordinary columns() are default nullable(?)

# "Primary Key": a column which is unique, and not NULL. Allow sql to generate a incrementing primary key column for you, using 'id INTEGER PRIMARY KEY'. (If you provide an id value, sql will use your id value, otherwise id will autoincrement

# "Foreign Key": A column in table which references another_table's columns, so as to prevent insertion into table if values dne within referenced column(s):  FKs prevent insertion into table, if the values dne in the referenced columns. Referenced columns are often the primary key for another_table; your foreign keys often will and should reference another tables' primary keys. Foreign keys are important for joins and rule enforcement and more. Foreign keys enforce relationships between tables; if a column is set as a foreign key, and you try to add a value, which doesn't exist in the referened column, sqlite will stop the action. If you try to delete row in table, but row's FK is referenced in other_table(and no cascading behavior is set), sql will stop the action

# "Surrogate" key : A primary key generated for you, by sql. The most common primary key, "id integer primary key" is a surrogate key. Basically in all cases, you should use 'id integer primary key',  including when natural keys( think mpn) can't be used. the autoincrement starts at 1; autoincrement is not zero-indexed.

#To add a row, provide a value for each column where you want to add data: 
# INSERT INTO table(col1, col7) VALUES('a', 'g')
# col2-6, since no data was given, get 'NULL' values: NULL means 'no data'. 
# Empty strings, or the integer 0, are actually data. "" and 0 are not the same as NULL; no data.

# Adding a row with a PK which does not exist in a referenced FK's column,  will fail; sql will stop the transaction 

# SQL keywords are often written all capital letters. Thats why your info should be lowercase(so you can easily distinguish)

# "CREATE TABLE IF NOT EXISTS my_table (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)" 

# SQL Datatypes: 
# 
# Sqlite would classify floating point numbers like 3.14159265 as REAL. REAL uses 8 bytes, 64 bits.  sqlite does not have float, or double types. 

# id, name, age, are the names I chose for my columns. 
# INTEGER and TEXT are datatypes in sql. INTEGER is whole numbers, like 1, 35, but no decimals, 1.0 and 35.0 would stored as REAL(8bytes).
# "PRIMARY KEY" is a 'constraint'. "PRIMARY KEY" designates this column as the primary key of this table. When a column is designated a 'PRIMARY KEY', each value in the column must be unique, and none can be NULL. (Q: What happens if I try to give an identical primary key?)
#In sqlite, an INTEGER PRIMARY KEY creates an autoincrementing column 
        
#    Do you have to specify the primary key value with every INSERT? No. In sqlite, PK is automatically incremented, if no PK supplied. 
# AUTOINCREMENT prevents reuse of PKs; previents reuse of pks from deleted rows. Not AUTOINCREMENT is not needed. 

# SQL constraints or rules limit the type of data that may be entered to ensure reliability. If there is a violation between the constraint and the data action, the action is aborted. Plus, INDEX and PRIMARY KEY constraints make lookups faster(but that isnt' useful to me) 

# Common Constraints: 
# PRIMARY KEY - A combination of a NOT NULL and UNIQUE. Uniquely identifies each row in a table
# FOREIGN KEY - Only allow values in the FK column which are present in another tables' set of columns
# NOT NULL - Ensures that a column cannot have a NULL value
# CHECK - Ensures that the values in a column satisfies a specific condition
# UNIQUE - Ensures that all values in a column are different
# DEFAULT - Sets a default value for a column if no value is specified
# CREATE INDEX - Used to create and retrieve data from the database very quickly

# In Sqlite, foreign key enforcement is actually default off, enforce foreign keys with "PRAGMA foreign_keys = ON;" 

In [ ]:
#PRIMARY KEY
#Primary key is unique, and not NULL, and its an int. 
# MPN actually doens't make for a good PRIMARY KEY. 

# Store the MPN with a UNIQUE constraint to ensure no duplicates:
# Column constraints may be added right after column name, or they may be defined at the end of the query in the table constraints section. 


"""CREATE TABLE Parts (
    id INTEGER PRIMARY KEY,                         # Designate id as primary key. unless id explicity given, sql will auto-increment id's value 
    mpn TEXT UNIQUE,                                # Give mpn the UNIQUE constraint; attempts to add 'abc123' more than once will not succeed
    description TEXT NOT NULL,                      # Give description the NOT NULL constraint; data for description must be provided          
    manufacturer TEXT DEFAULT 'default value'       # Give manufacturer the DEFAULT value 'defaultValue'
    voltage INTEGER CHECK (voltage >= 0)            # Give voltage constraint 'voltage >=0>'  
);"""

# Could use a composite key, a primary key spanning two columns.

# Primary key may be made of two+ columns: 
"""CREATE TABLE table_name(
   column_1 INTEGER NOT NULL,
   column_2 INTEGER NOT NULL,
   ...
   PRIMARY KEY(column_1,column_2,...)
);"""

"""CREATE TABLE Parts (
    id INTEGER PRIMARY KEY,          -- Unique surrogate key
    mpn TEXT NOT NULL,               -- MPN column
    manufacturer TEXT NOT NULL,      -- Manufacturer to differentiate
    UNIQUE (mpn, manufacturer)       -- Enforce MPN uniqueness per manufacturer
);"""

# CREATE TABLE default makes a 'rowid' table; adds column 'rowid' with 64bit signed number. 
# If user sets primary key to single integer column: column_name INTEGER PRIMARY KEY:  column_name will serve as an alias to rowid. This makes for fast querying with 'B-Tree' algorithm.




In [ ]:
# Foreign Key : references another tables' columns, so as to prevent insertion into table if primary key dne. Only allow values in the FK column which are present in another tables' set of columns
# Often, FK references another table's 'id integer primary key' column, since this is a common PK column; FK would be integer
#FK must be explicitly assigned in sqlite. 
# Write your FK constraints in the CREATE TABLE query, at the end of the query, where table constraints go 

# linking tables : FOREIGN KEY (col_name) REFERENCES table (col_name) 


# foreign keys reference a column(often a pk) in another table to stop insertion of values if dont' exist in referenced columns
# 
"""
CREATE TABLE IF NOT EXISTS table_name(
    col1 INTEGER PRIMARY KEY
    col2 STRING 
    col3 FOREIGN KEY REFERENCES other_table.pri_key_col
"""

"""
CREATE TABLE IF NOT EXISTS table_name(
    col1 INTEGER NOT NULL
    col2 STRING 
    col3 FOREIGN KEY REFERENCES other_table.pri_key_col
"""



In [ ]:
# Add FK with ALTER TABLE
ALTER TABLE Retailer
ADD FOREIGN KEY (Retailer_id) 
REFERENCES Customer(Customer_id);

# Drop FK with ALTER TABLE
ALTER TABLE table_name
DROP FOREIGN KEY foreign_key_name;

# Propogate changes to foreign keys on record deletion/update 
FOREIGN KEY (id) REFERENCES product_details(id) ON DELETE CASCADE ON UPDATE CASCADE;

In [ ]:
    
https://www.geeksforgeeks.org/how-to-create-a-table-with-a-foreign-key-in-sql/

CREATE TABLE Customer(
    Customer_id int primary key,
    Customer_name varchar(20),
    Customer_Address varchar(20),
)

INSERT INTO Customer (Customer_id, Customer_name, Customer_address)
VALUES
(101, 'Geek 1', 'Chennai'),
(102, 'Geek 2', 'Delhi'),
(103, 'Geek 3', 'Bombay'),
(104, 'Geek 4', 'Pune'),
(105, 'Geek 5', 'Nashik');


CREATE TABLE Sales (
    Sale_id INT PRIMARY KEY,
    Customer_id INT,
    Item_id INT,
    Payment_mode VARCHAR(20),
    FOREIGN KEY (Customer_id) REFERENCES Customer(Customer_id)
);



INSERT INTO Sales (Customer_id, Item_id, Payment_Mode)
VALUES
(101, 1334151, 'COD'),
(101, 16652, 'Debit Card'),
(104, 1009, 'Paypal'),
(102, 14543, 'COD');


DESCRIBE Sales;

In the Sales table, the Customer_id column is a foreign key.
This foreign key references the Customer_id column in the Customer table, ensuring that each sale record corresponds to a valid customer in the Customer table.

#ALTERNATIVE: FOREIGN KEY Constraint After Column Declaration allows FK consisting of multiple columns

 Customer_id INT PRIMARY KEY,
 Item_id INT,
 Payment_Mode varchar(20),
 Payment_id INT,

 CONSTRAINT customer_payment_id
 FOREIGN KEY (customerId, paymentId) 
 REFERENCES Customer(customerId, paymentId)
);

In [ ]:
import sqlite3 

#USING PLACEHOLDERS SQL INJECTION PROTECTION

cur.execute("CREATE TABLE IF NOT EXISTS test(col1, col2)")
# in SQL--any SQL including sqlite, you have to specify column names in all tables: no tables without column names 
#Check existing tables in db:
# You have to create a cursor object. Your cursor will look ar the rows and give you data.


#GET tuple of tables in db ('capacitors', 'resistors', etc)
table_names = cur.execute("SELECT name FROM sqlite_master")
table_names.fetchone()

# "SELECT column FROM table"
res = cur.execute("SELECT capacitance FROM capacitors") 
res.fetchall()#->[(8nf,), (7nf,)] one tuple per row

#INSERT new rows;records, into table
cur.execute("""
    INSERT INTO movies VALUES
        ('Monty Python and the Holy Grail', 1975, 8.2),
        ('And Now for Something Completely Different', 1971, 7.5)
""")
# The INSERT opens a 'transaction'. To save the changes, we must COMMIT the changes. 


# Always use placeholders (?) instead of string formatting to bind Python values to SQL statements, to avoid SQL injection attacks 
data = [('d1.1', 'd1.2', 'd1.3'), ('d2.2', 'd2.2', 'd2.3'), ('d3.1', 'd3.2', 'd3.3')]
cur.executemany("INSERT INTO movie VALUES(?, ?, ?)", data)
con.commit()  # Remember to commit the transaction after executing INSERT.
# Avoidance of sql injection attacks does mean that we must do some string formatting-- since, sqlite3 must take string literals as arguments.

# Tutorial from https://docs.python.org/3/library/sqlite3.html
# the sqlite_master table is built-in to SQLite -- 
# Use the sqlite_master table to see 'name' of all your tables
# Note again, sqlite_master is builtin. so is 'name'. 
result = cur.execute("SELECT name FROM sqlite_master")
result.fetchone()
#fetchone(). fetch one record; fetch one row. -> ('table_1',) Note the weird , at the end is normal/OK, it tells python its a tuple with one element instead of a string surrounded by parentheses
result = cur.execute("SELECT name FROM sqlite_master WHERE name='select_this_table'")
# if table DNE. NO error is raised, but, if you result.fetchone(), None is returned.



# Always use placeholders (?) instead of string formatting to bind Python values to SQL statements, to avoid SQL injection attacks 
data = [
    ("Monty Python Live at the Hollywood Bowl", 1982, 7.9),
    ("Monty Python's The Meaning of Life", 1983, 7.5),
    ("Monty Python's Life of Brian", 1979, 8.0),
]
cur.executemany("INSERT INTO movie VALUES(?, ?, ?)", data)
con.commit()  # Remember to commit the transaction after executing INSERT.


#Select certain columns from table, and order them by column descending DESC. Ascending ASC is default.   
result = cur.execute("SELECT year, title FROM movie ORDER BY year DESC")
    #-> List[tuple(year, item)] like [(1998, 'Dinosaur), (1975, Monty Python), ...]
for row in result: 
    print(row)

#Select certain columns from table, and order them first by column1 ascending. If entries for column1 match, then order by column2 descending.
result = cur.execute("SELECT year, title FROM movie ORDER BY year ASC, rating DESC")   

#
#Order by 
# SQL Structured Query Language. sqlite3 is a python builtin, so you can import sqlite3 without downloading addtnl packages
# db file is a binary-- not a text file -- so it needs a sql DB viewer to view it. 
# https://inloop.github.io/sqlite-viewer/


# Get all rows and all olumn by order
# Queries should be in "" -- that way you can use '' within the query
# Tell the cursor to execute some SQL code 
cur.execute("SELECT * FROM 'ips' ORDER BY asn LIMIT 0,30")
print(cur.fetchall())

# Ge all certain rows and certain columns
cur.execute("SELECT 'address', 'asn' FROM 'ips' WHERE asn < 300 ORDER BY asn")
print(cur.fetchall())


#Get all rows WHERE condition AND pattern matches '%sa'
cur.execute("SELECT * FROM 'ips' WHERE asn < 300 AND domain LIKE '%sa'")
# domain is NOT in quotes
# % means 'every character' 
# %sa means 'every character followed by 'sa'
results1 = cur.fetchall()
print(results1)

for row in results1:
    print(row)

results2 = cur.fetchall()   
# fetching all AFTER fetchall() was already done : cursor has already run to the end for this SQL query. Each query can only do fetchall() once




In [ ]:
# cGPT advice on : MPNS are 99% unique, but what if they aren't? 
# sql allows UNIQUE rules 
#VendorPN vs ManufacturerPN : a problem worth bothering with? 
CREATE TABLE Parts (
    id INTEGER PRIMARY KEY,          -- Unique surrogate key
    mpn TEXT NOT NULL,               -- MPN column
    manufacturer TEXT NOT NULL,      -- Manufacturer to differentiate
    UNIQUE (mpn, manufacturer)       -- Enforce MPN uniqueness per manufacturer
);
#No way a manufacturer reuses an MPN. BUT, 
panasonic makes abc123, and hypothetically Kyocera could make 123abc, too. 
UNIQUE (mpn, manufacturer) : one mpn per manufacturer : panasonic abc123 gets one entry


panasonic makes abc123, and sells it to vendors Digikey and Arrow. 
UNIQUE (mpn, vendor)    : one mpn per vendor :  digikey abc123 gets one entry : 
not ideal : bc dk could carry abc123 from panasonic AND abc123 from Kyocera. 
This is so unlikely I could just prevent it for now. 
o

# Values in the specified combination of columns must be unique.
# I need UNIQUE( mpn, 

# I may buy resistorA from Panasonic off of Mouser AND/OR Digikey. 
# So, I may have mpn duplicates. 
# Its a big problem If Panasonic has ResistorA, but I buy ResistorA from KOA by accident. 
# So, if mpns are the same, I should prompt user to append manufacturer onto part_name. 

#Really, we are buying parts from vendors. As such, I should go concern myself if Digikey's dk_part_number matches arrow's arrow_part_number. 


In [ ]:
# MyFav queries
query="SELECT name FROM squlite_master"   # sqlite_master is built-in to sqlite. See 'name' of all tables in your db
query="CREATE TABLE IF NOT EXISTS my_table (col1, col2, col3)"
query="SELECT * FROM my_table LIMIT 0" 
query= "SELECT year, title FROM movie ORDER BY year DESC"
query= "SELECT * FROM 'ips' ORDER BY asn LIMIT 0,30"
query= "SELECT 'address', 'asn' FROM 'ips' WHERE asn < 300 ORDER BY asn "
query=  "SELECT * FROM 'ips' WHERE asn < 300 AND domain LIKE '%sa'"
q= "SELECT * FROM users WHERE name='Alice' LIMIT 1"
# SELECT column FROM table
"SELECT capacitance FROM capacitors"

query = """
    INSERT INTO my_table VALUES 
        (val11, val12), 
        (val21, val22)
    ) 
 """


query = """
    INSERT INTO my_table VALUES 
        (?, ?)
    ) 
 """
data = """
    (val11, val12), 
    (val21, val22)
"""

cursor.executemany()
cursor.execute()

result.fetchone()
result.fetchall()


In [ ]:


install vscode symbol_select #     sqliteViewer            ctrlShiftP -> Run Query -> select database. start .sql file, type sqlite3 queries. end queries with; 
https://www.youtube.com/watch?v=Jujl45qkCv8
Install pluging then : ctrlShiftP --> SQLite: RunQuery | ctrlShiftQ. 





SELECT * FROM table1 
INNER JOIN table2
ON table1.col2 = table2.col2
INNER JOIN : select matching data. (both tables have same column?)                        
LEFT JOIN : All of table1 AND matching records of table2
FULL OUTER: records if match in either table

CREATE TABLE users (user_id INTEGER PRIMARY KEY, username STRING);
CREATE TABLE orders(order_id INTEGER PRIMARY KEY, user_id INTEGER, product STRING);
                    
INSERT INTO users(username) VALUES('Stacy');
INSERT INTO users(username) VALUES ('Amy'),  ('Mike');   -- Insert multiple rows in one INSERT 

INSERT INTO orders(user_id, product) VALUES(1, 'skittles'), (2, 'sword'), (3, 'car'), (4, 'XboxOne'), (5, 'puzzle');

SELECT * FROM orders;
SELECT * FROM users;

SELECT * FROM orders INNER JOIN users ON orders.user_id = users.user_id;
SELECT * FROM orders INNER JOIN users ON orders.user_id = users.user_id AND product = 'cucumber'; -- orders.product more explicit; product OK here bc product col is only product col.
SELECT * FROM orders INNER JOIN users ON orders.user_id = users.user_id AND orders.product = 'cucumber'; --better practice to use table.col syntax 

SELECT users.username, order_id FROM orders LEFT  JOIN users ON orders.user_id = users.user_id;
SELECT users.username, order_id FROM orders INNER JOIN users ON orders.user_id = users.user_id AND orders.product = 'cucumber';
SELECT users.username, order_id FROM orders LEFT JOIN orders ON orders.user_id = users.user_id;



# The concept of one to one, one to many, and many to many 

In [ ]:
https://www.geeksforgeeks.org/sql-join-set-1-inner-left-right-and-full-joins/
SELECT s.roll_no, s.address, s.phone, s.age, sc.course_id 
FROM Student s 
JOIN StudentCourse sc ON s.roll_no = sc.roll_no

In [ ]:

column_names = part.get('product_attribute').keys()
table = part.get('product_attributes').get(category)
column_rules = 'NotSupported atm'
column_name_and_rule = column_names 


column_defs = ','.join([{column_name} {column_rule} for column_name, column_rule in column_name_and_rule.items()])
key = f"FOREGIN KEY {id} REFERENCES "
f"CREATE TABLE IF NOT EXISTS {table} ({columns}) {key}"




In [ ]:
#String methods 
# .join()   
# .join(iter) takes all items in iter and joins them into a string. 
# .join() takes an iterable, so, string or list both count

# These do the same thing basically
columns = ['col1', 'col2']
column_defs = ','.join([f"{column_name}"  for column_name in columns])
column_defs = ','.join(map(str, [10,20,30]))

# .split() -> list of the string 
s = 'This is a sentence'
word_len = [] 
for w in s.split(): 
    word_len.append(len(w))
shortest = min(word_len)
print(shortest)



In [ ]:


# SQLite does not support placeholders for table names or column names, only VALUES. Construct the query using Python's string formatting except @ VALUES (To prevent SQL injection, you may wish to check table name are alphanum_ only ) 
#safe sqlite3 : execute a query. Using sqlite3 placeholders (good) just do NOT use any of pyhton's string formatting techniques. 
'''

data = [
    ("Monty Python Live at the Hollywood Bowl", 1982, 7.9),
    ("Monty Python's The Meaning of Life", 1983, 7.5),
    ("Monty Python's Life of Brian", 1979, 8.0),
]
cur.executemany("INSERT INTO movie VALUES(?, ?, ?)", data)

con.commit()  
# Remember to commit the transaction after executing INSERT.
Notice that ? placeholders are used to bind data to the query. 
Always use placeholders instead of string formatting to bind 
Python values to SQL statements, to avoid SQL injection attacks 

'''

In [ ]:
#python requests module
import requests 
# The 'Response' object has several useful methods: 
# requests.request()            -> requests.Response object
# response.headers                        
# response.content              Returns the byte string received from the server. Used when images or videos or compressed file is coming in.             
# response.status_code          Returns the HTTP status code. 200 level indicates success
# response.text                 Returns the content of the response as a string.                     
# response.json                 If the response is JSON, it parses and returns the content as a Python dictionary.( if response.status_code == 200 else "Not available")
# response.url                  
# response.reason                     
# response.cookies                        
# response.history

In [ ]:
import os 
cwd = os.getcwd()                                                                   #CurrentWorkingDirectory cwd 
print(os.listdir(cwd))


In [ ]:
#The python os module useful commands 
import os 


cwd = os.getcwd()                                                                   #CurrentWorkingDirectory cwd 
os.listdir(cwd)
if not os.path.exists("test_dir"):                                                  #check if path exists.             
    os.mkdir("test_dir")                                                            #create directory. Fails if dir already exists, so use if os.path.exists() to skip, if path already exists. 

os.makedirs("test_dir1/test_dir2/test_dir3")                        #create directories and all branch directories. 
# os.makedirs() has no ability to make files. os.makedirs("dir/file.txt") creates a folder 'file.txt' never a file. 


# os.makedirs('notadir/file.csv') 
# os.makedirs() threrrors if dir already exists
# os.makedirs('a/b/c/c.csv')

file = 'log/robs_robots_parts/A/B/C/C_attribute_group.csv'
# os.makedirs(file)

with open(file, 'w' , newline="") as fo:
    csv_writer = csv.writer(fo)
    csv_writer.writerow([1,2,3])
# Permission denied: 'log/robs_robots_parts/A/B/C/C_attribute_group.csv' ... perhaps bc os.makedirs("C_attribute_group.csv") creates a folder, not a file, So trying to open folder to write to it like a file could throw permission denied... on top of it being a folder not file, file explorer->permissions also shows it it read-only
# I can makedirs(file) but I cannot write to it ! 
# os.makedirs() makes folders. folders are read-only by default, just as folders made manually with file explorer are read-only by default. (read-only applies to files in folder) 
# linux: chmod +w or something... how do I make a windows file writeable? 
# I made this file, why is it write-protected? 

#TODO REMOVE WRITE PROTECTION ON OS.MAKEDIRS files. was not the problem, os.makedirs(file.txt) makes folders, not files, and you can't write folders sooo...


#Make a file. (Check if the file exists, first. This will avoid overwrite)
if not os.path.exists("file.txt"):
    with open("file.txt", "w") as file: 
        file.write('Starter text')
        
# change file permissions
# file_path = "log/robs_robots_parts/A/B/C/C_attr_group.csv"
# if os.path.exists(file_path):
#     os.chmod(file_path, 0o666)  # Set read and write permissions for the file
#     os.chmod(file_path, 0o777)  # Full read, write, and execute permissions for all users
    # Windows 'attrib' command attrib -r "log/robs_robots_parts/A/B/C/C_attr_group.csv"

# octal values are useful to set permissions. 
# Owner, group, others
# read(4) write(2) execute(1). 
# 0o777	rwxrwxrwx	Full access for all users.
# 0o755	rwxr-xr-x	Owner: full, others: read+exec.
# 0o644	rw-r--r--	Owner: read+write, others: read.
# 0o600	rw-------	Owner: read+write, others: none.
# 0o444	r--r--r--	Read-only for all.
# 0o000	---------	No permissions for anyone.

    

In [ ]:
#python csv module 
import csv 
import os 

path= 'file.csv'
data_list = """ 
a,b,c
1,2,3
"""
#This string can't work with csv.reader() as csv.reader() expects a file-like object, not a string. 
#use io.StringIO to turn a string into a file-like object. fo= io.StringIO(str)

data_dict = {'a':1, 'b':2}

if not os.path.exists(path):
    with open(path, 'w') as file: 
        csv.DictWriter()
        file.write(data_list)
else:
    records = []
# reading csv file
with open(path, 'r') as csvfile:
    # creating a csv reader object
    csvreader = csv.reader(csvfile)

    # extracting field names through first row
    fields = next(csvreader)

    # extracting each data row one by one
    for row in csvreader:
        records.append(row)
        
with open(path, 'r') as file: 
    csvreader = csv.DictReader(path) #read csv file into dict form. {key:value} pairs' key takes header row; value takes remaining rows
    dict_list = []
    for record in csvreader:
        dict_list.append(record)
for data in dict_list: 
    print(data)
    
with open(path, 'w') as file: 
    writer=csv.DictWriter(file, fieldnames = list(data_dict.keys()))
    writer.writeheader()
    writer.writerows(list(data_dict.values()))
    


In [ ]:
def func(a,b):
print('h')

In [ ]:
# Basic basics Python Basics 
# This hyphen greater than -> is shorthand for 'returns' -> means returns like 's'+'tring' -> 'string'  or  add(2,2) -> 4 
# In coding, a 'function' is code that does stuff, like adding 2+2. In python, you can define your own functions with the keyword 'def'. Functions take arguments;parameters, like an 'add' function would take two numbers as parameters;arguments  and add them together. 
# In python, there are positional;default arguments, and nonpositional;keyword arguments.

x = 1 
y = 2 
z = 3 


def add( a , b ):  # define a function with the 'def' keyword, followed by the function name, followed by (arguments enclosed in parentheses), FINALLY FOLLOWED BY A COLON : Forget the colon, you will get a  SyntaxError: expected ':' 
    # then, INDENT the code of the function. If you do not indent, python will IndentationError: expected an indented block after function definition on line 1
    print( a + b ) 
# This function has positional; default, arguments. a is supposed to go first, and b goes second, mess up the order and... well nothing bad happens here because a + b is the same as b + a.


    

    
# A positional;default argument's position is important: is the argument supposed to be the first, second, or third argument? The order matters.

# A nonpositional;keyword arguemnt, is supplied with a keyword,, like x=2 and the keyword 




# Understanding conditional statements doubleEquals 
print(5 == 5) # True, five equals five 
print(5 == 6) # False, five is not six 
x = 10 
y = 10
print(x == y) #True 

x = 1
y = 2 
# print(x = y) Trick question! = vs == ! 
print(x == y) # False 


x = 100 
y = x 
print(x == y) #True, y and x are equal 
print(x is y) #  True, x IS y 

x = 10 
y = 10 
print(x == y) # True 
print(x is y) # also true funFact: 10 is 10 -> True because ints are immutable. Lists are mutable, so [1] is [1] -> False 

x = [1]
y= [1]
print(x is y) # -> False, as x and y are mutables, each with their own memory address 
print(id(x)) #-> 2498322152320 or similar-- changes each time--

print(id(y)) 



In [ ]:
D = {'a':None}
print(D.get('a')) # dicts can hold None as values -- but xml cannot 
import lxml.etree as etree 

# root = etree.Element('root', key = 'value', key = None) None disallowed in xml 
def func(a,b,c,d): 
    print(a,b,c,d)
    
func(*map(str,(1,2,3,4)))

In [ ]:
# myfav painter review essentials brush pen
#         painter.setBrush(Qt.GlobalColor.red)
#         painter.setBrush(Qt.BrushStyle.NoBrush)
#         painter.setBrush(QColor(100,100,100))
#         painter.drawEllipse(QRectF(-8 + eye_movement, 0, 4,4)) # algorithm for moving eye 
    
# items vs paintings: paintings can't be grabbed by mouse. Items are clickable, draggable, with signals and slots and more. Paints are visual. in my program, I would want my symbols to be items, not paints, so the user can click on them

# .paint() is called by the View, and paints the contents on it's item (widget, if not None). Call .update() if changes are made to .paint(), else visual artifacts. Constrain all painting within boundaries of boudningRect(), else, risk artifats. QPainter drawn outlines(QPen) reside half in,half out, the rendering shape. QGraphicsView does not clip the painter for you.
   # QGraphicsItem.paint() vs QWidget.paintEvent(). Analogous.  but widgets get events, while items don't. PE() called when widget needs to be repainted(programmatically, via .update(), or automatically, after drag resize, uncovering, etc). paint() is usually called by the View, paints the contents on an item, in local, item, coordinates. 
   # def paint(self, painter, option, widget): # widget default 0. cached painting, 0. if widget, painted on widget. 


# Scene tings
scene = QGraphicsScene()
rect_item = QGraphicsRectItem(10,10, 100,100)
scene.additem(rect_item)

scene.setItemIndexMethod(QGraphicsScene.ItemIndexMethod.NoIndex)
scene.setSceneRect(-300, -300, 600, 600) # Combine this with 

# View tings: antialiasing, cacheing, viewport update mode, drag mode(scrollHandDrag), WindowTitle, window size:
view = QGraphicsView()
view.resize(400, 300)
view.setScene(scene)
view.setWindowTitle("Colliding Mice")

view.setRenderHint(QPainter.RenderHint.Antialiasing)
view.setCacheMode(QGraphicsView.CacheModeFlag.CacheBackground)
view.setViewportUpdateMode(QGraphicsView.ViewportUpdateMode.BoundingRectViewportUpdate)
view.setDragMode(QGraphicsView.DragMode.ScrollHandDrag) # Does not allow me to drag 







In [ ]:
# myfav tutorials 
# To get started with PyQT/PySide:
# 1) Free and good rewrite of the offical PySide6 examples: https://github.com/Erriez/pyside6-getting-started
# 2) free & good Pythonguis.com 
# 3) Martin fitzpatrick aka github user Erriez aka Reddit user u/mfitzp is on pythonguis.com, he has this sweet tutorial:  https://www.pythonguis.com/tutorials/pyqt-qgraphics-vector-graphics/.    Fitzpatrick has a $20 book I can fully reccomend for beginners/intermediate. (book is in QT5- as of 2025 QT6 current -- but 5 and 6 are practically the same thing) https://www.pythonguis.com/pyqt6-book/. 
# 4) Download the 'essential' examples from the official pyside6 website. There's ~30 projects, download em all after ~40-100 hours on pythonguis.com

# Intro to scenes/views https://www.pythonguis.com/tutorials/pyqt-qgraphics-vector-graphics/
# official PySide6 examples. Download the 'essentials'-- there's ~30 of them. Verbose, uncommented. Work through them with the excellent Qt documentation(whihc is in C, need to google C syntax to read)

In [ ]:
#affect the 'stacking order' of items with item.setZValue(500). Z as in z-coordinate, also see item.stackBefore(otherItem) and stackAfter()
# Flags CrashCourse Flags Essentials 

# QGraphicsItem Flags: Five flags available in enum:QGraphicsItem::QGraphicsItemFlags
# # Most used: movable, selectable.
# ItemIsMovable: Allow clickNDrag on item 
# ItemIsSelectable: enable setSelected() to toggle selection for the item; allow item to be selected with rubber band selection. Selected items appear with a dotted rect surrounding them.
# ItemIsFocusable: enable keyboard input focus; item is an input item; allow item to accept focus, which allows the delivery of keyboard events to QGraphicsItem.keyPress/ReleaseEvent()
# ItemClipsToShape: The item clips to its own shape; disable interaction outside its own shape.
# ItemClipsChildrenToShape: Children of item cannot draw outside item's shape.  
#Enforce flags ItemClipsToShape and ItemClipsChildrenToShape with QGraphicsScene/View().drawItems()

In [ ]:
# git github https://www.reddit.com/r/learnprogramming/comments/evpxcm/an_introduction_to_git_and_github/

Basic/ Most Used Bash Commands (There are several modifiers for each command)

# ls - lists the folders and files in the working directory (the current directory you are in)

# cd - changes directory

# pwd- used to find the path for the current directory

# mkdir- make a directory

# touch - update the access and or modification date of a file or directory without opening, saving or closing the file.

# cat - print files to stdout

# mv - moves files and folders

# cp - copies files or folders

# rm - remove files and folder (look into modifiers for this one)

# chmod - Change mode so you can set permissions for read, write and execute for the user, members of your group and others. (Binary can be used for this)

# man - can be used to look up any commands ie man cd

Using GitBash/Terminal to Access GitHub

# Configure Git via git config --global user.name "[name]" and git config --global user.email "[email address]"

# Navigate to your working directory (Keep in mind you cannot just cd to the directory, you have to work your way to it, so I personally keep a folder called Programming in my home directory)

# Initialize a Git Repo via git init

# Now, this is where you can branch-of, you have two options, pushing a new repo or pushing a preexistent repo.

# Pushing a New Repo

# Commit your repo via git commit -m "first commit"

# Remote add your repo via git remote add origin <url>

# Push to your repo via git push -u origin master

# For Pushing an Existing Repo

# Remote add your repo via git remote add origin <url>

# Push to your repo via git push -u origin master

# Now that you have your repo set up, these are some helpful commands:

# git status Used to check what has changed ie additions and deletions

# git add <file> Used to add files to commit if used with a period (.) it adds all of the files

# git commit -m "message" Use to commit changed, but it is on the local system, the -m can be changed to things such as -u which is an update but it is recommended to keep with an -m

# git push Used to push changes to GitHub

# git reset Can be used after commit to reset the commits (Good if you accidentally add a file you did not want)

# git pull <url> Can be used to pull from any git repo, leave the URL out if your updating your current repo


# .gitignore

# The .gitignore file is useful for stopping certain files from committing automatically. It should automatically be in a repo when you create a project. To use it just cd to the directory where the file you want to exclude is and use pwd to find the directory pathing. Then copy the path into the file, it should look like a text file, and then add the name of the file you want to exclude.

# Example: User/Jun/Programming/src/something.java


# Branching in Git (For advanced user)

# Branching is useful when many people are working on the same project or when you have multiple versions of the same project. The major advantage of branching is when you want to add a feature without compromising the integrity of the master branch.

# Branching Commands

# git branch [branch-name] Used to create a new branch

# git checkout [branch-name] Used to switch branches

# git merge [branch] Used to merge branch commits (usually people use this with a branch and the master)

# git branch -d [branch-name] Used to delete a branch

PS C:\Users\robby\OneDrive\part_database> git config --global user.name "[robby]" 
PS C:\Users\robby\OneDrive\part_database> git config --global user.email "[rdrisc33@gmail.com]"
PS C:\Users\robby\OneDrive\part_database> git init 
Initialized empty Git repository in C:/Users/robby/OneDrive/part_database/.git/
PS C:\Users\robby\OneDrive\part_database> git commit -m "firstCommit"
On branch main



In [ ]:
import lxml.etree as etree 
# root = etree.Element('root', {'A':1})  TypeError: Argument must be bytes or unicode, got 'int'

root = etree.Element('root', {'a':None}) 
print(root.get('d',None))# TypeError: Argument must be bytes or unicode, got 'NoneType'
print(root.attrib.get('a'))

root.attrib.update({'b':'c'})
print(root.attrib)

        

In [ ]:
# python lists, list methods, lists, List Slicing, String Slicing, list basics 
# Select parts of lists with the slice operator []
# [ start : end ] 
# [ start : end : step ]
# [sliceStart(default startOfList) : sliceEnd(default endOfList)]


# minus sign indicates to slice from the end of the list. 
# slicing happens inplace; slicing returns nothing; L=L[x:y] returns nothing and won't affect L; use L[x:y] not L=L[x:y].
#sliceStart includes sliceStart, sliceEnd excludes sliceEnd. 

l= [0,1,2,3]

# l[len(l)] # Because l is zero indexed, l[4] is out of range  
l[len(l) -1] 
# l[include:exclude]
l[0]        
l[-1:]      #sliceStart at last character sliceEnd to end of list
l[:-1]      #sliceStart at begin of list sliceEnd at last character 
# l[9000] # err, list index out of range 

l[2] 
l[2:3]
reverse = l[::-1] 
# reverse a list by slicing it from st to end with a step of -1 

# cast as binary string. will include prefix '0b'.
bin(10) 
bin(10)[2:]


L = [2,0,1]
L.sort()
print(L)
print()

L.append(3)
print(L)
print()

L.insert(1,4) # L.insert(index, object) inserts object before index. Here, we insert '4' before index '4'. Helpfully, if you overshoot the list index, it'll just insert at the end, like an append().
print(L)
print()

# Confusing copies shallow copy deep copy list copies 
l = [1,2,3,4,5] 
slice = l[2:4] 
slice[0] = 100
print(slice) # the slice chagned
print(l)  # but the list unchanged
# slices are copies of lists(shallow)


l = [[0,0],[1,1],[2,2]]
copy = l[0:2]
print(copy)
copy [1][0] == 100
print(copy)
print(l)
# slices are shallow copies. Changing a nested value in the copy changed the OG list, too

copy[0] = [999,999,999]
print(copy)
print(l)
# slices are shallow copies. changing a unnested value didn't change the OG list

nested_numbers = [[0, 1, 2], [3, 4, 5], [6, 7, 8]] # this creates four list objects. One outer, three inner. Outer is named 'nested_numbers'. 
copy = nested_numbers[:2]
# creates a list object named 'copy'. BUT, the inner lists are the SAME objects as the origional list: 
print( id(nested_numbers[0]) == id(copy[0]) ) # True-- same objects in memory.
print(id(nested_numbers) == id(copy)) # false -- different objects in memeory

### SLICING in ARRAYS/PANDAS 
# Concept of : being your slice start:end:step and , delineating row_slice , column_slice
# Note that slicing of 2D ARRAYS has a syntax that LOOKS like slicing a LIST, but you cannot slice a 2D list like you can slice a 2d array. This is kinda how pandas.loc[] works.

In [ ]:
#Rote Memorization Practice 
# You shouldn't memorize everything-- thats what google is for-- but you should memorize stuff you find yourself looking up again and again. 

# A program is a chair, and programmers are carpenters. The carpenter who has a tablesaw, power drill, and circular saw can build better chairs than the carpenter with a hammer 

# Turn this list into a string 
l = ["zebra", "apple", "bat", "ant"]
s = ','.join(l) 

sentence = "This is a sentence" 

def rev_words(sentence): 
    l=  []
    for w in sentence.split(): 
        l.append(w[::-1])
    return ' '.join(l) 
print(rev_words(sentence))

def rev_words_using_list_comp(sentence):
    return  ' '.join(w[::-1] for w in sentence.split())

print(rev_words_using_list_comp(sentence))

# print(sorted( [1, 5, 3] , key = ))



x = sorted(sentence.split() , key = lambda w: sorted(w)) 
print(x)


In [ ]:
l       = [1,2,3]
idx1    = l[1]
idx23    =l[2:3]
idxlast = l[-1]
idx3to5last= l[3:-5]
reverse = l[::-1]




In [ ]:
# VOCAB VOCABULARY 

# "Protected" functions can only be accessed by member functions of the same class, derived class, or friend class. 
# # syntax: the structure of a sentence. How a sentence is written. How code is written. 
# # syntacticsugar: code shorthand. Lets you type less
# @decorator is syntactic sugar -- decorator is actually a function
# count += 1 is syntactic sugar -- 
# unpacking operators ** and * in python (Note in C++, * is for pointers)
# 'list comprehension' is syntactic sugar too:  L = [x for x in 'asdf']

# "handler" "handler function" : a function that deals with a task; that handles a task. "Call the handler function to process the data"

# 'Iterable'
# 'iterator' 
# 'sequence' : A generic term for ordered data: Lists, tuples, and strings. Dict, and sets, are not ordered, so they are not sequences. 




In [ ]:
# vsCode CrashCourse 

# MyFavHotkeys 


# altZ              ; view --> word wrap                toggle word wrap 
# alt_LeftArrow     ; Go --> Back                       Revert to previous cursor location(Use after "Go To Definition")
# ctrl_K,ctrl_Q     ; Go-->LastEditLocation             Revert to last edit location
# altEnter          ;                                   Create newline below cursor, then move cursor there(.py files, dnw in .ipynb)
# ctrl_K,ctrl_M     ; Top Right ... btn -> maximize group  Make current file fullscreen
# ctrl_shift_P_ChangeTabDisplaySize ;                   Change all indents to size of one tab
# ctrl_H            ; find&replace                      locate all occurences of text, replace them with different text. NOTE: The Find and Replace widget has little buttons to toggle case matching, match only within higllighted text, and exact word matching.
# highlight+F2      ; find&replace across files; renameHighlighted selection across files

# ctrlK, M                              Select language Mode-->Select file association. ( used this to tell vscode that a .sym file is .xml)

# ctrlK,ctrl0                           collapse code blocks  (.py)
# ctrlK,ctrlJ                           expand code blocks    (.py)

# MyFav vscode extensions
#     LiveServer              See your HTML/CSS/JS in a live webpage 
#     SQLite3 Editor          View .db and .sqlite files in HR forms 
#     Outline Map             Outlines code functions/classes, for a high-level overview of your file
#     Kicad Sexpr Highlighter Colors kicad files

# Jupyter Notebooks 
#     (click on cell) 
#         dd              delete cell 
#         ctrlShiftEnter  Enter cell above 
# !pip install pandas #! in jupyter causes the line to be executed in cmd line. Note its use for installing pandas, libraryX, etc, on this particular jupyter kernel. 




Configuration, Initialization in python:

In [ ]:
# Most basic, just store your configs in variables:

client_id = "V2qpx2G3InnrDgGE3zcc11I6AjJy6o1c"
client_secret = "Q5WGGdsHyFyGE5s7"

# Q: What if I have sensitive data I want to keep encrypted? How do I handle that? 


In [ ]:
#String concatenating, Fstrings. 
# Use the + plus symbol to concatenate strings
concatenation ='asdf' + ';lkh' 
# Create strings with f-strings( as in formatted strings) 
s1 = 'simpleString'
s2 = f"Put an F before start quote to engage Fstring formatting. F-string formatting uses curlyBraces to put variables into your string. Put variable s1 into your string with {s1}."
print(s2)

In [ ]:
#Passing functions as parameters 
# python functions are "first class" objects, which means they can be passed as arguments. 

def process(data):
    return data+' processed'

def speak(func, data):
    words =func(data)
    print(words)
    
speak(process, 'calculations')

In [ ]:
# CONFIGURATION WITH JSON : Which is just saving variables, in json, in a separate file. 
# json,yaml,other file types, can also be used to save configurations. 

# json,yaml,other file types, can also be used to save configurations. 

#in file 'config.json': Note not top level name, everything is in {}. 

{
    "app_name": "MyApp",
    "debug": true,
    "database": {
        "url": "sqlite:///mydb.sqlite",
        "timeout": 30
    }
}
import json

# Load configuration
with open("config.json", "r") as f:
    config = json.load(f)

# Access data
print(config["app_name"])           # Output: MyApp
print(config["database"]["url"])    # Output: sqlite:///mydb.sqlite

In [ ]:
# CONFIGURATION WITH ENUMS

# enums assign names to numbers, like READY = 0. Use a dict, whose keys are the enum names, like {Status.READY: './image.png'}. Status.Ready is 0. Dict keys can be numbers. 

from enum import IntEnum

class Status(IntEnum):
    READY = 0
    PLAYING = 1
    FAILED = 2
    SUCCESS = 3
    
STATUS_ICONS = {
    Status.READY: "./images/plus.png",
    Status.PLAYING: "./images/smiley.png",
    Status.FAILED: "./images/cross.png",
    Status.SUCCESS: "./images/smiley-lol.png",
}





In [ ]:
from enum import Enum

class Season(Enum):
    SPRING = 1
    SUMMER = 2
    AUTUMN = 3
    WINTER = 4
    
print(Season.SPRING)
#Enums have names and values. The name of Season.SPRING is SPRING. The value of Season.SPRING is 1.
print(Season.SPRING.name)
print(Season.SPRING.value)
print(type(Season.SPRING))
print(repr(Season.SPRING))
print(list(Season))

In [ ]:
# pydantic, dynaconf, configparser, are libraries for python configuration.


In [ ]:
# # CrashCourse 'pseudocode' :
#     # -> means 'return' or returns its like the output of the thing
#     

In [ ]:
#MyFav functions; 
isinstance(obj, type)
str.join(iter[str]) # only works on string iterables such as 'st' and ['s','t']
map(func, iter) # apply func on every element in iter
map(str, [1,2]) # apply str on 1 and 2. -> ['1', '2']
list.extend()   # similar to append. Appends a list at element level. [1,2].extend([3,4]) -> [1,2,3,4]

            for x in range(0,self.width(), 100):

# Python range(). Zero indexed default. start inclusive stop exclusive.
range(3) -> [0,1,2]

range(4,10,2) ->[4,6,8/]

In [ ]:
#hardest, most difficult functions
map() lambda() sorted() 

column_defs = ','.join([f"{column_name}"  for column_name in columns]) #Note that f"{col} for col in columns returns the same list but elements are all now string
string = ','.join(map(str, [10,20,30]))


In [ ]:
#https://docs.python.org/3/tutorial/classes.html#private-variables
^^ py docs private variables 
#Static 
"static" means different things between python, java, c, C++, etc. 
'static' often associated with 'private', but the terms may be loosely static or private, depending on language
where 'private' means the object is only visible to other objects in its same file/same folder. Private functions thus have a limited scope;namespace.

Py: No such thing as private methods. However, convention to indicate which functions ought to be treated as private exists. 
Private methods can only be called from within the class. 
C: static keyword makes function only visible to other functions in same file:
static void my_private_function(){}
#For example, constructor's purpose is to initialize the class. IF you write some functions for initialization, they are likely agood candidate for private functions-- since they are used by the constructor, internally, and a user should not be using them, since the constructor is doing it for them.


In [ ]:

# CRASHCOURSE function basics
def f(non_default_argument_1, non_default_argument_2, non_default_argument_3, default_arg = None):
    print('study hard')
    '''
    Non-default arguments, have no default values; are given values when f() called: 
                f('one', 'two', 'three')
                f('one', 'two', 'three')
    default arguments, already have values; user can 'update' values, else, default values are used.
                f('one', 'two', 'three', 'four')
    positional arguments; position of arguments is important. Python knows that 'four' is default_arg becasue default_arg came fourth, and 'four' came fourth, too. 
    Call default arguments by name, if you don't want to rely on position
                f('one','two','three', default_arg = 'four')
    default arguments must go after non-default arguments, always, else error thrown
    Each non-default arguments MUST be given a value upon calling of the function. Default arguments, already have a value, and will use that value by default. You can choose to update the value 
    
    '''

In [ ]:


# CRASH COURSE python map(fun, iter) : apply a function fun to each item of an iterable iter
# returns a map object of the results after applying the given function to each item of iter.
# create a list from map object with list() or a set  from map object with set() or a tuple from map object with tuple()
# namedtuples and map and sqlite all work well together : 

def square(n):
    return n*n
iter = (2,4,6,8)
result = map(square, iter)
print(result)
print(list(result))

#map() lambda, filter(), sorted() often used with each other 


In [ ]:
# filter(func, iter)
# filter iter according to func
l = [1,2,3,4]
def filter_list(l):
    return filter( lambda x : x%2 == 0 , l)

print(filter_list(l))
print(list(filter_list(l)))

In [ ]:
# zip() pairs up two iterables, producing tuples. 
x= zip(['name1', 'name2'], ['value1', 'value2']) # ->  <zip object at 0x000001BF5A490F00> 
print(list(x)) # -> [('name1', 'value1'), ('name2', 'value2'), ...]

l1= ['a','b','c']
l2  = [1,2,3]
y= list(zip(l1,l2))

print(y)
print()

for i, j in y: 
    
    print(i, j)
    


In [ ]:


# CRASHCOURSE named tuples : lightweight data structures 
# Behavior similar to both classes and dictionaries.
# (key,value) linked pairs CAN be accessed mid-iteration, a property which dictionaries LACK. (dictionaries cannot mutate during iteration-- copies of dictionary can be made to emulate mutation during iteration)
# NamedTuple class is part of the collections module thus you must : from collections import namedtuple 
#namedtuple(typename, field_names)
# typename- The name of the namedtuple
# field_names-The list of attributes stored in the namedtuple.
# 
# The field_names are a sequence of strings such as ['x', 'y'] OR as a single comma or whitespace separated string : 'x y' or 'x, y'
# field_names must not start with an underscore. 

mutable_list = [1,2,3]
_tuple = (mutable_list) # Immutable tuple contains mutable element
# "immutable" tuples is true -- you cannot add or delete tuple elements, you cannot change the size of the tuple. 
# you cannot reassign the tuple's elements once created: You could not change:
# _tuple[0] = 'string'  # TUPLES DO NOT SUPPORT REASSIGNMENT 
# _tuple.append()       # NOT A VALID TUPLE METHOD
# _tuple[0].append(10)  # TUPLES DO NOT SUPPORT REASSIGNMENT
# But you COULD change the list, which would then REFLECT in the tuple 
mutable_list.append(10)
print(_tuple) #-> ([1,2,3,10])
# _When a list mutates, its name;its ID; its memory location; stays the same, so the tuple is OK with mutating the lists. 
# Python code to demonstrate namedtuple()
from collections import namedtuple # This module implements specialized container datatypes providing alternatives to Python’s general purpose built-in containers, dict, list, set, and tuple.
# Declaring namedtuple() 
SpecifyClassName           = namedtuple('TypeNameHere', ['key_1','key_2']) 
#The typename is used in print(instance_of_nt) such as Student(name='Nandini', age='19', DOB='2541997')
print(SpecifyClassName.__doc__) #__doc__ shows (?)
# Make an instance of the namedtuple 'ClassName'
instance_name            = SpecifyClassName('value_1', 'value2')
print(instance_name[0])
print(instance_name.key_1)

Student                 = namedtuple('Student', ['name', 'age', 'DOB'])
Part                    = namedtuple('Part', ['MPN','product_attributes'])
# 'Part' is classname             type is 'Part' ,'product attributes' and 'MPN' are required 'field_names'
print(Part) #<class '__main__.Part'>
product_attributes  = [ 'Category', 'Capacitors']
Part_1 = Part('01ab', product_attributes)
S = Student('Nandini', '19', '2541997')
print(S)# Student(name='Nandini', age='19', DOB='2541997')

print(Part_1)
print(Part_1[0])
print(Part_1.product_attributes)
# Access using index
print(f"Get Student age using index: {S[1]}")
print(f"part data using index: {Part_1.product_attributes}")
# Access using name
print(f"Get Student name using keyname: {S.name}")
print(f"part data using keyname : {Part_1.product_attributes}")
from collections import namedtuple
# In addition to the methods inherited from tuples, 
# named tuples support three additional methods and two attributes. 
# To prevent conflicts with field names, the method and attribute names start with an underscore.
# Underscore _var is merely a suggestion, to other developers: “Don’t use this outside the class/module.” _var is seen as a different variable than 'var', as x is a different variable than y.
# a double underscore triggers 'name mangling', in which python actually renames the variable to _ClassName__variable. This is needed to avoid accidental name overwriting between subclasses.
# Convention trailing underscores are used to avoid keyword conflicts: 'list' is a keyword. You can not assign list = [1,2]. But, you can use list_ = [1,2]
# alone, the underscore is conventionally a 'throwaway' variable. Oft used in for loops. 'for _ in [1,2]:'
# Double underscore methods, shorthand 'dunder' methods

# classmethod somenamedtuple._make(iterable)
# Class method that makes a new instance from an existing sequence or iterable.
Point = namedtuple('Point', ['x','y'])
t = [11, 22]
Point._make(t) # -> Point(x=11, y=22)
p_a_names = ['mpn', 'category']
p_a_values =['01ab', 'capacitor']
Part = namedtuple('Part', p_a_names)
#Part._make(p_a_values) #-> Part(mpn = 01ab, category = capacitor)
part = Part._make(p_a_values) 
print(part)
print(part._fields)            # view the field names
print([pa for pa in part])     # view the value names -- aka , unpacking like a normal tuple; like a normal iterable

# In addition to the methods inherited from tuples, namedtuples additionally support: 
# Three methods, two attributes:
#_make(iterable) 
#_asdict()
#_replace(++kwargs)
#_fields
#_field_defaults

# _make() method makes an instance, avoids accessing every element. below is tedious:
part = Part(p_a_values[0], p_a_values[1])
print(part)

# somenamedtuple._asdict()
# Return a new dict which maps field names to their corresponding values:

# somenamedtuple._replace(**kwargs)
# ** is the 'unpacking' feature of python dictionaries. 
property_effects = {
    "font": {"size": (1.27, 1.27)},
    "justify": "left"
}
new_dict = {**property_effects, "hide": True}

# new_dict will be:
# {
#     "font": {"size": (1.27, 1.27)},
#     "justify": "left",
#     "hide": True
# }
# Return a new instance of the named tuple replacing specified fields with new values:
p = Point(x=11, y=22)
p._replace(x=33)
#part._replace(mpn = '23bc') #-> Part(mpn=23bc, category = capacitor)
part_updated = part._replace(mpn = '23bc')
print(part_updated)

# for partnum, record in inventory.items():
#     inventory[partnum] = record._replace(price = newprices[partnum], timestamp = time.now())
d = {'x': 11, 'y': 22}
Point(**d) # -> Point(x=11, y=22)

# somenamedtuple._fields
# Tuple of strings listing the field names. Useful for introspection and for creating new named tuple types from existing named tuples.
print(p._fields)            # view the field names
Color = namedtuple('Color', 'red green blue')
Pixel = namedtuple('Pixel', Point._fields + Color._fields)
Pixel(11, 22, 128, 255, 0)

# To convert a dictionary to a named tuple, use the double-star-operator
# # CRASHCOURSE Type hints, aka, type annotations. (type as in string aka str or integer aka int  or floating point integer aka float, etc.) Here are the basics:
# # you can choose to annotate type with comments in python. You can also annotate type directly in python syntax. (I did not know you could do these! The idea is that type annotation, either with comments or syntax, increases code transparency.) As in below :
# # NOTE: using python syntax to annotate type is generally better than using comments. So I will use that way. For example: 
# # type annotation comments have to be initialized, even if only to None, in order to work:( if 'location' is a string):
# location  = None    # type: str #Note: initial value is needed # note: may not work
# # type annotation syntax need not be initialized:
# location:str # Note: no initial value is needed
# # NOTE: dataclasses module uses python syntax type annotations as well. So I focus on type annotation via python syntax

# # Annotate type with python syntax as in: ( if 'primes' is a list of integers ):
# primes: List[int] = []
# # Annotate type with comments as in:
# primes = [] # type: List[int]

# class ExampleClass:
#     # 'my_data' is a class variable
#     my_data = {}    # type: Dict[str,int]
#     my_data: ClassVar[Dict[str,int]] = {}
#     #A little confusing. Focus on dataclasses usage

In [ ]:
class Foo:
    _a = 1
    
class Bar: 
    _b = 2
    
    class BarBar: 
        _b=1
        
print(Foo()._a)
print(Bar()._b)
print(Bar.BarBar()._b)




In [ ]:
#DECORATORS : In python, fucntions are 'first class' objects. This means functions may be passed to a function just like a variable. To pass a function as a parameter, just use the function name without its parentheses. 
# A decorator  takes a function and returns a new modified function. decorators take  another function as an argument, and return a new function after modifying behavior. A decorator takes another function as an argument, modifies, and returns new function object, leaving origional unchanged. Use decorator with @ syntactic sugar. You can write your own decorators, and python has some built-in decorators such as @classmethod and @staticmethod. Python uses the @ symbol to denote decorators. 
    
def func(fx_arg): # take arg, return modified function.
    def new_fx_obj():
        print('a function was passed to func(), this is a modification, func()-> a new function object, making func a decorator')
        fx_arg()
    return new_fx_obj

@func           # decorator demands function definition
def f(): 
    print('asdf')
f()

############
def f():
    print('yolo')
############
x= func(f)      # manually modify function
x()
############
f = func(f)     # equal to decorator
f()

############


# @func # BAD define func after decorator
# f()



In [ ]:
#another decorator 
#decorator: a function takes function, changes function, returns a new function with changes. 
def funcTakesFuncReturnsWithChanges(FuncToChange):
    def FuncToReturn():
        FuncToChange() # execute function
        print('This is technically a change')
    return FuncToReturn # function object. May execute later with FuncToReturn()

def f():
    print('f() prints this message')

changedFunc = funcTakesFuncReturnsWithChanges(f)

changedFunc()

In [ ]:

# #CRASHCOURSE Python import and folder structure 
# import module_name        # import module_name into this file. Access its functions, variables, etc, with mod_name.function() method. "Best Practice" because the source file of import is clear.
# from module_name import * # asterisk meanks all. Make available all functions, classes, and variables. Directly make all functions, classes and variables into the current namespace. Allows direct usage, such as ki_prep(). This WILL cause clashes IF funcs/methods/vars from imported file have same names as this file. Also, makes it unclear where imported functions/modules/vars sourced from. MAY prefer to 'leave namespace attached' as in # import module_name in future.I like this way 
                            

In [ ]:
#     # CrashCourse python dictionary      type: dict



# {} Curly brackets. key:value pairs. COLON denotes key:value pairs NOT commas
# cannot mutate a dict, while iterating through it; Cannot change keys while iterating through dict, cannot change values while iterating through dict. Cannot use dict methods to change the dict while iterating through it. But, you can iterate over a copy. Can only make copies of the dict(can you use dict.methods() while iterating through dict? NO python will raise a runtime error )
# dict.copy() returns a copy of dict    (shallow copy)(don't worry about it). 
#     copy = dictionary.copy()
# key COLON value pairs as in {key:value, key2:value2, key3COLONvalue3, ...}
# The dict.update() method 
#     takes EITHER a dictionary{} or an iterable object of key:value pairs [ ('key',value), ('k':'v')...] or [ ['key',value], ['k':'v']...] 
# The dict.update() method 
#     returns None; is an inplace method; 
# dictionary.update( {'k':'v'} })
# dictionary.update( [other]    )   
# dict.get(key)  ->  value 

# Dict.update() works 'inplace'. It returns None. Thus rr_part = dict.update(dict2) returns None. To create all_attribtutes, .copy a dict into rr_part, then .update() rr_part. 
#dict unpacking 

# Many ways to combine a dictionary: 
# combined = {**dict1, **dict2} 
# dict1.update(dict2)
# dict1 | dict2 

In [ ]:
# () and functions 

# r.json instead of r.json() returns the method r.json itself. Putting that method into a dict creates a "bound method" I guess.
# Object of type method is not JSON serializable
# <class 'dict'> {'product_details': <bound method Response.json of <Response [401]>>, 'media': <bound method Response.json of <Response [401]>>}

In [ ]:
# lists, advanced 
l = [1,2,3]
i = 0

l[i]            # Refers to the entire list object itself.
l[i][:]         # Refers to the contents of the list, preserving the reference to the original list.
# Why It Matters:
# l[i] replaces the reference with a new list. 
# l[i][:] modifies the list in place, preserving the original reference.

In [ ]:
# sorted(iterable, key) RETURNS list containing all items from iterable in ascending order. create custom sorting method with key =.
# Adjacent to sorted() is the list.sort() method 
# # Sorted default sorts by numnerical if int types, lexicographical if string type

sentence = "This is a sentence" 

sentence = ["zebra", "apple", "bat", "ant"]
sentence = ','.join(sentence)

print(sorted(sentence)) # note that sorted() lexicographical order places Capital letters before lowercase. Thus "H" comes before 'a'. 

print(sorted(sentence.split(',') )) # 'ant' comes first
sorted(d, key = lambda k: d.get(k), reverse=True)
# use custom sorting key. sorted(w) 
print(sorted(sentence.split(',') , key = lambda w: sorted(w))) # This is  trippy. sorted() is used twice. words.split(',') is sorted acoording to key = lambda w: sorted(w).sorted(w) takes each element of words.split, and it would sort elements according to their lexico order. 'zebra' becomes 'aberz', and 'aberz' actually goes first, not last. 

# Problem statment: Take a sentence. arrange the letters of each word alphabetically. Then arrange each of those words alphabetically. 




In [ ]:
     # CrashCourse list.index()      # Return index of element in list.
# list.index(element, start, end) 
# my_list = ['apple', 'banana', 'cherry']
# index = my_list.index('banana')
# print(index)  # Output: 1

In [ ]:
#     # CrashCourse Lambda functions           Do a function but it has no name; unlike def myFunc(): ;lambdas are 'inline functions'; functions that can be written in one line;
# lambda arguments: expression          argument         :   expression
# lambda x, y, z  : sum(x,y,z)          argumnents xyz   :   expression sum(x,y,z)
# lambda: True                          NO argument      :   True
# lambda x: x + 10                      argument,x       :   expression,x+10. 
# lambda: print("Button Clicked!")      NO argument      :   expression print()
# The simplest lambda function: A function that returns None: 
lambda:None # -> function object, not none
print(lambda: None) # prints <function <lambda> at 0x000001E48A0CAC00>
x = lambda:None 
print(x) # x is assigned lambda: None, aka, a function object. Still prints <function <lambda> at 0x000001E48A0CAC00>
x() 
# print(x())  We must CALL the lambda function using x(), else -> lambdafunction OBJECT, instead of the result of the runned function! 

In [ ]:
def reggie(x, y): 
    print(x,y)

D = {0: lambda x, y: reggie(x,y), 1: lambda: x}    

D.get(0, lambda:None)(3,5)

# A Super Confusing Mess of Python: master this, and you get a sticker.


In [ ]:
# advanced dictionary lambda 

x = 'predefined elsewhere'
y= 'predefined elsewhere'
def drawLine(x,y):
    print(x,y)
def drawPoints(z):
    print(z)
def drawRect(x,a):
    print(x,a)
draw_methods = {
    0: lambda: drawLine(x, y),
    1: lambda: drawPoints(z),
    2: lambda x,a: drawRect(x,a)
}


draw_methods.get(0) #-> function object, so nothing happens
draw_methods.get(0)() # calls the function object, (but no args passed- x and y were not passed to lambda, xy are defined outside the Dict.  )
# print(draw_methods.get(1)()) Err, z not defined 
z = 1
draw_methods.get(1, lambda:None)() 

draw_methods.get(2, lambda:None)



In [ ]:
D = {0.390:1}
print(D[0.39])
# Dict Keys can be numbers no problem(even floats! Though, due to floats being binary, you shouldn't use floats, but if you HAD to, convert float to string or round the floats, but never do that. 


In [ ]:
# #pandas crashCourse
# Remember, in pandas, some function are in_place; modify the dataframe directly, while some return a NEW DataFrame, requiring you to 'capture' that DataFrame by assigning it to a variable. Often you wish it would just always happen in-place.. Intermediate pandas demands you be versed in creating, and indexing with, boolean DataFrames/Series(df>0 vs df[df>0] and df.loc[df.loc[:, 'B'] == 4). print() statements can fail to print, when 'advanced' stuff is printing, like a .plot() or .display() are being used

import pandas as pd

# df.shape -> tuple (num_cols, num_rows)

# Load a DataFrame from a CSV file
# df = pd.read_csv("file_path.csv")

# first_row = df.iloc[0]  #access row via integer(Iloc). index0 would be the first row.
# rows_4_5_10 = df.iloc[[4, 5, 10]] # select rows 4,5 & 10. 
# first_column = df.iloc[:, 0] # : -> allrows. L = [1,2,3]  L[:] ->[1,2,3]

# # Sample DataFrame
# df = pd.DataFrame({
#     'mpn': ['123abc', '456def', '789ghi'],
#     'category': ['capacitor', 'resistor', 'diode']
# })

# df['category']  #access column by name w/ square brackets. 

# NOTE: Pandas' .loc and .iloc are confusing because .loc[] uses square brackets, not parentheses. This is weird. On top of that, .loc[firstArgument , secondArgument], firstArgument and SecondArgument are sometimes themseleves lists. This makes it harder for the human eyeball to tell where firstArguemnt ends and secondArgument begins. Als.o, NOWHERE in the docs does pandas let you know that .loc[firstArguemnt , secondArguemnt] firstArguemnt is ROWS, secondArguemnt is COLUMNS. 

# calling rows by a number is hard to remember. You can index your records; provide a name to access records with:
# df.setindex('mpn') 
# df.head()
# df.loc['123abc'] -> returns record where index == 123abc
# df.loc[ ['123abc', '456def'] ] #may give a list of records to get 

# columns = df.columns.tolist()     #columns, also called headers. # Turn column index  into a list : ['Category', 'Mfr', 'Series']
# category_first_row = df['Category'].iloc[0] #Access cell by column name and row. 
# cell_value = df.loc['RowName', 'ColName'] # accesses the value in the row labeled 'RowName' and the column labeled ColName 

# so called 'Boolean Indexing': df['mpn'] > 100 creates a series with booleans indicating where condition is T/F. Can use with and and or & and |

# Numpy has a concept called 'broadcasting' which carries over to pandas. It lets you do stuff like 'df > 0'. DataFrame isn't a number, but we can broadcast the notion of comparing against 0 to every cell in df, and python handles that for you. 
# This is similar(?) to the concept of 'vectorized' operations

print(df > 0) # Returns a dataframe of bool, True where cells are > 0. 
for i in df.loc[:,'c']: # get all of column 'C' 
    print(i)
    
#CONFUZE ALERT
df[df>0] 
df>0 returns a boolean DataFrame, True where cell values are > 0. (Err thrown if any values not int or float)
squarebrackets can take that boolean DataFrame, using it to...


    
bool_list = df.loc[:, 'c'] > 9  # This gets all values for col 'c' where value > 9. 
x = df.loc[:, ['a', 'c'] ] > 9 # -> boolean shadow of cols 'a' and 'c', True where value > 9

x = df[:,:] # equivalent to df>9 

print(x)
# df['mpn'] > 100

#     # Crash Course Python TYPE HINTS (nonbinding. Show what the type of the function or variable or argument is
# def add(a: int, b: int) -> int:
#     return a + b
# # Create custom type names for complex types: 
# # # WsObj = gspread.spreadheet.worksheet()

# # Convert column 'A' to a list
# list_a = df['A'].tolist()
# print(list_a)  # Output: [1, 2, 3]

# # Convert the entire DataFrame to a list of lists
# list_of_lists = df.values.tolist()
# print(list_of_lists)  # Output: [[1, 4], [2, 5], [3, 6]]

# # Convert the first row to a list
# first_row_list = df.iloc[0].tolist()
# print(first_row_list)  # Output: [1, 4]



In [ ]:
import pandas 

d = {"A":[0,1,2] , "B":[3,4,5]}
df = pandas.DataFrame(d)

df.head()

In [ ]:
df>0 # DataFrame is not an number, but, pandas overrides the '>' operator so you can easily compare every cell in df against a number. pandas will throw error if any cell in df is not a number. This is an example(?) of concepts call 'vectorization' and 'broadcasting' in action, but dwai.
# comparing DataFrame against number

# df>0 has a special printout but df['A'] > 0 does not, why is that

In [ ]:
# df[df>0] # pandas's squarebracket indexing, .loc, and .iloc can take a DataFrame of booleans, as returned by 'df>0'. squarebracket indexing via boolean DataFrame will return a DataFrame, NaN where the boolean df was False

# so whats the difference between pandas square bracket indexing df[] and pandas.loc[]/iloc[]? 
idx = pandas.Index(['x','y','z'])
df.set_index(idx , inplace=True) 
print(df)
print(df[df.loc[:, 'B'] ==3])

print(df.loc[df.loc[:,'B'] == 3])




In [ ]:
import pandas 
# Change column order reorder columns 
# Strategy 1 .loc() and .iloc[] to reorder 
data = {"A":'1'} 
# df = pandas.DataFrame([list(data.keys()), list(data.values())])



data = [{"A":'1'} , {"B":2} ] 
# df = pandas.DataFrame(data)



data = [ {"A":1, "B":2 } ] 
df = pandas.DataFrame(data)
print()
print(df)

In [ ]:

import os
print(os.path.join('/parts', 'hey' '/there', *['a','c']))
# os.makedirs('one/two\\three')  The nice thing about 'os' module is that it handles fwdslash and backslash the same(Windows uses backslash in file paths for no good reason)

In [ ]:
# WRITING TO FILES 

# json.dump(jso, 'NewFile.txt') No Bad second arg must support write: pass a file object made with acontext manager
with open('NewFile.txt', 'w') as fo: 
    json.dump(jso, fo)


In [ ]:
L = ['a','b']
print('/'.join(L) + '.csv')

In [ ]:
# python typing module, syntax 
# The module 'typing' is good. but does make things harder for beginners, because its got all weird syntax that you don't know what it means. The 'typing' module gives hints on what type a variable is. 

# Union[...] syntax comes from Python's typing module and  means “this function accepts either one type or another. Its basically an 'OR'. Union[A, B] means accept either A or B"
# How to zoom the view such that all items on the scene are shown? 
# Ah-- BOTH - and + worked. HOWEVER, I had to drag resize the little window in order to see the line in the -y. 
# How to toggle my print statements on/off,without having to retype/delete them each time? 

In [ ]:
import lxml.etree as etree 
root = etree.Element('pin')
at = dict(zip(('X', 'Y', 'ANGLE'), map(str, (10, 11 , 60))))
print(at)
etree.SubElement(root, 'at', **at)
etree.SubElement(root, 'asdf', key = 'value')
print(etree.tostring(root, pretty_print=True, encoding=str))
root.getchildren()
print()
print('CHILDREN:')
print(root.getchildren())


desc = list(root.iterdescendants('at'))[0]
chi = list(root.iterchildren('at'))

# return list of all child/descendant elements with  .iterdescendants()/iterchildren(). May specify tag_name to only return tags of tag_name
print()
print('DESC:')
print(desc)
for d in desc:
    print( d.tag )
    
print()
print('CHILDREN')
print(chi)
for ch in chi: 
    print(ch.tag)
    


In [ ]:
import pandas as pd 
D = {'a':[1,2,3,4] , 'b':[5,6,7,8], 'c':[8,9,10,11]}
df = pd.DataFrame(D)
# df.set_index('a') This line does NOTHING, bc .set_index() does NOT mutate the dataframe, instead it returns A NEW dataframe
df = df.set_index('a') 
print('DF:')
print(df)

print()
print("DF.COLUMNS:")
print(df.columns)

print()
print("DF.INDEX:")
print(df.index)
for i in df.index:
    print(i)
print()

print(df.loc[2])

print()
print(df.columns.get_loc('b'))




# choose pins for drawing 


In [ ]:
# Advanced (?) dict and list 
D = {'a':[1]}
x = id(D)
D.get('a').append(4)
print(D)
print(x == id(D))

# get values in a dictionary to mutate inplace, no problems, and memeory adresses remain the dame for the dict, and the list 


In [ ]:
L = [1]
x=  id(L)

key = L
# Nothing being reassigned here. key takes on mem address of L 
print(id(key) == x )
print()

L= [2] # L is being reassigned here. L is being overwritten, and L now refers to another memeory address 
print(id(key) == x )

print(key)
print(L)

In [ ]:
type(1)
type = 'hey'

# type(3) Don't name your variables 'type' as you will overshadow the builting 'type' function

In [ ]:
#Vocab
#record         row
#columns        the names of the columns. 
#index          the names of the rows. 
#
import pandas as pd

s = pd.Series([10, 20, 30, 40, 50])         # Series ~ list
L = [[1,2,3], [4,5,6]]                   # A list of lists is easy to make into a DF. Each index becomes a row.  
D = {'A': [1, 2, 3], 'B': [4, 5, 6]}     # A dictionary is easy to make into a DF. keys become columns. 

df = pd.DataFrame(L)        
print(df)            #A DataFrame. columns is [0,1, etc]. index is [0,1,etc] too. 
print()
print(df[0])         # column '0'. 
print()
print(df.iloc[0,0])                     #iloc() works if index & columns are non-integer. 
print()
print(df.set_index(0))                  # Name the rows  according to an existing column with set_index. The column will be removed from columns and become the DataFrame's index. Here, index will become column 0; first row named '1' and second row is named '4' 
# Note that set


df = pd.DataFrame(D)
df.set_index('A',inplace=True)          #set_index() grabs an existing column, to name rows. 


#attempt access on non-int index or columns with int will not work; you have renamed the index and columns, they are no longer called '0' '1' '2' etc. 


In [ ]:
#INDEXING

#Index with df[] to get columns by name. df[] can NOT fetch rows.
df[0]                   #get the 0th column. Can't do this after you rename the columns to something else
df['col1']              #get a column, col1. If you didn't name a column =='col1', this will not work.
df[['cat', 'voltage']]  #provide a list of column names to fetch

df['col9'][0]           #get 'col9' at 0th index.
#Index with df.loc[] and df.iloc
# .loc ONLY works with non-int
# df.loc[ ROWS ]
# df.loc[ COLS ]
# df.loc[ ROWS, COMMA, COLS]
df.loc[     ['row1','row2','row8'] , ['col1','col4']     ] # provide a list of rows, and a list of columns, to get.  

#df.iloc[] ONLY works with integer
#df.iloc[ROWS, COMMA, COLUMNS]
df.iloc[0:2 , 5:9] # Get rows 0-2 and columns 5-9



s.pop(0) #rm and return index0. Works on Series, and DataFrame columns
print(s)

df = pd.DataFrame({'A' : [5,15,25,35],
                   'B' : [2, 4, 6, 8],
                   'C' : ['a', 'b', 'c','d']})
# print(df['A'][0])
# print(df.iloc[1, 1])

df.set_index('A') # take a column and put it as the 'index'; name rows with it. Removes the column from the columns, as its an index now, if you need to access it, its still associated with the DF. setindex -> new df, so , I've lost the set_index() changes. 

#assign a variable, use inplace=True, or, loose the result
df = df.set_index('B')
#

df.set_index( 'C', inplace = True)
# column a has int values, but, you can still set it as an index. Index can be int, doesn't have to be str
print(df)

A = df.pop('A')
print(type(A))
#once an index is assigned, you HAVE to use that index: you can't reference with numbers anymore. 
A = A.pop('a')
print('pop', A)

In [ ]:
# Sample DataFrame
data = {'A': [1, 2, 3], 'B': [4, 5, 6]}
df = pd.DataFrame(data, index=['row1', 'row2', 'row3'])
# df.pop('B')
df = df.reset_index()
display(df)
df.pop('index') #df.pop() can only pop column names. If 

data = [[1,2,3], [4,5,6]]
df = pd.DataFrame(data)
display(df)
df.columns = ['mpn', 'voltage']
# pop = df.pop(1) After we name the columns, we HAVE to use their new names. integer accessing no longer works.
# print(pop)

In [ ]:
#CRASH COURSE GSHEETS APIv4
#https://developers.google.com/sheets/api/guides/values#append_values


# # CrashCourse gspread.spreadsheet.worksheet methods
# spreadsheet.add_worksheet() # renames the worksheet : use to add new worksheets
# worksheet.insert_row # Adds a row to the worksheet at the specified index and populates it with values.
# worksheet.append_row() # Adds a row to the worksheet and populates it with values.
# worksheet.freeze() # perhaps I should freeze whole worksheet before I API on it, future proofing
# worksheet.delete_row(1) #delete the header row
# worksheet.add_row()
# worksheet.update_cells() # copy many cells at once.
# worksheet.update_acell() # copy the value of a cell.
# If you name a variable with the same name as a function, the variable will overshadow the function

# """         
# #python hack: Triple quotes: technically not comments, they are multiline strings, yet they can be used as comments if not assigned to a variable. Won't be executed by Python.

# """
# #Could digikey be blocking my access? Yes, it brought me to " please click and hold to verify that you are human " "At digikey, we love robots, but our site is for humans. please click/hold btn to verify you are one" I could getDKey from my cookied DKey account on my PC, but got press/hold button from a incognito browser. After clicking/holding, I was able to again access the site. 









In [ ]:
# Loops For loop while loop python loops continue break 
#                 continue # continue: Used in loops, for, while, to skip the current iteration and begin the next iteration. 
for i in range(5): 
    if i == 2:
        continue # stop the current iteration and begin the next iteration. If i==2, stop this iteration and bein the next iteration.
    print(i)
    
Note Break, continue are for for, while, loops. if statements are not loops.

In [ ]:
# Hash Hashing Hash Table: 
# Hashing is the process of converting one value into another, based on a specified string of characters called a 'key'. hashing is used in the creation of hash tables. 
# Hashing is done in dictionaries, which use key:value pairs, although 'hash tables' do this on a much larger scale.

# ---> 40             self.pins.update({k,[]}) # bc of , rather that : {k,[]} is a SET not a dict. sets can only contain hashable(aka immutable) types
# TypeError: unhashable type: 'list'
#hashing, is the act of taking some data, runniing it through a hashing function, to return a fixed-size value called a hash
print(hash('Hello')) # -> 8664487403194276553
# immutables(str, int, set) may be hashed, but mutables (list) cannot. 
# rmbr that strings are immutable.
# dict keys are hashed bts


In [ ]:
# code order, code sequence, 
# in python, code runs top to bottom. 
# you gotta define things before you use them
# note that functions:
def func(param1): 
    print(param1)
func('Hello')
# but if func() were to use otherFunc(), otherFunc() better be above func().

In [ ]:
import os

os.path.exists('True/if/relative/path/exists')
# check if a path exists. path is relative to cwd. 
os.getcwd() # get current working directory 

# raw strings and filepaths 
# A raw string in Python is defined by prefixing a string with r or R. This tells Python to treat backslashes (\) as literal characters, instead of normal escape characters. This is particularly useful for windows file paths, which use backslashes as separators. 


In [ ]:
# The itertools module : combinatorics 
import itertools 
pin = 4855

for digit in pin: 
    L = [[digit] + adjacent[digit] for digit in pin]
    L = [ [4157], [123], [], [] ]
    itertools.product()
    return [''.join(ele) for ele in list(itertools.product(*L))] # here, *L unpacks L. Since L is a listoflists, it'll pass each list as a parameter to itertools.product()

         
# itertools.product() as in the cartesian product of two sets, as in the set of all ordered pairs (a,b) where a belongs to A and b belongs to B
# combinatoric itertools are included in itertools

# The four python combinatoric iterators: From itertools: 

# product()
# permutation()
# combinations()
# combinations_with_replacement()

# product(): A cartesian product() is a permutation, with replacement. 

# permutation():Order matters. Permutations are arrangements. Anagrams, which are words whose letters are rearranged from another word, are permutations. There are n factorial, n!, permutations of n objects. "iamlordvoldemort" and "tommarvoloriddle" are permutations;anagrams. permutations and passwords. 

# combinations() A combination is a selection of n things taken from k things. A combination is a selection. A poker hand, which has 5 cards, from a deck of 52 cards, is a 5-combination where n=5, k=52. There are three combinations of two that can be drawn from the set (apple, orange, pear) (apple,pear), (apple, orange), (orange, pear). There are 2,598,960 possible combinations.Although the set of three fruits was small enough to write a complete list of combinations, this becomes impractical as the size of the set increases. For example, a poker hand can be described as a 5-combination (k = 5) of cards from a 52 card deck (n = 52). The 5 cards of the hand are all distinct, and the order of cards in the hand does not matter. 

# combinations_with_replacement()A combination_with_replacement() is a selection but your picks remain in the deck and could be re-chosen again; a lock with a rotary dial, where each number could be re-used. 

# Those rotarty 'Combination locks' should really be called 'combination_with_replacement' locks, becasue the same number can be reused. Oh wait-- combination_with_replacement ignores 

# those locks with 5 spinners with numbers/letters should really be called "permutation_with_replacement" ; "cartesian_product" locks, because their available codes include all permutations with replacement of their values.  

# The number of results of a combinatoric of an iterable is least with combinations, equal between combinations_with_replacement and permutations, (always?), and greatest with permutations_with_replacement aka cartesian product







In [ ]:
from itertools import product, permutations, combinations, combinations_with_replacement

l = 'ABC'
n=2

print(f"The list is {l}")
print()

print(f"All the {n}-permutations of  {l} is:")
print(list(permutations(l, 2)))
print()


print (f"The {n}-combination of {l} is:") 
print(list(combinations(l, n))) 
print()

print(f"The {n}-combination of {l} (with replacement):")
print(list(combinations_with_replacement(l, n)))
print()

repeat = n
print(f"The cartesian product of {l} using repeat={repeat}:")
print(list(product(l, repeat=2)))
print()

In [ ]:
# 1) Cartesian product itertools.product() generate cartesian product of two sets; the set of all ordered pairs (a,b) where a belongs to A and b belongs to B 
# 'ordered pairs': a pair in which the order is significant. Like coordinates (x,y). 
# import the product function from itertools module
from itertools import product
 
print("The cartesian product using repeat:")
print(list(product([1, 'geeks'], repeat=2)))
print()
 
print("The cartesian product of the containers:")
print(list(product(['geeks', 'for', 'geeks'], '2')))
print()
 
print("The cartesian product of the containers:")
print(list(product('AB', [3, 4])))

In [ ]:
# 2) Permutations itertools.permutations(iter, group_size=len(iter)) generate all permutations of an iterable. 
# import the product function from itertools module

from itertools import permutations
 
print("All the permutations of the given list is:")
print(list(permutations([1, 'geeks'], 2)))
print()
 
print("All the permutations of the given string is:")
print(list(permutations(['geeks', 'for', 'geeks'])))
print()
 
print("All the permutations of the given container is:")
print(list(permutations(range(3), 2)))

In [ ]:
#permutations 
perm = itertools.permutations([1,2,3])
for p in perm: 
    print(p)
    # Time complexits : O(n!), becasue there are n! permutations of n elements. 


In [ ]:
# Combinations(): This iterator prints all the possible combinations(without replacement) of the container passed in arguments in the specified group size in sorted order.
# import combinations from itertools module 

from itertools import combinations 
   
print ("All the combination of list in sorted order is:")  
print(list(combinations(['A', 2], 2))) 
print() 
   
print ("All the combination of string in sorted order is:") 
print(list(combinations(['geeks', 'for', 'geeks'], 2)))
print() 
   
print ("All the combination of list in sorted order is:") 
print(list(combinations(range(2), 1))) 



In [ ]:
# Combinations_with_replacement(): This function returns a subsequence of length n from the elements of the iterable where n is the argument that the function takes determining the length of the subsequences generated by the function. Individual elements may repeat itself in combinations_with_replacement function.
# import combinations from itertools module
 
from itertools import combinations_with_replacement
 
print("All the combination of string in sorted order(with replacement) is:")
print(list(combinations_with_replacement("AB", 2)))
print()
 
print("All the combination of list in sorted order(with replacement) is:")
print(list(combinations_with_replacement(['geeks', 'for', 'geeks'], 2)))
print()
 
print("All the combination of container in sorted order(with replacement) is:")
print(list(combinations_with_replacement(range(2), 1)))

In [ ]:
# I have 20 .zip files. How can I extract them all to a folder? In Python: 
import os
import zipfile
import datetime

# Define source and destination folders
downloads_folder = os.path.expanduser(r"~\Downloads")  # Downloads folder
destination_folder = os.path.expanduser(r"~\Documents\Extracted")  # Destination folder

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# Get today's date
today = datetime.date.today()

# Get a list of .zip files modified today
zip_files = [
    f for f in os.listdir(downloads_folder) if f.endswith(".zip")
    and datetime.date.fromtimestamp(os.path.getmtime(os.path.join(downloads_folder, f))) == today
]

if not zip_files:
    print("No new zip files found today.")
else:
    # Extract each zip file
    for zip_file in zip_files:
        zip_path = os.path.join(downloads_folder, zip_file)
        extract_path = os.path.join(destination_folder, os.path.splitext(zip_file)[0])  # Subfolder for each zip

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)  # Extract contents

        print(f"Extracted: {zip_file} -> {extract_path}")

    print("Extraction completed.")

   


In [ ]:
 
# PARABOLA BEZIER CURVES PATH STROKER STROKING OFFSET HULL DECASTELJAU  QPAINTERPATHSTROKER 

# MAPS BUFFERING GIS: https://en.wikipedia.org/wiki/Buffer_analysis 
# A quadratic function is a parabola. A quadratic's degree(highest exponent) is 2, as in f(x) = x**2+3x+4. The solutions to a quadratic are the 1,2,or 3, 'zeros'/'roots', which can be found w/ quadratic formula, x = -b+/- sqrt(b**2-4ac)/2a, or by ( whats the other method called?). )) 
        # Quadratic bezier curves are a parabola. Bezier curves represent curves with 'control points' P0 to Pn. The first and last control points are the start and end of the curve. n is said to be the 'order' of the curve.
        # A curve with n=1 is a first order curve, or 'linear'. A bezier curve with n=1 has two control points P0 P1, and is a straight line. 
        # A bez with n=2 is a second order curve, or 'quadratic'. A bez with n=2 has control points P0 P1 P2, and is a parabola. 
        # A bez with n=3 is a third order curve, or 'cubic'. A bez with n=3 has control points P0 P1 P2 P3, and is not a parabola. ( It is a cubic function, yes? Which is not a parabola) 
        # A bez with n >= 4 is a 'fourth order' 'fifth order', etc, curve, and these are basically never used, because they are expensive to calculate. Instead, we use 'composite beziers' which are many quadratic/cubic beziers chained together.
        # N-order Beziers can be described by (N-1)Order beziers; a cubic bezier can be described by a quadratic bezier.
        # There are three conic sections: Ellipses, parabolas, and hyperbolas. Circles are ellipses, so circles are a conic section too, and they fall under ellipses. 
            # Look at drawings of conic sections: A plane which slices a cone forms an ellipse, parabola, or hyperbola, depending on the angle of the plane. Hyperbolas have an 'upper' curve and a 'below' curve, but parabolas and circles have just one curve. Bezier curves 

        # A lower-order bez, can always be rewritten as a higher order bez, but this doesn't work the other way. 
        # Offsetting a Bezier curve is a known difficult problem, which is called 'stroking' in computer graphics. Other names include offset/expansion/buffering/envelope/stroking. How can you create a curve which is always the same distance from; 'parallel to', another curve? 
        # The curve offset from a given bezier curve, cannot be exactly formed by a Bezier curve( except in some trivial cases, like when the bez is just a line...) 
        # The offset curve of a parabola is a higher-order algebraic function : 
        # The two-sided offset of curve of a cubic bez , is a 10th order algebraic curve, and the offset curve of a N-th order bez is an algebraic curve of degree 4n-2 ! This is why people usually use decasteljau's Algo to approximate the offset curve as abunch of short segments. 
        # DeCasteljau's Algorithm performs 'recursive flattening'(breaking the curve into smaller and smaller segments). This turns the curve into a bunch of lines, which is just a busy polygon.
        # In Qt, The class QPainterPathStroker is used to offset paths. QPainterPathStroker packs up these curve offsetting strategies for you.


# SO example on why cubic bezs may be preferable to quadratic(parabola)




In [ ]:
Gravitation Inside A Uniform Hollow Sphere https://www.grc.nasa.gov/WWW/K-12/Numbers/Math/Mathematical_Thinking/grvtysp.htm
NASA Glenn research center in Ohio

The gravitational force inside a hollow sphere shell of uniform areal mass density is everywhere equal to zero, and may be proved by the following argument:

Let the sphere have a radius a. Place a point P inside the sphere at a distance r from the center where r < a; i.e., r is strictly less than a. Draw a line through P to intersect the sphere at two opposite points. Call these points alpha andbeta. Let the distance from P to alpha be r1, and the distance from P tobeta be r2.

Now place a differential area dAalpha at alpha, and project straight lines through P to acquire its image dAbeta at beta. These two areas subtend a solid angle dflux at P. Let the sphere have areal mass density rho(kg/m2). Then the net differential attraction dF of dAalpha and dAbeta at P directed toward alpha is just

dF = rho( dAalpha /r12 - dAbeta/r22).

But dAalpha = r12 dflux, and dAbeta = r22 dflux by definition of the solid angle. Thus,

dF = rho((r12 dflux)/r12 - (r22 dflux)/r22) = 0.

This result is true for all choices of dAalpha and dAbeta. The gravitational force within the sphere is everywhere equal to zero.

#### Use $$ to denote LaTEX formulas: Here's one for a Bezier curve

$$
\left(\left(1-t\right)\left(\left(1-t\right)\left(\left(1-t\right)x_{0}+tx_{1}\right)+t\left(\left(1-t\right)x_{1}+tx_{2}\right)\right)+t\left(\left(1-t\right)\left
(\left(1-t\right)x_{1}+tx_{2}\right)+t\left(\left(1-t\right)x_{2}+tx_{3}\right)\right),\left(1-t\right)\left(\left(1-t\right)\left(\left(1-t\right)

y_{0}+ty_{1}\right)+t\left(\left(1-t\right)y_{1}+ty_{2}\right)\right)+t\left(\left(1-t\right)\left(\left(1-t\right)y_{1}+ty_{2}\right)+t\left(\left(1-t\right)y_{2}+ty_{3}\right)\right)\right)
$$

In [ ]:
# This was back before i knew about visibility graphs. I was trying to insert float locations into a grid.

#     def grid(self): ### Insert a row and a column into a matrix. Not robust against say, same position being inserted into graph twice
#         self._graph = np.array()
# # CREATE ON-GRID GRAPH
#         for i in int(self.itemsBoundingRect().width()):
#             for j in int(self.itemsBoundingRect().width()):
#                 self._graph[i,j] == (i*self.board_grid_step , j*self.board_grid_step) # if not self.is_occluded(i*self.board_grid_step , j*self.board_grid_step) else 1
# # CREATE OFF-GRID GRAPH , WHEN NEEDED : when user is routing trace, when user is ... ? 
#         print('SELF._GRAPH:', self._graph)
#         return self._graph

    def insert_into_graph(self, graph, position):
        x, y = position
        num_rows = len(graph)
        num_columns = len(graph[0])

        # graph.insert(math.floor(y)+1 , l) # can't just insert at math.floor(y). May have x= .7 while a x=.5 already exists in the graph-- ( This would cause other routing problems but those r dealt with later)
        # So we need to know all the y's already in list, then insert after the nearest lesser match 

        row_insert_index = first_greater_y(graph, y)
        # print('ROW_INSERT_INDEX:', row_insert_index)
        row_to_insert = [ [graph[0][col_idx][0] , y ] for col_idx in range(num_columns) ] # graph[0][col_idx][0] bc graph[0] is the first row, row[col_idx] is a specific position, and specific_position[0] is the x-coordinate of specific_position. We HAVE to use x-coordinate already existing in graph, bc we don't know whats been added to our graph
        graph.insert( row_insert_index,  row_to_insert)
        # print('GRAPH STAGE1:')
        # for i in graph:
        #     print(i)
        column_insert_index = first_greater_x(graph, x) 
        for row_index, row in enumerate(graph): 
            cell_to_insert = [ x , graph[row_index][0][1] ] # graph[row_index][0][1] because graph[row_index] is a specific row that exists in graph, specific_row[0] is an actual cell, and actual_cell[1] is the y-coordinate of a specific position. We HAVE to use y-coordinate already existing in graph,bc we don't know whats been added to our graph.
            row.insert(column_insert_index, cell_to_insert)
        # print()
        # print('GRAPH STAGE2:')
        # for i in graph:
        #     print(i) 
            
        def first_greater_x(graph, value):
            num_columns = len(graph[0])
            
            graph_xs = []
            row = graph[0] 
            for column_index in range(num_columns):
                graph_xs.append(row[column_index][0])
            # print('GRAPH XS:', graph_xs)
            for count, item in enumerate(graph_xs):
                if item>value:
                    return count
            return count+1 # Is used with list.insert() and inserting a value at index greater than length of list appends value.
        
        def first_greater_y(graph , value): # Returns index of first number in lst which is greater than value, or len(lst) if no greater number exists
            graph_ys  = [ row[0][1] for row in graph]
            # print('GRAPH_YS:', graph_ys)
            for count, item in enumerate(graph_ys): 
                if item>value: 
                    return count
            return count+1 # Ok (?) to return index-out-of-range. insert() would merely append



        # import numpy as np
        # import matplotlib.pyplot as plt
        # arr = np.array(normal_graph)
        # print('ARR:')
        # print(arr)
        # for row in arr: 
        #     plt.plot(row[:,0] , row[:,1], 'go')

        # Hey there's gotta be a library that already does this--matrix insertion (?) matrix expansion(?)